# Analyse approfondie des retraits terminal et détection des anomalies

**Objectif.** Comprendre le comportement normal de chaque terminal et faire émerger toutes les anomalies, y
compris les plus cachées, au grain de la transaction. Trois familles de détection :
- **anomalies directes (01 à 11)** : un indicateur sort nettement de l'habitude du terminal ;
- **anomalies cachées (12 à 17)** : rien n'est extrême pris séparément, mais la forme de la journée, le
  mélange des réseaux, l'écart aux terminaux comparables, une dérive lente, un trou horaire récurrent ou une
  combinaison de petits écarts trahit le problème ;
- **anomalies avancées (18 à 26)** : cassette vide, incident de zone, attaque coordonnée, motifs de capture,
  échecs d'un réseau, rythme trop régulier, heure morte, même carte sur plusieurs terminaux, écart à l'année N-1.

Le catalogue repère beaucoup de journées atypiques ; la **priorisation** (section 11) ne transmet au métier
que les plus graves, dans la limite de sa capacité de traitement quotidienne.

**Organisation.** Notebook autonome : tout le code est dans ce notebook (aucune librairie à installer) :
la section 1 (configuration) est la seule à modifier ; la section « Code de l'analyse » contient les fonctions
(préparation, indicateurs, règles d'anomalies, graphiques) et ne se modifie pas.

**Toutes les lignes sont utilisées.** Les calculs sont faits par Spark sur la table entière ; seuls des
résultats agrégés sont ramenés en pandas. Kernel : environnement **distrib** (ex. `python36distrib_new`).

## 0. Imports et session Spark

In [ ]:
import copy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from IPython.display import display
from collections import namedtuple
from functools import reduce
from pyspark.sql import functions as F, SparkSession, types as T, Window


spark = SparkSession.builder.enableHiveSupport().getOrCreate()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

## 1. Configuration
Noms des colonnes de la requête d'entrée (dataset des opérations 2026) et règles de gestion de la fiche
d'identité terminal (codes réseau, codes de carte privative, libellés).
Seule la partie droite se modifie si un nom diffère dans la table ; les clés à gauche sont utilisées par
le code. Les seuils des règles se règlent dans `seuils={...}` (liste complète : `SEUILS_DEFAUT`).

In [ ]:
DATASET_OPERATIONS = "nom_du_dataset"        # nom du dataset des opérations dans le Flow Dataiku
TABLE_HIVE_OPERATIONS = None                 # ou nom Hive complet "base.table" (si pas de dataset)
DATASET_SORTIE_TRANSACTIONS = None           # ex. "retraits_enrichis" : enregistre la table enrichie (section 4)
DATASET_SORTIE_PHOTO = None                  # ex. "photo_terminal_jour" : photo quotidienne des terminaux (section 13)
DATASET_SORTIE_OPERATIONS = None             # ex. "operations_anomalies" : opérations en anomalie (section 15)

# --- noms des colonnes : code -> nom dans la table
COLONNES = {
    "num_automate": "num_automate",
    "date_operation": "date_operation",               # JJMMAAAA
    "heure_operation": "heure_operation",             # HHMMSS (les zéros de tête peuvent manquer)
    "montant_ope": "montant_ope",                     # en centimes ('00020000' = 200 euros)
    "lib_ope": "lib_ope",                             # Retrait / Autorisation-Retrait / Capture de Carte
    "type_reseau": "type_reseau",
    "code_carte_privative": "code_carte_privative",
    "top_annule": "top_annul",                        # 1 = retrait annulé
    "lib_motif_capture_carte": "lib_motif_capture_carte",
    "date_validite_carte": "date_valid_carte",
    "type_client": "type_client",                     # Porteur / Non_porteur
    "code_site": None,                                # absent de la requête
    "libelle_site": "lib_site_implantation",
    "code_postal": "cd_pst",
    "ville": "ville",
    "longitude": "longitude",
    "latitude": "latitude",
    "nb1": "nb_billetsk_7_1", "nb2": "nb_billetsk_7_2",           # billets délivrés par cassette
    "nb3": "nb_billetsk_7_3", "nb4": "nb_billetsk_7_4",
    "coupure1": "val_coupurek_7_1", "coupure2": "val_coupurek_7_2",   # valeur du billet de chaque cassette
    "coupure3": "val_coupurek_7_3", "coupure4": "val_coupurek_7_4",
    "id_carte": None,                                 # facultatif (règle 25) : absent de la requête
}

# --- règles de gestion de la fiche : type_reseau, puis code_carte_privative
CODES_RESEAU = {
    # à compléter
}
CODES_CARTE_PRIVATIVE = {
    # à compléter
}
GROUPES_RESEAU = {      # réseau -> groupe affiché dans les graphiques (les autres : « Privatives / autres »)
    "CB": "CB", "Carte interbancaire": "Interbancaire", "Visa": "Visa", "Mastercard": "Mastercard",
    "JCB": "Etrangeres rares", "UPI": "Etrangeres rares", "Amex": "Etrangeres rares",
    "Carte Diners et Discovery": "Etrangeres rares", "cos": "COS",
}
RESEAU_INTERNE = ""                          # captures interne / hors réseau interne
VALEUR_PORTEUR = "Porteur"                   # valeur de type_client ; tout retrait interne est mis à « Porteur »
RESEAUX_ETRANGERS_RARES = []   # à compléter
TYPES_OPERATION = [     # (motif LIKE sur lib_ope en majuscules, type) ; le premier trouvé l'emporte
    ("%AUTORISATION%", "RETRAIT_EXTERNE"),   # Autorisation-Retrait
    ("%CAPTURE%", "CAPTURE"),                # Capture de Carte
    ("%RETRAIT%", "RETRAIT_INTERNE"),            # Retrait
]
MOTIFS_CAPTURE = {
    "OUBLI_OU_INCIDENT_LECTURE": ["OUBLI CARTE (OU INCIDENT LECTEUR CARTE)", "OUBLI DE CARTE",
                                  "INCIDENT LECTURE DE CARTE", "INCIDENT LECTURE CARTE"],
    "CODE_CONFIDENTIEL_DEPASSE": ["NOMBRE ESSAIS CODE CONFIDENTIEL DEPASSE", "NB ESSAI CODE CONFIDENTIEL DEPASSE",
                                  "NOMBRE ESSAI CODE DEPASSE"],
    "CARTE_PERDUE": ["CARTE PERDUE"],
    "CONSERVER_LA_CARTE": ["CONSERVER LA CARTE"],
    "CARTE_VOLEE": ["CARTE VOLEE", "CARTE VOLÉE", "CARTE DECLAREE VOLEE", "CARTE DÉCLARÉE VOLÉE", "VOL DE CARTE"],
}

# --- formats et période
FORMAT_DATE = "ddMMyyyy"     # format de date_operation
DIVISEUR_MONTANT = 100       # montant_ope en centimes
DATE_DEBUT, DATE_FIN = None, None   # ex. "2026-09-01", "2026-09-30" pour un premier essai ; None = toute la table

## Code de l'analyse (ne pas modifier)
Toutes les fonctions utilisées par le notebook, une par cellule. Exécuter ces cellules sans les modifier :
les valeurs à adapter sont dans la section 1.

### Code : Configuration par défaut
Configuration par défaut : noms de colonnes, codes métier, options et seuils.

Rien n'est codé en dur ailleurs dans la librairie : tout passe par le dictionnaire renvoyé par
`configuration()`, que le notebook peut surcharger partiellement, par exemple :

    conf = configuration(colonnes={"num_automate": "id_terminal"}, seuils={"rafale_min_10min": 15})

In [ ]:
# Noms des colonnes de la table des opérations (fiche d'identité terminal). Clé = nom utilisé dans le code ;
# valeur = nom dans la table. À ajuster dans la cellule « Configuration » du notebook si un nom diffère.
COLONNES_DEFAUT = {
    "num_automate": "num_automate",
    "date_operation": "date_operation",               # JJMMAAAA
    "heure_operation": "heure_operation",             # HHMMSS (les zéros de tête peuvent manquer)
    "montant_ope": "montant_ope",                     # en centimes ('00020000' = 200 euros)
    "lib_ope": "lib_ope",                             # Retrait / Autorisation-Retrait / Capture de Carte
    "type_reseau": "type_reseau",
    "code_carte_privative": "code_carte_privative",
    "top_annule": "top_annul",                        # 1 = retrait annulé
    "lib_motif_capture_carte": "lib_motif_capture_carte",
    "date_validite_carte": "date_valid_carte",
    "type_client": "type_client",                     # Porteur / Non_porteur
    "code_site": None,                                # absent de la requête
    "libelle_site": "lib_site_implantation",
    "code_postal": "cd_pst",
    "ville": "ville",
    "longitude": "longitude",
    "latitude": "latitude",
    "nb1": "nb_billetsk_7_1", "nb2": "nb_billetsk_7_2",           # billets délivrés par cassette
    "nb3": "nb_billetsk_7_3", "nb4": "nb_billetsk_7_4",
    "coupure1": "val_coupurek_7_1", "coupure2": "val_coupurek_7_2",   # valeur du billet de chaque cassette
    "coupure3": "val_coupurek_7_3", "coupure4": "val_coupurek_7_4",
    "id_carte": None,                                 # facultatif (règle 25) : absent de la requête
}
MOTS_CLES_DEFAUT = {
    "num_automate": [["automate"], ["terminal"]],
    "date_operation": [["date", "op"]],
    "heure_operation": [["heure"]],
    "montant_ope": [["montant"]],
    "lib_ope": [["lib", "ope"]],
    "type_reseau": [["type", "reseau"], ["reseau"]],
    "code_carte_privative": [["privative"]],
    "top_annule": [["annul"]],
    "type_client": [["type", "client"], ["porteur"]],
    "lib_motif_capture_carte": [["motif"]],
    "date_validite_carte": [["validit"], ["valid", "carte"]],
    "code_site": [["code", "site"]],
    "libelle_site": [["lib", "site"]],
    "code_postal": [["postal"], ["cd_pst"]],
    "ville": [["ville"]],
    "longitude": [["longitude"], ["lon"]],
    "latitude": [["latitude"], ["lat"]],
    "id_carte": [["id_carte"], ["hash"]],
}
MOTS_CLES_DEFAUT.update({"nb{}".format(i): [["nb{}".format(i)]] for i in range(1, 5)})
MOTS_CLES_DEFAUT.update({"coupure{}".format(i): [["coupure", str(i)]] for i in range(1, 5)})
COLONNES_OBLIGATOIRES_DEFAUT = ["num_automate", "date_operation", "montant_ope", "lib_ope"]
# Règles de gestion de la fiche d'identité terminal : code type_reseau, puis code_carte_privative.
CODES_RESEAU_DEFAUT = {
    # à compléter
}
CODES_PRIVATIVE_DEFAUT = {
    # à compléter
}
GROUPES_RESEAU_DEFAUT = {
    "CB": "CB", "Carte interbancaire": "Interbancaire", "Visa": "Visa", "Mastercard": "Mastercard",
    "JCB": "Etrangeres rares", "UPI": "Etrangeres rares", "Amex": "Etrangeres rares",
    "Carte Diners et Discovery": "Etrangeres rares", "cos": "COS",
}
METIER_DEFAUT = {
    "codes_reseau": CODES_RESEAU_DEFAUT,
    "codes_privative": CODES_PRIVATIVE_DEFAUT,
    "groupes_reseau": GROUPES_RESEAU_DEFAUT,
    "groupe_par_defaut": "Privatives / autres",
    "reseau_interne": "",   # à compléter
    "valeur_porteur": "Porteur",                 # un retrait sur le réseau interne est toujours fait par un porteur
    "reseaux_etrangers": [],   # à compléter
    # Type d'opération selon le libellé : retrait interne, retrait externe, capture
    # (sans majuscules ; le premier motif trouvé l'emporte)
    "types_operation": [("%AUTORISATION%", "RETRAIT_EXTERNE"), ("%CAPTURE%", "CAPTURE"),
                        ("%RETRAIT%", "RETRAIT_INTERNE")],
    "motifs_capture": {
        "OUBLI_OU_INCIDENT_LECTURE": ["OUBLI CARTE (OU INCIDENT LECTEUR CARTE)", "OUBLI DE CARTE",
                                      "INCIDENT LECTURE DE CARTE", "INCIDENT LECTURE CARTE"],
        "CODE_CONFIDENTIEL_DEPASSE": ["NOMBRE ESSAIS CODE CONFIDENTIEL DEPASSE",
                                      "NB ESSAI CODE CONFIDENTIEL DEPASSE", "NOMBRE ESSAI CODE DEPASSE"],
        "CARTE_PERDUE": ["CARTE PERDUE"],
        "CONSERVER_LA_CARTE": ["CONSERVER LA CARTE"],
        "CARTE_VOLEE": ["CARTE VOLEE", "CARTE VOLÉE", "CARTE DECLAREE VOLEE", "CARTE DÉCLARÉE VOLÉE",
                        "VOL DE CARTE"],
    },
    "billets_par_defaut": {"nb1": 5, "nb2": 10, "nb3": 20, "nb4": 50},
}
OPTIONS_DEFAUT = {
    "format_date": "ddMMyyyy",       # format de date_operation quand c'est du texte
    "diviseur_montant": 100,         # montant_ope en centimes -> euros (1 si déjà en euros)
    "heure_debut_jour": 7,           # plage horaire étudiée pour les silences et les trous horaires
    "heure_fin_jour": 22,
    "date_debut": None,              # None = toute la table
    "date_fin": None,
    "persister": True,               # garde les tables préparées en mémoire / disque du cluster
    "groupe_pairs": "libelle_site",  # terminal comparables entre eux (type de site, ville, code_postal...)
    "zone": "departement",           # zone géographique des règles 19-20 : departement, code_postal ou ville
    "sites_interieurs": None,        # règle 24 limitée à ces sites (ex. ["Agence"]) ; None = tous les terminaux
    "jours_particuliers": [],        # dates à atténuer dans le score (vacances, événements) ; fériés automatiques
    "metropole": {"longitude": (-5.5, 10.0), "latitude": (41.0, 51.5)},
}
SEUILS_DEFAUT = {
    # --- anomalies directes (01 à 11)
    "silence_proba": 1e-5,           # 01 : probabilité maximale que le silence arrive par hasard
    "muet_quantile_parc": 0.90,      # 02 : terminal « très fréquenté » = 10 % du parc le plus actif
    "muet_retraits_habituels": 7,    # 02 : retraits habituels minimum ce jour de la semaine
    "rafale_min_10min": 10,          # 03 : opérations en 10 minutes
    "rafale_facteur": 2,             # 03 : x niveau élevé habituel du terminal (99e centile)
    "meme_montant_min": 6,           # 04 : retraits du même montant...
    "meme_montant_fenetre_s": 300,   # 04 : ... en 5 minutes
    "inaboutis_serie_min": 3,        # 05 : inaboutis consécutifs
    "captures_heure_min": 3,         # 06 : captures hors réseau interne en une heure
    "etrangers_heure_min": 20,       # 07 : retraits de cartes étrangères rares en une heure
    "etrangers_facteur": 5,          # 07 : x niveau horaire habituel du terminal (90e centile)
    "nuit_min": 3,                   # 08 : retraits entre 0 h et 5 h...
    "nuit_part_max": 0.01,           # 08 : ... sur un terminal dont moins de 1 % des retraits sont la nuit
    "volume_ratio": 3,               # 10 : retraits x3
    "volume_ecart": 4,               # 10 : et à plus de 4 écarts
    "volume_habitude_min": 10,       # 10 : sur un terminal qui fait au moins 10 retraits ce jour-là
    "montant_ratio": 3,              # 11 : montant moyen x3
    "montant_retraits_min": 5,       # 11 : avec au moins 5 retraits dans la journée
    # --- anomalies cachées (12 à 17)
    "profil_volume_min": 20,         # 12 : retraits minimum dans la journée pour juger sa forme horaire
    "profil_ecarts_robustes": 6,     # 12 : divergence > médiane + 6 écarts robustes du terminal...
    "profil_divergence_min": 0.20,   # 12 : ... et > 0,20 (divergence de Jensen-Shannon, entre 0 et 1)
    "reseaux_volume_min": 50,        # 13 : idem pour le mélange des réseaux de cartes
    "reseaux_ecarts_robustes": 6,
    "reseaux_divergence_min": 0.12,
    "pairs_ratio_bas": 0.4,          # 14 : le terminal tombe sous 40 % de son habitude...
    "pairs_ratio_normal": 0.8,       # 14 : ... alors que ses pairs restent au-dessus de 80 %
    "pairs_ratio_haut": 2.5,         # 14 : ou dépasse 250 % quand ses pairs restent sous 130 %
    "pairs_ratio_haut_pairs": 1.3,
    "pairs_nb_min": 5,               # 14 : au moins 5 terminaux pairs ce jour-là
    "pairs_habitude_min": 20,        # 14 : terminal qui fait habituellement au moins 20 retraits ce jour-là
    "derive_ratio": 0.6,             # 15 : les 4 dernières semaines < 60 % des 8 précédentes...
    "derive_semaine_min": 0.25,      # 15 : ... sans semaine quasi nulle (ce serait une panne)
    "derive_pairs_min": 0.85,        # 15 : ... alors que les pairs restent au-dessus de 85 %
    "derive_reference_min": 50,      # 15 : référence d'au moins 50 retraits par semaine
    "trou_part_jours_vides": 0.5,    # 16 : heure vide plus d'un jour sur deux dans le mois...
    "trou_taux_horaire_min": 3,      # 16 : ... alors que le terminal devrait y faire au moins 3 retraits ce mois-là
    "trou_jours_min": 20,            # 16 : sur au moins 20 jours d'activité dans le mois
    "multivarie_part": 0.001,        # 17 : 0,1 % des journées terminal, parmi celles qu'aucune autre règle n'a vues
    "multivarie_echantillon": 300000,
    # --- anomalies avancées (18 à 26)
    "cassette_retraits_min": 30,         # 18 : journée d'au moins 30 retraits...
    "cassette_part_habituelle_min": 0.15,  # 18 : ... coupure qui pèse habituellement au moins 15 % des billets...
    "cassette_part_restante_max": 0.10,  # 18 : ... tombée sous 10 % de sa part habituelle
    "zone_silence_proba": 1e-3,          # 19 : terminal muet = dans un silence de probabilité < 1/1 000 (règle 01)...
    "zone_terminal_min": 5,                   # 19 : ... au moins 5 terminaux d'une même zone muets à la même heure
    "coordonnee_rafale_min": 15,         # 20 : heure suspecte = rafale d'au moins 15 opérations en 10 min...
    "coordonnee_meme_montant_min": 6,    # 20 : ... ou 6 retraits du même montant en 5 min...
    "coordonnee_etrangers_min": 10,      # 20 : ... ou 10 retraits de cartes étrangères rares
    "coordonnee_terminal_min": 3,             # 20 : sur au moins 3 terminaux d'une même zone dans la même heure
    "motifs_surveilles": ["CARTE_VOLEE", "CARTE_PERDUE", "CODE_CONFIDENTIEL_DEPASSE", "OUBLI_OU_INCIDENT_LECTURE"],
    "motif_min": 3,                      # 21 : au moins 3 captures d'un même motif dans la journée...
    "motif_proba": 1e-4,                 # 21 : ... avec une probabilité de hasard < 1/10 000 (Poisson)
    "reseau_retraits_min": 5,            # 22 : au moins 5 retraits du réseau dans la journée...
    "reseau_taux_min": 0.3,              # 22 : ... dont au moins 30 % d'inaboutis...
    "reseau_facteur": 5,                 # 22 : ... soit 5 fois le taux habituel du réseau sur ce terminal...
    "reseau_taux_autres_max": 0.1,       # 22 : ... alors que les autres réseaux restent sous 10 %
    "regulier_nb_retraits": 8,           # 23 : le même délai entre deux retraits (à la seconde) revient 8 fois...
    "regulier_delai_min_s": 10,          # 23 : ... délai compris entre 10 s...
    "regulier_delai_max_s": 180,         # 23 : ... et 3 minutes...
    "regulier_fenetre_s": 900,           # 23 : ... dans une fenêtre de +/- 15 minutes...
    "regulier_proba": 1e-6,              # 23 : ... avec une probabilité de hasard < 1/1 000 000 (Poisson)
    "heure_morte_retraits_min": 3,       # 24 : au moins 3 retraits dans l'heure...
    "heure_morte_part_max": 0.01,        # 24 : ... à une heure active moins de 1 % des jours
    "carte_vitesse_max_kmh": 300,        # 25 : déplacement impossible au-delà de 300 km/h
    "annee_ratio": 3,                    # 26 : retraits x3 ou /3 par rapport à l'année précédente...
    "annee_habitude_min": 10,            # 26 : ... pour un terminal qui faisait au moins 10 retraits
}

In [ ]:
def configuration(**surcharges):
    """Configuration complète, avec les valeurs par défaut surchargées par `surcharges`.

    Les clés sont : colonnes, mots_cles, colonnes_obligatoires, metier, options, seuils.
    Une surcharge de dictionnaire ne remplace que les clés indiquées.
    """
    conf = copy.deepcopy({
        "colonnes": COLONNES_DEFAUT, "mots_cles": MOTS_CLES_DEFAUT, "colonnes_obligatoires": COLONNES_OBLIGATOIRES_DEFAUT,
        "metier": METIER_DEFAUT, "options": OPTIONS_DEFAUT, "seuils": SEUILS_DEFAUT,
    })
    for cle, valeur in surcharges.items():
        if cle not in conf:
            raise KeyError("Clé de configuration inconnue : {} (attendu : {})".format(cle, sorted(conf)))
        if isinstance(conf[cle], dict) and isinstance(valeur, dict):
            conf[cle].update(valeur)
        else:
            conf[cle] = valeur
    return conf

### Code : Outils
Petits outils Spark / pandas réutilisés dans toute la librairie (compatibles Spark 2.3, Python 3.6).

In [ ]:
CLE_TERMINAL = "num_automate"

In [ ]:
def en_pandas(sdf, max_lignes=5000000):
    """Ramène un résultat Spark DÉJÀ AGRÉGÉ en pandas (les calculs portent sur toutes les lignes)."""
    pdf = sdf.limit(max_lignes).toPandas()
    if len(pdf) == max_lignes:
        print("Attention : résultat tronqué à {} lignes.".format(max_lignes))
    return pdf

In [ ]:
def fmt_nombre(n):
    """12345678 -> '12 345 678'."""
    return "{:,.0f}".format(n).replace(",", " ")

In [ ]:
def en_nombre(colonne):
    """Colonne texte ou numérique -> double (virgule décimale acceptée)."""
    return F.regexp_replace(F.trim(F.col(colonne).cast("string")), ",", ".").cast("double")

In [ ]:
def centile(colonne, q):
    """Centile approché (expression SQL : compatible Spark 2.3). `q` peut être un nombre ou 'array(...)'."""
    return F.expr("percentile_approx({}, {})".format(colonne, q))

In [ ]:
def somme_si(condition, valeur=1):
    """Somme de `valeur` sur les lignes où `condition` est vraie."""
    return F.sum(F.when(condition, valeur).otherwise(0))

In [ ]:
def selon_correspondance(colonne, correspondance, defaut=None):
    """Expression CASE WHEN : valeur de `correspondance` pour chaque valeur de `colonne`."""
    expression = F.when(F.lit(False), F.lit(""))
    for cle, valeur in correspondance.items():
        expression = expression.when(colonne == cle, valeur)
    return expression.otherwise(defaut) if defaut is not None else expression

In [ ]:
def fenetre_meme_jour_semaine(nb_semaines=8):
    """Les `nb_semaines` mêmes jours de semaine précédents, pour chaque terminal."""
    return (Window.partitionBy(CLE_TERMINAL, F.dayofweek("date")).orderBy("date")
            .rowsBetween(-nb_semaines, -1))

In [ ]:
def fenetre_jours_precedents(nb_jours=7):
    """Les `nb_jours` journées actives précédentes, pour chaque terminal."""
    return Window.partitionBy(CLE_TERMINAL).orderBy("date").rowsBetween(-nb_jours, -1)

In [ ]:
def ecart_robuste(colonne, reference, dispersion, plancher=1.0):
    """(valeur - référence) / max(dispersion, plancher) : écart exprimé en « écarts habituels »."""
    return (F.col(colonne) - F.col(reference)) / F.greatest(F.coalesce(F.col(dispersion), F.lit(0.0)),
                                                            F.lit(float(plancher)))

In [ ]:
def union_toutes(tables):
    """Union de DataFrames Spark de même schéma (liste non vide)."""
    return reduce(lambda gauche, droite: gauche.union(droite), tables)

In [ ]:
def persister(table, actif=True):
    """Persiste la table en mémoire / disque du cluster si `actif`."""
    from pyspark import StorageLevel
    return table.persist(StorageLevel.MEMORY_AND_DISK) if actif else table

### Code : Préparation des données
Lecture de la table brute, correspondance des colonnes, typage et enrichissement au grain transaction.

In [ ]:
def lire_table(dataset=None, table_hive=None):
    """Lit la table brute : par le dataset Dataiku (recommandé) ou par son nom Hive."""
    spark = SparkSession.builder.getOrCreate()
    if dataset:
        import dataiku
        import dataiku.spark as dkuspark
        from pyspark.sql import SQLContext
        return dkuspark.get_dataframe(SQLContext(spark.sparkContext), dataiku.Dataset(dataset))
    if table_hive:
        return spark.table(table_hive)
    raise ValueError("Indiquer le dataset Dataiku ou la table Hive à lire.")

In [ ]:
def associer_colonnes(colonnes_table, colonnes, mots_cles):
    """Pour chaque colonne attendue : nom exact, sinon sans tenir compte des majuscules, sinon mots-clés.

    Renvoie {colonne attendue: colonne trouvée dans la table, ou None}.
    """
    minuscules = {c.lower(): c for c in colonnes_table}
    deja_prises, correspondance = set(), {}
    for cle, nom in colonnes.items():
        trouve = nom if nom in colonnes_table else minuscules.get((nom or "").lower())
        for mots in mots_cles.get(cle, []) if trouve is None else []:
            candidates = [c for c in colonnes_table
                          if c not in deja_prises and all(m in c.lower() for m in mots)]
            if candidates:
                trouve = candidates[0]
                break
        correspondance[cle] = trouve
        if trouve:
            deja_prises.add(trouve)
    return correspondance

In [ ]:
def standardiser(source, conf):
    """Renomme les colonnes de la table selon la configuration ; les colonnes absentes sont créées vides.

    Renvoie (table standardisée, correspondance).
    """
    correspondance = associer_colonnes(source.columns, conf["colonnes"], conf["mots_cles"])
    manquantes = [cle for cle in conf["colonnes_obligatoires"] if not correspondance[cle]]
    if manquantes:
        raise ValueError("Colonnes indispensables introuvables : {} (configuration « colonnes »)".format(
            manquantes))
    brut = source.select([F.col(nom).alias(cle) if nom else F.lit(None).cast("string").alias(cle)
                          for cle, nom in correspondance.items()])
    return brut, correspondance

In [ ]:
def coupures_presentes(correspondance):
    """Vrai si les 4 colonnes coupure existent dans la table."""
    return all(correspondance.get("coupure{}".format(i)) for i in range(1, 5))

In [ ]:
def geographie_presente(correspondance):
    """Vrai si longitude et latitude existent dans la table."""
    return bool(correspondance.get("longitude") and correspondance.get("latitude"))

In [ ]:
def expression_date(type_colonne, format_date):
    """Date d'opération, que la colonne soit une date ou du texte."""
    if type_colonne in ("date", "timestamp"):
        return F.to_date(F.col("date_operation"))
    texte = F.trim(F.col("date_operation").cast("string"))
    if format_date in ("ddMMyyyy", "yyyyMMdd"):
        texte = F.lpad(texte, 8, "0")
    return F.to_date(texte, format_date)

In [ ]:
def expression_heure():
    """Heure au format HHMMSS sur 6 chiffres (séparateurs retirés, zéros initiaux rétablis)."""
    chiffres = F.regexp_replace(F.trim(F.col("heure_operation").cast("string")), "[^0-9]", "")
    return F.lpad(chiffres, 6, "0")

In [ ]:
def expression_reseau(metier):
    """Catégorie de réseau : code type_reseau, puis code de carte privative, sinon le code brut."""
    code_reseau = F.lpad(F.trim(F.col("type_reseau").cast("string")), 2, "0")
    code_privative = F.trim(F.col("code_carte_privative").cast("string"))
    categorie = selon_correspondance(code_reseau, metier["codes_reseau"])
    categorie = categorie.when(code_privative == "", "Carte interbancaire")
    for code, nom in metier["codes_privative"].items():
        categorie = categorie.when(code_privative == code, nom)
    return (categorie.when(code_privative.isNotNull() & (code_privative != ""), code_privative)
            .otherwise(F.coalesce(code_reseau, F.lit("inconnu"))))

In [ ]:
def expression_groupe(metier):
    """Groupe de réseau affiché dans les graphiques."""
    if not metier["groupes_reseau"]:
        return F.col("cat_reseau")
    correspondance = F.create_map([F.lit(x) for paire in metier["groupes_reseau"].items() for x in paire])
    return F.coalesce(correspondance[F.col("cat_reseau")], F.lit(metier["groupe_par_defaut"]))

In [ ]:
def expression_motif(metier):
    """Motif de capture normalisé ; tout autre libellé renseigné devient AUTRE_MOTIF."""
    texte = F.upper(F.trim(F.col("lib_motif_capture_carte")))
    motif = F.when(F.lit(False), F.lit(""))
    for nom, libelles in metier["motifs_capture"].items():
        motif = motif.when(texte.isin(libelles), nom)
    return motif.when(F.col("lib_motif_capture_carte").isNotNull(), "AUTRE_MOTIF")

In [ ]:
def expression_type_operation(metier, correspondance=None):
    """Type d'opération : RETRAIT_INTERNE, RETRAIT_EXTERNE ou CAPTURE.

    Avec la colonne lib_ope : d'après le libellé (sans tenir compte des majuscules ni des espaces).
    Sans lib_ope (table déjà filtrée sur les retraits et captures) : capture si un motif de capture est
    renseigné, sinon retrait interne si le réseau est le réseau interne, retrait hors réseau interne sinon.
    """
    if correspondance is not None and not correspondance.get("lib_ope"):
        motif = F.trim(F.col("lib_motif_capture_carte").cast("string"))
        return (F.when(motif.isNotNull() & (motif != ""), F.lit("CAPTURE"))
                .when(expression_reseau(metier) == metier["reseau_interne"], F.lit("RETRAIT_INTERNE"))
                .otherwise(F.lit("RETRAIT_EXTERNE")))
    libelle = F.upper(F.trim(F.col("lib_ope").cast("string")))
    type_op = F.when(F.lit(False), F.lit(""))
    for motif, nom in metier["types_operation"]:
        type_op = type_op.when(libelle.like(motif), nom)
    return type_op

In [ ]:
def expression_montant_delivre(brut, correspondance, metier):
    """Montant délivré = somme des (nombre de billets x coupure) ; coupures en euros ou en centimes."""
    if not coupures_presentes(correspondance):
        print("Coupures absentes : valeurs par défaut", metier["billets_par_defaut"])
        termes = [F.coalesce(en_nombre(c), F.lit(0.0)) * v for c, v in metier["billets_par_defaut"].items()]
    else:
        plus_grande = brut.select(F.max(F.greatest(*[en_nombre("coupure{}".format(i)) for i in range(1, 5)]))
                                  .alias("m")).collect()[0]["m"] or 0
        diviseur = 100.0 if plus_grande > 500 else 1.0
        print("Coupures : valeur maximale {} -> divisées par {:.0f}".format(plus_grande, diviseur))
        termes = [F.coalesce(en_nombre("nb{}".format(i)), F.lit(0.0)) *
                  F.coalesce(en_nombre("coupure{}".format(i)), F.lit(0.0)) / diviseur for i in range(1, 5)]
    total = termes[0]
    for terme in termes[1:]:
        total = total + terme
    return total

In [ ]:
def construire_operations(brut, correspondance, conf):
    """Une ligne par opération (retrait ou capture), typée et enrichie de ses indicateurs propres."""
    metier, options = conf["metier"], conf["options"]
    date_op = expression_date(dict(brut.dtypes)["date_operation"], options["format_date"])
    heure_txt = expression_heure()
    type_op = expression_type_operation(metier, correspondance)
    est_retrait = F.col("est_retrait") == 1
    ops = (brut
           .filter(type_op.isNotNull())
           .select(F.col(CLE_TERMINAL),
                   date_op.alias("date"),
                   heure_txt.substr(1, 2).cast("int").alias("heure"),
                   heure_txt.substr(3, 2).cast("int").alias("minute"),
                   F.to_timestamp(F.concat(F.date_format(date_op, "yyyyMMdd"), heure_txt), "yyyyMMddHHmmss")
                   .alias("ts"),
                   (en_nombre("montant_ope") / options["diviseur_montant"]).alias("montant"),
                   F.trim(F.col("date_validite_carte").cast("string")).alias("validite_txt"),
                   type_op.alias("type_op"),
                   expression_reseau(metier).alias("cat_reseau"),
                   expression_motif(metier).alias("motif_capture"),
                   F.coalesce(F.col("top_annule").cast("int"), F.lit(0)).alias("top_annule"),
                   expression_montant_delivre(brut, correspondance, metier).alias("montant_billets"),
                   *[F.coalesce(en_nombre("nb{}".format(i)), F.lit(0.0)).alias("billets_nb{}".format(i))
                     for i in range(1, 5)],
                   F.col("id_carte").cast("string").alias("id_carte"),
                   F.trim(F.col("type_client").cast("string")).alias("type_client"))
           .withColumn("est_retrait", F.col("type_op").isin("RETRAIT_INTERNE", "RETRAIT_EXTERNE").cast("int"))
           .withColumn("est_capture", (F.col("type_op") == "CAPTURE").cast("int"))
           .filter((F.col("est_capture") == 1) | (F.col("montant") > 0))
           .withColumn("groupe_reseau", expression_groupe(metier))
           .withColumn("est_etranger", F.col("cat_reseau").isin(metier["reseaux_etrangers"]).cast("int"))
           .withColumn("est_reseau_interne", (F.col("cat_reseau") == metier["reseau_interne"]).cast("int"))
           .withColumn("type_client", F.when(F.col("est_reseau_interne") == 1, F.lit(metier["valeur_porteur"]))
                       .otherwise(F.col("type_client")))   # réseau interne (interne) = porteur
           .withColumn("est_capture_interne", F.col("est_capture") * F.col("est_reseau_interne"))
           .withColumn("est_capture_externe", F.col("est_capture") * (1 - F.col("est_reseau_interne")))
           .withColumn("ecart_billets", F.col("montant") - F.col("montant_billets"))
           .withColumn("inabouti_annule", (est_retrait & (F.col("top_annule") == 1)).cast("int"))
           .withColumn("inabouti_billets", (est_retrait & (F.abs(F.col("ecart_billets")) > 0.001)).cast("int"))
           .withColumn("inabouti", F.greatest("inabouti_annule", "inabouti_billets"))
           .withColumn("est_nuit", ((F.col("heure") >= 22) | (F.col("heure") < 6)).cast("int"))
           .withColumn("jour_semaine", F.dayofweek("date"))
           .withColumn("mois", F.trunc("date", "month")))
    if options["date_debut"]:
        ops = ops.filter(F.col("date") >= F.lit(options["date_debut"]).cast("date"))
    if options["date_fin"]:
        ops = ops.filter(F.col("date") <= F.lit(options["date_fin"]).cast("date"))
    return persister(ops, options["persister"])

In [ ]:
def enrichir_transactions(ops, actif_persistance=True):
    """Ajoute à chaque opération des indicateurs calculés à partir de ses voisines sur le même terminal.

    delai_precedente_s / delai_suivante_s, rang_dans_journee, nb_ope_5min (opérations à +/- 5 min),
    meme_montant_precedent, inabouti_precedent, capture_dans_l_heure (captures dans l'heure qui précède).
    """
    dans_le_temps = Window.partitionBy(CLE_TERMINAL).orderBy("t")
    journee_dans_le_temps = Window.partitionBy(CLE_TERMINAL, "date").orderBy("t")
    est_retrait = F.col("est_retrait") == 1
    transactions = (ops.filter(F.col("ts").isNotNull())
                    .withColumn("t", F.unix_timestamp("ts"))
                    .withColumn("delai_precedente_s", F.col("t") - F.lag("t").over(dans_le_temps))
                    .withColumn("delai_suivante_s", F.lead("t").over(dans_le_temps) - F.col("t"))
                    .withColumn("rang_dans_journee", F.row_number().over(journee_dans_le_temps))
                    .withColumn("nb_ope_5min", F.count("*").over(dans_le_temps.rangeBetween(-300, 300)))
                    .withColumn("meme_montant_precedent",
                                (est_retrait & (F.col("montant") == F.lag("montant").over(dans_le_temps)))
                                .cast("int"))
                    .withColumn("inabouti_precedent", F.coalesce(F.lag("inabouti").over(dans_le_temps), F.lit(0)))
                    .withColumn("capture_dans_l_heure",
                                F.sum("est_capture").over(dans_le_temps.rangeBetween(-3600, -1))))
    return persister(transactions, actif_persistance)

In [ ]:
def vue_terminal_jour(ops, actif_persistance=True):
    """Comptages par terminal et par jour (vue pour les graphiques et les règles journalières)."""
    est_retrait = F.col("est_retrait") == 1
    vue = (ops.groupBy(CLE_TERMINAL, "date")
           .agg(F.sum("est_retrait").alias("retraits"),
                F.sum("est_capture").alias("captures"),
                F.sum("est_capture_interne").alias("captures_interne"),
                F.sum("est_capture_externe").alias("captures_externe"),
                F.sum("inabouti").alias("inaboutis"),
                F.sum(F.when(est_retrait, F.col("est_etranger")).otherwise(0)).alias("etrangers"),
                F.sum(F.when(est_retrait, F.col("montant")).otherwise(0)).alias("montant"),
                F.sum(F.when(est_retrait, F.col("est_nuit")).otherwise(0)).alias("retraits_nuit")))
    return persister(vue, actif_persistance)

In [ ]:
def referentiel_terminal(brut, correspondance):
    """Une ligne par terminal : coordonnées, nombre de coordonnées différentes, lieu et site d'implantation."""
    colonnes = [c for c in ("code_postal", "ville", "code_site", "libelle_site") if correspondance.get(c)]
    mesures = [F.first(c, True).alias(c) for c in colonnes]
    if geographie_presente(correspondance):
        mesures += [F.first(en_nombre("longitude"), True).alias("longitude"),
                    F.first(en_nombre("latitude"), True).alias("latitude"),
                    F.countDistinct(F.concat_ws("/", F.col("longitude").cast("string"),
                                                F.col("latitude").cast("string"))).alias("nb_coordonnees")]
    if not mesures:
        return brut.select(CLE_TERMINAL).distinct()
    return brut.groupBy(CLE_TERMINAL).agg(*mesures)

In [ ]:
def lire_validite(retraits, cles=(CLE_TERMINAL, "date"), afficher=True):
    """Date de fin de validité des cartes (format MMAA, AAMM, AAAAMM ou AAAAMMJJ détecté automatiquement).

    Renvoie une table (`cles`, mois_restants) ou None si le format n'est pas reconnu.
    """
    validite = retraits.select(*cles, F.regexp_replace("validite_txt", "[^0-9]", "").alias("v"))
    longueurs = (validite.filter(F.length("v") > 0).groupBy(F.length("v").alias("longueur")).count()
                 .orderBy(F.desc("count")).limit(1).collect())
    longueur = longueurs[0]["longueur"] if longueurs else 0
    if longueur == 4:
        sens = validite.filter(F.length("v") == 4).agg(
            F.avg((F.substring("v", 1, 2).cast("int") <= 12).cast("int")).alias("mois_devant"),
            F.avg((F.substring("v", 3, 2).cast("int") <= 12).cast("int")).alias("mois_derriere")).collect()[0]
        mois_devant = sens["mois_devant"] >= sens["mois_derriere"]
        if afficher:
            print("Date de validité : format", "MMAA" if mois_devant else "AAMM")
        mois_v = F.substring("v", 1, 2) if mois_devant else F.substring("v", 3, 2)
        annee_v = F.substring("v", 3, 2) if mois_devant else F.substring("v", 1, 2)
        fin = F.last_day(F.to_date(F.concat(F.lit("20"), annee_v, F.lit("-"), mois_v, F.lit("-01"))))
    elif longueur in (6, 8):
        fin = F.last_day(F.to_date("v", "yyyyMM" if longueur == 6 else "yyyyMMdd"))
    else:
        print("Format de date de validité non reconnu.")
        return None
    return validite.withColumn("mois_restants", F.months_between(fin, F.col("date"))).drop("v")

In [ ]:
def resumer(ops):
    """Chiffres clés de la table préparée."""
    resume = ops.agg(F.count("*").alias("operations"), F.sum("est_retrait").alias("retraits"),
                     F.sum("est_capture").alias("captures"), F.countDistinct(CLE_TERMINAL).alias("terminal"),
                     F.min("date").alias("debut"), F.max("date").alias("fin"),
                     F.countDistinct("date").alias("jours")).collect()[0].asDict()
    for cle, valeur in resume.items():
        print("{:<12}: {}".format(cle, fmt_nombre(valeur) if isinstance(valeur, (int, float)) else valeur))
    return resume

### Code : Indicateurs descriptifs
Calculs descriptifs : chaque fonction renvoie un tableau pandas (déjà agrégé) ou une table Spark réduite.

In [ ]:
JOURS_FR = {2: "lun", 3: "mar", 4: "mer", 5: "jeu", 6: "ven", 7: "sam", 1: "dim"}   # F.dayofweek
ORDRE_JOURS = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]

In [ ]:
# ------------------------------------------------------------------ qualité de la donnée brute
def qualite_colonnes(brut, nb_lignes):
    """Valeurs nulles et chaînes vides, colonne par colonne (une seule passe sur la table)."""
    mesures = []
    for colonne in brut.columns:
        mesures.append(F.sum(F.col(colonne).isNull().cast("int")).alias(colonne + "__null"))
        mesures.append(F.sum((F.trim(F.col(colonne).cast("string")) == "").cast("int")).alias(colonne + "__vide"))
    comptes = brut.agg(*mesures).collect()[0].asDict()
    qualite = pd.DataFrame([{"colonne": c, "nb_null": comptes[c + "__null"], "nb_vide": comptes[c + "__vide"]}
                            for c in brut.columns])
    qualite["pct_null"] = (100 * qualite["nb_null"] / nb_lignes).round(3)
    qualite["pct_vide"] = (100 * qualite["nb_vide"] / nb_lignes).round(3)
    return qualite.sort_values("pct_null", ascending=False)

In [ ]:
def valeurs_prises(brut, colonne, nb_lignes, nb_max=40):
    """Valeurs les plus fréquentes d'une colonne, avec leur part."""
    valeurs = en_pandas(brut.groupBy(colonne).count().orderBy(F.desc("count")).limit(nb_max))
    valeurs["pct"] = (100 * valeurs["count"] / nb_lignes).round(3)
    return valeurs

In [ ]:
def controle_formats(brut, date_op, heure_txt):
    """Dates, heures et montants invalides ; bornes des dates ; montant médian brut."""
    return brut.select(date_op.alias("date"), heure_txt.substr(1, 2).cast("int").alias("hh"),
                       heure_txt.substr(3, 2).cast("int").alias("mi"),
                       en_nombre("montant_ope").alias("montant_brut")).agg(
        F.sum(F.col("date").isNull().cast("int")).alias("dates_invalides"),
        F.min("date").alias("date_min"),
        F.max("date").alias("date_max"),
        F.sum(((F.col("hh") > 23) | F.col("hh").isNull()).cast("int")).alias("heures_invalides"),
        F.sum(((F.col("mi") > 59) | F.col("mi").isNull()).cast("int")).alias("minutes_invalides"),
        F.sum(F.col("montant_brut").isNull().cast("int")).alias("montants_non_numeriques"),
        F.sum((F.col("montant_brut") <= 0).cast("int")).alias("montants_nuls_ou_negatifs"),
        centile("montant_brut", 0.5).alias("montant_median_brut")).collect()[0].asDict()

In [ ]:
# ------------------------------------------------------------------ volumes, rythmes, profils
def indicateurs_par_reseau(ops):
    """Retraits, montants, inaboutis et captures par réseau, comptés transaction par transaction."""
    return indicateurs_par(ops, ["groupe_reseau", "cat_reseau"])

In [ ]:
def indicateurs_par(ops, groupes):
    """Retraits, montants, inaboutis et captures par `groupes` (réseau, type de client...)."""
    est_retrait = F.col("est_retrait") == 1
    table = en_pandas(ops.groupBy(*groupes).agg(
        F.sum("est_retrait").alias("retraits"),
        somme_si(F.col("type_op") == "RETRAIT_INTERNE").alias("retraits_interne"),
        somme_si(F.col("type_op") == "RETRAIT_EXTERNE").alias("retraits_externe"),
        somme_si(est_retrait, F.col("montant")).alias("montant"),
        F.sum("inabouti").alias("inaboutis"),
        F.sum("est_capture").alias("captures"),
        F.countDistinct(CLE_TERMINAL).alias("terminal")).orderBy(F.desc("retraits")))
    retraits = table["retraits"].replace(0, np.nan)
    table["pct_retraits"] = (100 * table["retraits"] / table["retraits"].sum()).round(3)
    table["montant_moyen"] = (table["montant"] / retraits).round(2)
    table["taux_inaboutis_pct"] = (100 * table["inaboutis"] / retraits).round(3)
    return table

In [ ]:
def volumes_par_jour(ops):
    """Retraits, captures, inaboutis et terminal actifs par jour, avec jours fériés et écart au même jour de semaine."""
    jour = en_pandas(ops.groupBy("date").agg(
        F.sum("est_retrait").alias("retraits"), F.sum("est_capture").alias("captures"),
        F.sum("inabouti").alias("inaboutis"), F.countDistinct(CLE_TERMINAL).alias("terminal_actifs")))
    jour["date"] = pd.to_datetime(jour["date"])
    jour = jour.sort_values("date").set_index("date")
    try:
        import holidays
        feries = holidays.France(years=sorted(jour.index.year.unique()))
        jour["ferie"] = [d.date() in feries for d in jour.index]
    except ImportError:
        jour["ferie"] = False
    jour["jour_semaine"] = jour.index.dayofweek.map(dict(enumerate(ORDRE_JOURS)))
    habitude = jour.groupby("jour_semaine")["retraits"].transform(
        lambda serie: serie.rolling(9, center=True, min_periods=3).median())
    jour["ratio_vs_meme_jour"] = (jour["retraits"] / habitude).round(2)
    return jour

In [ ]:
def croisement_jour_heure(retraits, nb_jours_par_semaine, valeur=None):
    """Matrice jour de semaine x heure : retraits moyens par jour, ou moyenne de la colonne `valeur`."""
    if valeur:
        table = en_pandas(retraits.groupBy("jour_semaine", "heure").agg(F.avg(valeur).alias("v")))
    else:
        table = en_pandas(retraits.groupBy("jour_semaine", "heure").count())
    table["jour"] = table["jour_semaine"].map(JOURS_FR)
    if not valeur:
        table["v"] = table["count"] / table["jour"].map(nb_jours_par_semaine)
    return table.pivot(index="jour", columns="heure", values="v").reindex(ORDRE_JOURS)

In [ ]:
def profils_normalises(retraits, colonne, modalites, min_retraits=500):
    """Part des retraits de chaque terminal par modalité de `colonne` (terminal d'au moins `min_retraits` retraits)."""
    comptes = en_pandas(retraits.groupBy(CLE_TERMINAL, colonne).count())
    if colonne == "jour_semaine":
        comptes[colonne] = comptes[colonne].map(JOURS_FR)
    profils = comptes.pivot(index=CLE_TERMINAL, columns=colonne, values="count").reindex(columns=modalites).fillna(0)
    profils = profils[profils.sum(axis=1) >= min_retraits]
    return profils.div(profils.sum(axis=1), axis=0)

In [ ]:
def carte_parc(journees, max_terminal=3000):
    """Matrice terminal x jour : log2(retraits / médiane du terminal pour ce jour de semaine), jours sans opération à 0."""
    volume = journees.groupby(CLE_TERMINAL)["retraits"].median().sort_values(ascending=False)
    terminal_retenus = volume.index[:max_terminal]
    jours = pd.date_range(journees["date"].min(), journees["date"].max())
    matrice = (journees[journees[CLE_TERMINAL].isin(terminal_retenus)]
               .pivot(index=CLE_TERMINAL, columns="date", values="retraits").reindex(index=terminal_retenus, columns=jours))
    vie = journees.groupby(CLE_TERMINAL)["date"].agg(["min", "max"]).reindex(terminal_retenus)
    dans_vie = np.array([(jours >= debut) & (jours <= fin) for debut, fin in zip(vie["min"], vie["max"])])
    matrice = matrice.where(~(matrice.isna() & dans_vie), 0)
    habitude = matrice.T.groupby(matrice.columns.dayofweek).transform("median").T
    return np.log2((matrice + 1) / (habitude + 1)).clip(-3, 3)

In [ ]:
# ------------------------------------------------------------------ le parc de terminaux
def statistiques_par_terminal(ops):
    """Une ligne par terminal : durée de vie, volume, taux d'inaboutis, captures, cartes étrangères, nuit, montant."""
    est_retrait = F.col("est_retrait") == 1
    par_terminal = en_pandas(ops.groupBy(CLE_TERMINAL).agg(
        F.min("date").alias("premier_jour"), F.max("date").alias("dernier_jour"),
        F.countDistinct("date").alias("jours_actifs"),
        F.sum("est_retrait").alias("retraits"), F.sum("est_capture").alias("captures"),
        F.sum("est_capture_externe").alias("captures_externe"), F.sum("inabouti").alias("inaboutis"),
        somme_si(est_retrait, F.col("est_etranger")).alias("etrangers"),
        somme_si(est_retrait, F.col("est_nuit")).alias("retraits_nuit"),
        somme_si(est_retrait, F.col("montant")).alias("montant")), max_lignes=200000)
    for colonne in ("premier_jour", "dernier_jour"):
        par_terminal[colonne] = pd.to_datetime(par_terminal[colonne])
    retraits = par_terminal["retraits"].clip(lower=1)
    par_terminal["duree_jours"] = (par_terminal["dernier_jour"] - par_terminal["premier_jour"]).dt.days + 1
    par_terminal["taux_presence"] = par_terminal["jours_actifs"] / par_terminal["duree_jours"]
    par_terminal["retraits_par_jour_actif"] = par_terminal["retraits"] / par_terminal["jours_actifs"]
    par_terminal["taux_inaboutis_pct"] = 100 * par_terminal["inaboutis"] / retraits
    par_terminal["captures_pour_1000"] = 1000 * par_terminal["captures"] / retraits
    par_terminal["part_etrangers_pct"] = 100 * par_terminal["etrangers"] / retraits
    par_terminal["part_nuit_pct"] = 100 * par_terminal["retraits_nuit"] / retraits
    par_terminal["montant_moyen"] = par_terminal["montant"] / retraits
    return par_terminal

In [ ]:
def filtrer_metropole(table, metropole):
    """Lignes dont les coordonnées sont en métropole."""
    return table[table["longitude"].between(*metropole["longitude"]) &
                 table["latitude"].between(*metropole["latitude"])]

In [ ]:
def controles_geographiques(par_terminal, metropole):
    """Coordonnées absentes, nulles, hors métropole, multiples ou partagées."""
    lon, lat = par_terminal["longitude"], par_terminal["latitude"]
    en_metropole = filtrer_metropole(par_terminal, metropole)
    return pd.Series({
        "terminal sans coordonnées": int((lon.isna() | lat.isna()).sum()),
        "coordonnées à 0": int(((lon == 0) | (lat == 0)).sum()),
        "hors métropole (DOM-TOM ou erreur)": int(lon.notna().sum() - len(en_metropole)),
        "terminal avec plusieurs coordonnées": int((par_terminal["nb_coordonnees"] > 1).sum()),
        "coordonnées partagées par plusieurs terminaux": int(en_metropole.duplicated(["longitude", "latitude"],
                                                                               keep=False).sum()),
    }).to_frame("nb_TERMINAL")

In [ ]:
# ------------------------------------------------------------------ montants, inaboutis, captures
def quantiles_montant(retraits, groupe=None, quantiles=(0.1, 0.25, 0.5, 0.75, 0.9)):
    """Quantiles du montant, globalement ou par `groupe` (une colonne par quantile)."""
    expression = centile("montant", "array({})".format(", ".join(str(q) for q in quantiles))).alias("q")
    if groupe is None:
        return pd.Series(retraits.select(expression).collect()[0]["q"], index=quantiles)
    table = en_pandas(retraits.groupBy(groupe).agg(expression).orderBy(groupe))
    return pd.DataFrame(table["q"].tolist(), index=table[groupe], columns=["p{:g}".format(100 * q) for q in quantiles])

In [ ]:
def controles_montants(retraits):
    """Montants non multiples de 5 €, écarts avec les billets délivrés, retraits sans billet renseigné."""
    return retraits.agg(
        somme_si(F.round(F.col("montant") * 100) % 500 != 0).alias("non_multiples_de_5"),
        somme_si(F.abs(F.col("ecart_billets")) > 0.001).alias("ecart_avec_billets"),
        somme_si(F.col("montant_billets") == 0).alias("aucun_billet_renseigne")).collect()[0].asDict()

In [ ]:
def taux_par(table, groupe, colonne):
    """Taux moyen (en %) de la colonne 0/1 `colonne` par `groupe`."""
    return en_pandas(table.groupBy(groupe).agg((100 * F.avg(colonne)).alias("taux"), F.count("*").alias("n"))
                     .orderBy(groupe))

In [ ]:
def taux_captures(ops):
    """Captures interne et hors réseau interne, et leurs taux pour 1 000 retraits du même réseau."""
    comptes = ops.agg(F.sum("est_capture_interne").alias("captures_interne"),
                      F.sum("est_capture_externe").alias("captures_externe"),
                      somme_si(F.col("type_op") == "RETRAIT_INTERNE").alias("retraits_interne"),
                      somme_si(F.col("type_op") == "RETRAIT_EXTERNE").alias("retraits_externe")).collect()[0]
    return pd.DataFrame([
        {"reseau": "interne", "captures": comptes["captures_interne"],
         "pour_1000_retraits": 1000 * comptes["captures_interne"] / max(comptes["retraits_interne"], 1)},
        {"reseau": "hors réseau interne", "captures": comptes["captures_externe"],
         "pour_1000_retraits": 1000 * comptes["captures_externe"] / max(comptes["retraits_externe"], 1)}])

In [ ]:
# ------------------------------------------------------------------ dynamique dans la journée
def grille_horaire(retraits_transactions, journees, heure_debut, heure_fin, actif_persistance=True):
    """Retraits par terminal, jour et heure (de `heure_debut` à `heure_fin`), heures vides comprises.

    Ajoute `taux_h`, le nombre moyen habituel de retraits du terminal à cette heure (semaine / week-end).
    """
    spark = SparkSession.builder.getOrCreate()
    heures = spark.range(heure_debut, heure_fin).select(F.col("id").cast("int").alias("heure"))
    comptes = retraits_transactions.groupBy(CLE_TERMINAL, "date", "heure").agg(F.count("*").alias("n"))
    grille = (journees.select(CLE_TERMINAL, "date").crossJoin(heures)
              .join(comptes, [CLE_TERMINAL, "date", "heure"], "left").fillna(0, subset=["n"])
              .withColumn("week_end", F.dayofweek("date").isin(1, 7).cast("int")))
    rythme = grille.groupBy(CLE_TERMINAL, "week_end", "heure").agg(F.avg("n").alias("taux_h"))
    return persister(grille.join(rythme, [CLE_TERMINAL, "week_end", "heure"]), actif_persistance)

In [ ]:
def calculer_silences(grille, journees, jours_recents=28):
    """Pire silence de chaque journée terminal : heures consécutives sans retrait et probabilité de hasard.

    Le nombre attendu pendant le silence vient du rythme horaire du terminal, multiplié par :
    - son niveau récent (moyenne des `jours_recents` journées précédentes / moyenne de toute la période), ce
      qui suit la saison (station de ski l'été, littoral l'hiver) sans utiliser la journée elle-même ;
    - l'activité du jour hors silence, seulement à la baisse (journée creuse), jamais à la hausse : une hausse
      calculée sur les quelques heures restantes est trop instable pour les terminaux peu actifs.
    Probabilité = exp(-attendu) (loi de Poisson).
    """
    niveau = (journees.select(CLE_TERMINAL, "date", "retraits")
              .withColumn("recent", F.avg("retraits").over(fenetre_jours_precedents(jours_recents)))
              .withColumn("global", F.avg("retraits").over(Window.partitionBy(CLE_TERMINAL)))
              .select(CLE_TERMINAL, "date", F.coalesce(F.col("recent") / F.col("global"), F.lit(1.0)).alias("saison")))
    niveau_jour = grille.groupBy(CLE_TERMINAL, "date").agg(F.sum("n").alias("n_jour"), F.sum("taux_h").alias("taux_jour"))
    cumul = Window.partitionBy(CLE_TERMINAL, "date").orderBy("heure").rowsBetween(Window.unboundedPreceding, 0)
    blocs = (grille.withColumn("vide", (F.col("n") == 0).cast("int"))
             .withColumn("bloc", F.sum(1 - F.col("vide")).over(cumul))
             .filter(F.col("vide") == 1)
             .groupBy(CLE_TERMINAL, "date", "bloc")
             .agg(F.count("*").alias("silence_h"), F.min("heure").alias("debut_silence"),
                  F.sum("taux_h").alias("attendus_bruts"))
             .join(niveau_jour, [CLE_TERMINAL, "date"]).join(niveau, [CLE_TERMINAL, "date"]))
    hors_silence = F.col("saison") * (F.col("taux_jour") - F.col("attendus_bruts"))
    journee_creuse = F.when(hors_silence > 1, F.least(F.col("n_jour") / hors_silence, F.lit(1.0))).otherwise(1.0)
    pire = Window.partitionBy(CLE_TERMINAL, "date").orderBy(F.desc("retraits_attendus_pendant_silence"))
    return (blocs
            .withColumn("retraits_attendus_pendant_silence",
                        F.col("attendus_bruts") * F.col("saison") * journee_creuse)
            .withColumn("rang", F.row_number().over(pire)).filter(F.col("rang") == 1)
            .withColumn("proba_hasard", F.exp(-F.col("retraits_attendus_pendant_silence")))
            .join(journees.select(CLE_TERMINAL, "date", F.col("retraits").alias("retraits_jour")), [CLE_TERMINAL, "date"])
            .drop("rang", "bloc", "attendus_bruts", "saison"))

In [ ]:
def tranches_delais(retraits_transactions):
    """Répartition des délais entre deux retraits successifs d'un même terminal."""
    bornes = [(10, "1. < 10 s"), (30, "2. 10-30 s"), (60, "3. 30-60 s"), (300, "4. 1-5 min"),
              (900, "5. 5-15 min"), (3600, "6. 15-60 min"), (10800, "7. 1-3 h"), (43200, "8. 3-12 h")]
    tranche = F.when(F.lit(False), F.lit(""))
    for borne, libelle in bornes:
        tranche = tranche.when(F.col("delai_precedente_s") < borne, libelle)
    table = en_pandas(retraits_transactions.filter(F.col("delai_precedente_s").isNotNull())
                      .groupBy(tranche.otherwise("9. >= 12 h").alias("delai")).count().orderBy("delai"))
    table["pct"] = (100 * table["count"] / table["count"].sum()).round(2)
    return table

In [ ]:
def ecart_parc_jour_heure(retraits):
    """Matrice jour x heure du parc : log2(retraits / médiane des mêmes jour de semaine et heure)."""
    parc = en_pandas(retraits.groupBy("date", "heure").count())
    parc["date"] = pd.to_datetime(parc["date"])
    parc = parc.pivot(index="date", columns="heure", values="count").reindex(columns=range(24)).fillna(0).sort_index()
    habitude = parc.groupby(parc.index.dayofweek).transform("median")
    return np.log2((parc + 1) / (habitude + 1)).clip(-2, 2)

In [ ]:
# ------------------------------------------------------------------ compléments pour les graphiques
def tranches_montants(retraits, pas=10, maximum=1500):
    """Nombre de retraits par tranche de `pas` euros (jusqu'à `maximum`)."""
    return en_pandas(retraits.filter(F.col("montant") <= maximum)
                     .groupBy((F.floor(F.col("montant") / pas) * pas).alias("tranche")).count().orderBy("tranche"))

In [ ]:
def montants_frequents(retraits, nb_retraits, nb=25):
    """Montants les plus fréquents et leur part."""
    table = en_pandas(retraits.groupBy("montant").count().orderBy(F.desc("count")).limit(nb))
    table["pct"] = (100 * table["count"] / nb_retraits).round(2)
    return table

In [ ]:
def ecarts_billets(retraits, nb=20):
    """Écarts (montant demandé - billets délivrés) les plus fréquents."""
    return en_pandas(retraits.filter(F.abs(F.col("ecart_billets")) > 0.001)
                     .groupBy(F.round("ecart_billets", 0).alias("ecart")).count()
                     .orderBy(F.desc("count")).limit(nb))

In [ ]:
def repartition_reseaux(retraits, colonne_temps):
    """Part de chaque groupe de réseau par modalité de `colonne_temps` (mois, heure...), en %."""
    table = en_pandas(retraits.groupBy(colonne_temps, "groupe_reseau").count())
    table = table.pivot(index=colonne_temps, columns="groupe_reseau", values="count").fillna(0).sort_index()
    return 100 * table.div(table.sum(axis=1), axis=0)

In [ ]:
def profils_reseaux(retraits, colonne_temps):
    """Profil de chaque réseau selon `colonne_temps` : part des retraits du réseau par modalité."""
    table = en_pandas(retraits.groupBy("groupe_reseau", colonne_temps).count())
    table = table.pivot(index=colonne_temps, columns="groupe_reseau", values="count").fillna(0)
    return table / table.sum()

In [ ]:
def silences_debut_duree(silences_improbables, duree_max=15):
    """Matrice durée (h) x heure de début des silences improbables."""
    table = en_pandas(silences_improbables.groupBy(
        "debut_silence", F.least(F.col("silence_h"), F.lit(duree_max)).alias("duree")).count())
    if table.empty:
        return table
    return table.pivot(index="duree", columns="debut_silence", values="count").fillna(0).sort_index()

In [ ]:
def delais_echelle_log(retraits_transactions):
    """Distribution des délais entre retraits successifs, par pas de 0,1 en log10(secondes)."""
    return en_pandas(retraits_transactions.filter(F.col("delai_precedente_s") > 0)
                     .groupBy(F.round(F.log10("delai_precedente_s"), 1).alias("log10_delai")).count()
                     .orderBy("log10_delai"))

### Code : Règles directes 01 à 11, registre et synthèses
Catalogue des anomalies : registre des règles, règles directes (01 à 11), détection et synthèses.

Ajouter une anomalie = écrire une fonction décorée par `@regle("NN", "Nom")` qui reçoit
(donnees, seuils) et renvoie une table Spark (num_automate, date, valeur), ou None si elle ne s'applique pas.
Les règles « cachées » (12 à 17) sont dans `anomalies_cachees.py`.

In [ ]:
Regle = namedtuple("Regle", ["code", "nom", "fonction"])
REGLES = []
SCHEMA_EVENEMENTS = T.StructType([T.StructField(CLE_TERMINAL, T.StringType()), T.StructField("date", T.DateType()),
                                  T.StructField("type", T.StringType()), T.StructField("valeur", T.DoubleType())])

In [ ]:
def regle(code, nom):
    """Décorateur : enregistre une règle d'anomalie dans le catalogue."""
    def enregistrer(fonction):
        REGLES.append(Regle(code, "{} {}".format(code, nom), fonction))
        return fonction
    return enregistrer

In [ ]:
class DonneesAnalyse(object):
    """Tables partagées par les règles, et mémoire des calculs intermédiaires coûteux (calculés une fois)."""

    def __init__(self, ops, transactions, journees, conf, validite=None, referentiel=None, correspondance=None):
        self.ops = ops
        self.transactions = transactions
        self.journees = journees
        self.conf = conf
        self.validite = validite
        self.referentiel = referentiel
        self.correspondance = correspondance or {}
        self._memoire = {}
        self.journees_signalees = set()   # (num_automate, date) déjà signalées par une règle

    def colonne_presente(self, nom):
        """Vrai si la colonne `nom` existe dans la table d'origine."""
        return bool(self.correspondance.get(nom))

    @property
    def coupures_presentes(self):
        """Vrai si les colonnes nb1-nb4 existent (billets délivrés par cassette)."""
        return all(self.colonne_presente("nb{}".format(i)) for i in range(1, 5))

    def validite_operations(self):
        """Mois restants avant expiration de la carte, pour chaque retrait (clé : terminal, date, rang dans la journée)."""
        return self.memoriser("validite_operations", lambda: lire_validite(
            self.retraits, (CLE_TERMINAL, "date", "rang_dans_journee"), afficher=False))

    @property
    def retraits(self):
        """Transactions de type retrait."""
        return self.transactions.filter(F.col("est_retrait") == 1)

    def memoriser(self, nom, calcul):
        """Renvoie le résultat de `calcul()`, calculé une seule fois puis gardé en mémoire."""
        if nom not in self._memoire:
            self._memoire[nom] = calcul()
        return self._memoire[nom]

    def resultat(self, nom):
        """Résultat déjà calculé sous ce nom (ou None s'il ne l'a pas été)."""
        return self._memoire.get(nom)

    def grille_horaire(self):
        """Retraits par terminal, jour et heure de la plage étudiée (heures vides comprises)."""
        options = self.conf["options"]
        return self.memoriser("grille_horaire", lambda: grille_horaire(
            self.retraits, self.journees, options["heure_debut_jour"], options["heure_fin_jour"],
            options["persister"]))

    def silences(self):
        """Pire silence de chaque journée terminal (voir indicateurs.calculer_silences)."""
        return self.memoriser("silences", lambda: persister(
            calculer_silences(self.grille_horaire(), self.journees), self.conf["options"]["persister"]))

    def groupes_pairs(self):
        """Groupe de pairs de chaque terminal (num_automate, groupe), ou None si la colonne est absente."""
        colonne = self.conf["options"]["groupe_pairs"]
        if self.referentiel is None or colonne not in self.referentiel.columns:
            return None
        return self.referentiel.select(CLE_TERMINAL, F.coalesce(F.col(colonne).cast("string"), F.lit("inconnu"))
                                       .alias("groupe"))

In [ ]:
# ------------------------------------------------------------------ règles directes
@regle("01", "Silence anormal")
def regle_silence(donnees, seuils):
    """Heures consécutives sans retrait, improbables vu le rythme horaire du terminal et l'activité du jour."""
    return (donnees.silences().filter(F.col("proba_hasard") < seuils["silence_proba"])
            .select(CLE_TERMINAL, "date", F.col("silence_h").alias("valeur")))

In [ ]:
@regle("02", "Journée sans opération (terminal très fréquenté)")
def regle_journee_sans_operation(donnees, seuils):
    """Aucune opération un jour où un terminal très fréquenté en fait habituellement."""
    journees = donnees.journees
    mediane = journees.groupBy(CLE_TERMINAL).agg(centile("retraits", 0.5).alias("mediane"))
    seuil = mediane.approxQuantile("mediane", [seuils["muet_quantile_parc"]], 0.001)[0]
    tres_frequentes = mediane.filter(F.col("mediane") >= seuil).select(CLE_TERMINAL)
    print("terminal très fréquentés : au moins {:.0f} retraits par jour -> {} terminal".format(
        seuil, fmt_nombre(tres_frequentes.count())))
    vie = journees.groupBy(CLE_TERMINAL).agg(F.min("date").alias("d0"), F.max("date").alias("d1"))
    habitude = (journees.groupBy(CLE_TERMINAL, F.dayofweek("date").alias("js"))
                .agg(centile("retraits", 0.5).alias("retraits_habituels")))
    calendrier = journees.select("date").distinct().withColumn("js", F.dayofweek("date"))
    attendus = (vie.join(tres_frequentes, CLE_TERMINAL).crossJoin(calendrier)
                .filter((F.col("date") > F.col("d0")) & (F.col("date") < F.col("d1")))
                .join(habitude, [CLE_TERMINAL, "js"])
                .filter(F.col("retraits_habituels") >= seuils["muet_retraits_habituels"]))
    return (attendus.join(journees.select(CLE_TERMINAL, "date"), [CLE_TERMINAL, "date"], "left_anti")
            .select(CLE_TERMINAL, "date", F.col("retraits_habituels").alias("valeur")))

In [ ]:
@regle("03", "Rafale")
def regle_rafale(donnees, seuils):
    """Opérations en 10 minutes très au-dessus du niveau élevé habituel du terminal (99e centile)."""
    transactions = donnees.transactions
    habitude = transactions.groupBy(CLE_TERMINAL).agg(centile("nb_ope_5min", 0.99).alias("p99"))
    return (transactions.join(F.broadcast(habitude), CLE_TERMINAL)
            .filter((F.col("nb_ope_5min") >= seuils["rafale_min_10min"]) &
                    (F.col("nb_ope_5min") >= seuils["rafale_facteur"] * F.col("p99")))
            .groupBy(CLE_TERMINAL, "date").agg(F.max("nb_ope_5min").alias("valeur")))

In [ ]:
@regle("04", "Même montant répété")
def regle_meme_montant(donnees, seuils):
    """Beaucoup de retraits du même montant en quelques minutes."""
    fenetre = (Window.partitionBy(CLE_TERMINAL, "montant").orderBy("t")
               .rangeBetween(0, seuils["meme_montant_fenetre_s"]))
    return (donnees.retraits.withColumn("meme_montant", F.count("*").over(fenetre))
            .filter(F.col("meme_montant") >= seuils["meme_montant_min"])
            .groupBy(CLE_TERMINAL, "date").agg(F.max("meme_montant").alias("valeur")))

In [ ]:
def series_consecutives(retraits, condition):
    """Suites de retraits consécutifs d'un terminal qui remplissent `condition` (longueur = nb de retraits)."""
    fenetre = Window.partitionBy(CLE_TERMINAL).orderBy("t").rowsBetween(Window.unboundedPreceding, 0)
    marquee = (retraits.withColumn("dans_serie", condition.cast("int"))
               .withColumn("serie", F.sum(1 - F.col("dans_serie")).over(fenetre)))
    return (marquee.filter(F.col("dans_serie") == 1).groupBy(CLE_TERMINAL, "serie")
            .agg(F.min("date").alias("date"), (F.count("*") + 1).alias("longueur")))

In [ ]:
@regle("05", "Inaboutis en série")
def regle_inaboutis_serie(donnees, seuils):
    """Retraits inaboutis consécutifs."""
    series = series_consecutives(donnees.retraits, (F.col("inabouti") == 1) & (F.col("inabouti_precedent") == 1))
    return (series.filter(F.col("longueur") >= seuils["inaboutis_serie_min"])
            .select(CLE_TERMINAL, "date", F.col("longueur").alias("valeur")))

In [ ]:
@regle("06", "Captures hors réseau interne en série")
def regle_captures_serie(donnees, seuils):
    """Plusieurs captures hors réseau interne en moins d'une heure."""
    heure_glissante = Window.partitionBy(CLE_TERMINAL).orderBy("t").rangeBetween(-3600, 0)
    return (donnees.transactions.filter(F.col("est_capture_externe") == 1)
            .withColumn("captures_1h", F.count("*").over(heure_glissante))
            .filter(F.col("captures_1h") >= seuils["captures_heure_min"])
            .groupBy(CLE_TERMINAL, "date").agg(F.max("captures_1h").alias("valeur")))

In [ ]:
@regle("07", "Afflux cartes étrangères")
def regle_cartes_etrangeres(donnees, seuils):
    """Pic horaire de cartes étrangères rares, très au-dessus de l'habitude horaire du terminal (90e centile)."""
    par_heure = (donnees.retraits.filter(F.col("est_etranger") == 1)
                 .groupBy(CLE_TERMINAL, "date", "heure").agg(F.count("*").alias("etrangers_heure")))
    habitude = par_heure.groupBy(CLE_TERMINAL).agg(centile("etrangers_heure", 0.9).alias("p90"))
    return (par_heure.join(F.broadcast(habitude), CLE_TERMINAL)
            .filter((F.col("etrangers_heure") >= seuils["etrangers_heure_min"]) &
                    (F.col("etrangers_heure") >= seuils["etrangers_facteur"] * F.col("p90")))
            .groupBy(CLE_TERMINAL, "date").agg(F.max("etrangers_heure").alias("valeur")))

In [ ]:
@regle("08", "Nuit inhabituelle")
def regle_nuit(donnees, seuils):
    """Retraits entre 0 h et 5 h sur un terminal qui n'en fait presque jamais."""
    retraits = donnees.retraits
    part_nuit = retraits.groupBy(CLE_TERMINAL).agg(F.avg((F.col("heure") < 5).cast("int")).alias("part_0h_5h"))
    return (retraits.filter(F.col("heure") < 5).groupBy(CLE_TERMINAL, "date").agg(F.count("*").alias("valeur"))
            .join(F.broadcast(part_nuit), CLE_TERMINAL)
            .filter((F.col("valeur") >= seuils["nuit_min"]) & (F.col("part_0h_5h") <= seuils["nuit_part_max"]))
            .select(CLE_TERMINAL, "date", "valeur"))

In [ ]:
@regle("09", "Carte expirée")
def regle_carte_expiree(donnees, seuils):
    """Retrait avec une carte dont la date de validité est dépassée."""
    if donnees.validite is None:
        return None
    return (donnees.validite.filter(F.col("mois_restants") < 0)
            .groupBy(CLE_TERMINAL, "date").agg(F.count("*").alias("valeur")))

In [ ]:
@regle("10", "Volume journalier anormal")
def regle_volume(donnees, seuils):
    """Retraits du jour multipliés par rapport aux 8 mêmes jours de semaine ET aux 7 derniers jours."""
    meme_jour, derniers_jours = fenetre_meme_jour_semaine(), fenetre_jours_precedents()
    return (donnees.journees
            .withColumn("moy8", F.avg("retraits").over(meme_jour))
            .withColumn("et8", F.stddev("retraits").over(meme_jour))
            .withColumn("n8", F.count("retraits").over(meme_jour))
            .withColumn("moy7", F.avg("retraits").over(derniers_jours))
            .withColumn("valeur", F.col("retraits") / F.col("moy8"))
            .withColumn("ecart", (F.col("retraits") - F.col("moy8")) /
                        F.greatest(F.coalesce(F.col("et8"), F.lit(0.0)), F.sqrt(F.greatest(F.col("moy8"), F.lit(1.0)))))
            .filter((F.col("n8") >= 4) & (F.col("moy8") >= seuils["volume_habitude_min"]) &
                    (F.col("ecart") >= seuils["volume_ecart"]) & (F.col("valeur") >= seuils["volume_ratio"]) &
                    (F.col("retraits") >= seuils["volume_ratio"] * F.col("moy7")))
            .select(CLE_TERMINAL, "date", "valeur"))

In [ ]:
@regle("11", "Montant moyen anormal")
def regle_montant_moyen(donnees, seuils):
    """Montant moyen du jour multiplié par rapport aux 8 mêmes jours de semaine."""
    meme_jour = fenetre_meme_jour_semaine()
    return (donnees.journees.filter(F.col("retraits") >= seuils["montant_retraits_min"])
            .withColumn("montant_moyen", F.col("montant") / F.col("retraits"))
            .withColumn("mm8", F.avg("montant_moyen").over(meme_jour))
            .withColumn("n8", F.count("montant_moyen").over(meme_jour))
            .withColumn("valeur", F.col("montant_moyen") / F.col("mm8"))
            .filter((F.col("n8") >= 4) & (F.col("valeur") >= seuils["montant_ratio"]))
            .select(CLE_TERMINAL, "date", "valeur"))

In [ ]:
# ------------------------------------------------------------------ détection et synthèses
def detecter_anomalies(donnees, seuils, codes=None):
    """Applique les règles du catalogue (toutes, ou celles dont le code est dans `codes`).

    Renvoie une table Spark (num_automate, date, type, valeur, semaine, jour_semaine).
    """
    morceaux = []
    for r in sorted(REGLES, key=lambda r: r.code):
        if codes and r.code not in codes:
            continue
        debut = time.time()
        resultat = r.fonction(donnees, seuils)
        if resultat is None:
            print("{:<48} ignorée (données indisponibles)".format(r.nom))
            continue
        # Chaque règle est calculée et ramenée séparément : unir les 17 plans Spark avant exécution produit
        # un plan trop gros pour l'optimiseur. Les résultats (événements) sont petits.
        evenements_regle = en_pandas(resultat.select(CLE_TERMINAL, F.col("date").cast("date").alias("date"),
                                                     F.col("valeur").cast("double").alias("valeur")))
        evenements_regle["type"] = r.nom
        morceaux.append(evenements_regle)
        donnees.journees_signalees.update(zip(evenements_regle[CLE_TERMINAL].astype(str),
                                              pd.to_datetime(evenements_regle["date"]).dt.date))
        print("{:<48} {:>8} événements  {:>6.0f} s".format(r.nom, len(evenements_regle), time.time() - debut))
    if not morceaux:
        raise ValueError("Aucune règle n'a pu être appliquée.")
    tous = pd.concat(morceaux, ignore_index=True)[[CLE_TERMINAL, "date", "type", "valeur"]]
    tous[CLE_TERMINAL] = tous[CLE_TERMINAL].astype(str)
    spark = SparkSession.builder.getOrCreate()
    evenements = (spark.createDataFrame(tous, schema=SCHEMA_EVENEMENTS)
                  .withColumn("semaine", F.weekofyear("date")).withColumn("jour_semaine", F.dayofweek("date")))
    return persister(evenements, donnees.conf["options"]["persister"])

In [ ]:
def synthese_par_type(evenements, nb_terminal, nb_jours, semaines_recurrence=4):
    """Fréquence, terminal touchés et récurrence de chaque anomalie ; et occurrences par type et par terminal."""
    recap = en_pandas(evenements.groupBy("type").agg(
        F.count("*").alias("evenements"),
        F.countDistinct(CLE_TERMINAL).alias("terminal_touches"),
        F.countDistinct(F.concat_ws("|", F.col(CLE_TERMINAL), F.col("date").cast("string"))).alias("journees_terminal"),
        F.countDistinct("date").alias("jours_concernes")))
    par_type_terminal = en_pandas(evenements.groupBy("type", CLE_TERMINAL).agg(
        F.countDistinct("semaine").alias("semaines"), F.count("*").alias("n")))
    recurrence = par_type_terminal.groupby("type").agg(
        occurrences_medianes_par_terminal=("n", "median"),
        pct_terminal_recurrents=("semaines", lambda semaines: 100 * (semaines >= semaines_recurrence).mean()))
    recap = recap.set_index("type").join(recurrence).sort_index()
    recap["pct_terminal_touches"] = 100 * recap["terminal_touches"] / max(nb_terminal, 1)
    recap["evenements_par_jour"] = recap["evenements"] / max(nb_jours, 1)
    return recap, par_type_terminal

In [ ]:
def cooccurrences(evenements):
    """% des journées terminal de l'anomalie (ligne) qui présentent aussi l'anomalie (colonne)."""
    journees_type = evenements.select(CLE_TERMINAL, "date", "type").distinct()
    paires = en_pandas(journees_type.alias("a").join(journees_type.alias("b"), [CLE_TERMINAL, "date"])
                       .groupBy(F.col("a.type").alias("type_a"), F.col("b.type").alias("type_b")).count())
    matrice = paires.pivot(index="type_a", columns="type_b", values="count").fillna(0)
    diagonale = pd.Series([matrice.loc[t, t] for t in matrice.index], index=matrice.index)
    matrice = matrice.div(diagonale, axis=0) * 100
    for t in matrice.index:
        matrice.loc[t, t] = 0
    return matrice

In [ ]:
def synthese_par_terminal(par_type_terminal, par_terminal, infos=("ville", "libelle_site", "retraits_par_jour_actif")):
    """Une ligne par terminal touché : nombre d'anomalies différentes, d'événements, et le détail par type."""
    occurrences = par_type_terminal.pivot(index=CLE_TERMINAL, columns="type", values="n").fillna(0)
    synthese = occurrences.copy()
    synthese["nb_anomalies_differentes"] = (occurrences > 0).sum(axis=1)
    synthese["evenements"] = occurrences.sum(axis=1)
    synthese["detail"] = occurrences.apply(
        lambda ligne: ", ".join("{} : {:.0f}".format(nom[3:], n) for nom, n in ligne.items() if n > 0), axis=1)
    colonnes = [c for c in infos if c in par_terminal.columns]
    synthese = synthese.join(par_terminal.set_index(CLE_TERMINAL)[colonnes])
    return synthese.sort_values(["nb_anomalies_differentes", "evenements"], ascending=False), occurrences

In [ ]:
def exemples_par_type(evenements):
    """Le cas le plus marqué (|valeur| maximale) de chaque anomalie."""
    plus_marque = Window.partitionBy("type").orderBy(F.desc("force"))
    return en_pandas(evenements.withColumn("force", F.abs(F.col("valeur")))
                     .withColumn("rang", F.row_number().over(plus_marque)).filter(F.col("rang") == 1)
                     .select("type", CLE_TERMINAL, "date", "valeur").orderBy("type"))

In [ ]:
def aide_au_choix(recap, part_trop_frequente=60, journees_trop_rares=20):
    """Classement indicatif des anomalies : fréquence et récurrence, avec repérage des extrêmes."""
    choix = recap[["journees_terminal", "pct_terminal_touches", "pct_terminal_recurrents", "evenements_par_jour"]].copy()
    choix["trop_frequente"] = choix["pct_terminal_touches"] > part_trop_frequente
    choix["trop_rare"] = choix["journees_terminal"] < journees_trop_rares
    choix["rang"] = (choix["journees_terminal"].rank(ascending=False)
                     + choix["pct_terminal_recurrents"].rank(ascending=False))
    return choix.sort_values("rang")

### Code : Règles cachées 12 à 17
Anomalies cachées (règles 12 à 17) : ce qu'aucun indicateur seul ne montre.

12  forme horaire de la journée inhabituelle (même si le total est normal)
13  mélange des réseaux de cartes inhabituel (même à volume normal)
14  décrochage ou emballement par rapport aux terminaux pairs le même jour
15  dérive lente sur plusieurs semaines, propre au terminal
16  trou horaire récurrent : une heure habituellement active reste vide un jour sur deux
17  combinaison de petits écarts simultanés (Isolation Forest sur les écarts robustes)

In [ ]:
INDICATEURS_MULTIVARIES = {
    # indicateur : plancher de dispersion ("racine" = bruit de comptage, sinon part de la moyenne)
    "retraits": "racine", "inaboutis": "racine", "captures_externe": "racine", "etrangers": "racine",
    "retraits_nuit": "racine", "rafale": "racine", "montant_moyen": 0.1,
}

In [ ]:
# ------------------------------------------------------------------ divergences de distribution
def divergence_jensen_shannon(observe, habituel, cles_jour, cles_habitude, modalite):
    """Divergence de Jensen-Shannon (log2, entre 0 et 1) entre la distribution du jour et l'habituelle.

    `observe` : cles_jour + [modalite, n] ; `habituel` : cles_habitude + [modalite, q] avec q sommant à 1.
    Les modalités absentes du jour comptent pour 0,5 x q (terme q.log2(q / (q/2)) / 2).
    """
    total = observe.groupBy(*cles_jour).agg(F.sum("n").alias("total"))
    jointe = (observe.join(total, cles_jour).withColumn("p", F.col("n") / F.col("total"))
              .join(habituel, cles_habitude + [modalite], "left").fillna(0.0, subset=["q"])
              .withColumn("m", (F.col("p") + F.col("q")) / 2))
    terme = (0.5 * F.col("p") * F.log2(F.col("p") / F.col("m"))
             + F.when(F.col("q") > 0, 0.5 * F.col("q") * F.log2(F.col("q") / F.col("m"))).otherwise(0.0))
    return (jointe.groupBy(*(cles_jour + ["total"]))
            .agg(F.sum(terme).alias("somme"), F.sum("q").alias("q_vue"))
            .withColumn("divergence", F.col("somme") + 0.5 * (1 - F.col("q_vue")))
            .drop("somme", "q_vue"))

In [ ]:
def divergences_horaires(retraits):
    """Divergence de la forme horaire de chaque journée terminal par rapport à l'habitude (semaine / week-end)."""
    avec_type_jour = retraits.withColumn("week_end", F.dayofweek("date").isin(1, 7).cast("int"))
    observe = avec_type_jour.groupBy(CLE_TERMINAL, "date", "week_end", "heure").agg(F.count("*").alias("n"))
    habitude = observe.groupBy(CLE_TERMINAL, "week_end", "heure").agg(F.sum("n").alias("n_h"))
    habitude = (habitude.join(habitude.groupBy(CLE_TERMINAL, "week_end").agg(F.sum("n_h").alias("n_tot")),
                              [CLE_TERMINAL, "week_end"])
                .select(CLE_TERMINAL, "week_end", "heure", (F.col("n_h") / F.col("n_tot")).alias("q")))
    return divergence_jensen_shannon(observe, habitude, [CLE_TERMINAL, "date", "week_end"], [CLE_TERMINAL, "week_end"],
                                     "heure")

In [ ]:
def divergences_reseaux(retraits):
    """Divergence du mélange des réseaux de chaque journée terminal par rapport à l'habitude du terminal."""
    observe = retraits.groupBy(CLE_TERMINAL, "date", "groupe_reseau").agg(F.count("*").alias("n"))
    habitude = observe.groupBy(CLE_TERMINAL, "groupe_reseau").agg(F.sum("n").alias("n_r"))
    habitude = (habitude.join(habitude.groupBy(CLE_TERMINAL).agg(F.sum("n_r").alias("n_tot")), CLE_TERMINAL)
                .select(CLE_TERMINAL, "groupe_reseau", (F.col("n_r") / F.col("n_tot")).alias("q")))
    return divergence_jensen_shannon(observe, habitude, [CLE_TERMINAL, "date"], [CLE_TERMINAL], "groupe_reseau")

In [ ]:
def journees_divergentes(divergences, volume_min, nb_ecarts, divergence_min):
    """Journées dont la divergence dépasse la médiane du terminal + `nb_ecarts` écarts robustes (et un minimum)."""
    suffisantes = divergences.filter(F.col("total") >= volume_min)
    mediane = suffisantes.groupBy(CLE_TERMINAL).agg(centile("divergence", 0.5).alias("med"))
    avec_med = suffisantes.join(mediane, CLE_TERMINAL)
    mad = (avec_med.withColumn("dev", F.abs(F.col("divergence") - F.col("med")))
           .groupBy(CLE_TERMINAL).agg(centile("dev", 0.5).alias("mad")))
    seuil = F.greatest(F.lit(float(divergence_min)), F.col("med") + nb_ecarts * 1.4826 * F.col("mad"))
    return (avec_med.join(mad, CLE_TERMINAL).filter(F.col("divergence") >= seuil)
            .select(CLE_TERMINAL, "date", F.col("divergence").alias("valeur")))

In [ ]:
@regle("12", "Forme horaire inhabituelle")
def regle_profil_horaire(donnees, seuils):
    """Répartition des retraits dans la journée très différente de l'habitude du terminal."""
    return journees_divergentes(donnees_divergences(donnees, "horaires"), seuils["profil_volume_min"],
                                seuils["profil_ecarts_robustes"], seuils["profil_divergence_min"])

In [ ]:
@regle("13", "Mélange de réseaux inhabituel")
def regle_melange_reseaux(donnees, seuils):
    """Part des réseaux de cartes du jour très différente de l'habitude du terminal."""
    return journees_divergentes(donnees_divergences(donnees, "reseaux"), seuils["reseaux_volume_min"],
                                seuils["reseaux_ecarts_robustes"], seuils["reseaux_divergence_min"])

In [ ]:
# ------------------------------------------------------------------ comparaison aux pairs
def ratios_habitude(journees, habitude_min):
    """Retraits du jour / moyenne des 8 mêmes jours de semaine précédents."""
    meme_jour = fenetre_meme_jour_semaine()
    return (journees.withColumn("moy8", F.avg("retraits").over(meme_jour))
            .withColumn("n8", F.count("retraits").over(meme_jour))
            .filter((F.col("n8") >= 4) & (F.col("moy8") >= habitude_min))
            .withColumn("ratio", F.col("retraits") / F.col("moy8")))

In [ ]:
@regle("14", "Écart aux terminaux pairs")
def regle_ecart_pairs(donnees, seuils):
    """Le terminal décroche (ou s'emballe) alors que les terminaux de son groupe restent normaux ce jour-là."""
    groupes = donnees.groupes_pairs()
    if groupes is None:
        return None
    ratios = ratios_habitude(donnees.journees, seuils["pairs_habitude_min"]).join(groupes, CLE_TERMINAL)
    pairs = ratios.groupBy("groupe", "date").agg(centile("ratio", 0.5).alias("ratio_pairs"),
                                                 F.count("*").alias("nb_pairs"))
    decroche = (F.col("ratio") <= seuils["pairs_ratio_bas"]) & (F.col("ratio_pairs") >= seuils["pairs_ratio_normal"])
    emballe = ((F.col("ratio") >= seuils["pairs_ratio_haut"]) &
               (F.col("ratio_pairs") <= seuils["pairs_ratio_haut_pairs"]))
    return (ratios.join(pairs, ["groupe", "date"])
            .filter((F.col("nb_pairs") >= seuils["pairs_nb_min"]) & (decroche | emballe))
            .select(CLE_TERMINAL, "date", F.col("ratio").alias("valeur")))

In [ ]:
@regle("15", "Dérive lente")
def regle_derive_lente(donnees, seuils):
    """Les 4 dernières semaines tombent sous 60 % des 8 précédentes, sans arrêt net, alors que les pairs tiennent."""
    semaine = F.date_trunc("week", F.col("date").cast("timestamp")).cast("date")
    completes = (donnees.journees.select("date").distinct().groupBy(semaine.alias("semaine")).count()
                 .filter(F.col("count") == 7).select("semaine"))
    hebdo = (donnees.journees.groupBy(CLE_TERMINAL, semaine.alias("semaine")).agg(F.sum("retraits").alias("ret"))
             .join(completes, "semaine"))
    recentes = Window.partitionBy(CLE_TERMINAL).orderBy("semaine").rowsBetween(-3, 0)
    reference = Window.partitionBy(CLE_TERMINAL).orderBy("semaine").rowsBetween(-11, -4)
    hebdo = (hebdo.withColumn("moy_rec", F.avg("ret").over(recentes))
             .withColumn("min_rec", F.min("ret").over(recentes))
             .withColumn("moy_ref", F.avg("ret").over(reference))
             .withColumn("n_ref", F.count("ret").over(reference))
             .filter((F.col("n_ref") >= 8) & (F.col("moy_ref") >= seuils["derive_reference_min"]))
             .withColumn("ratio", F.col("moy_rec") / F.col("moy_ref")))
    candidats = hebdo.filter((F.col("ratio") <= seuils["derive_ratio"]) &
                             (F.col("min_rec") / F.col("moy_ref") >= seuils["derive_semaine_min"]))
    groupes = donnees.groupes_pairs()
    if groupes is not None:
        pairs = (hebdo.join(groupes, CLE_TERMINAL).groupBy("groupe", "semaine")
                 .agg(centile("ratio", 0.5).alias("ratio_pairs")))
        candidats = (candidats.join(groupes, CLE_TERMINAL).join(pairs, ["groupe", "semaine"])
                     .filter(F.col("ratio_pairs") >= seuils["derive_pairs_min"]))
    premiere = Window.partitionBy(CLE_TERMINAL).orderBy("semaine")
    return (candidats.withColumn("rang", F.row_number().over(premiere)).filter(F.col("rang") == 1)
            .select(CLE_TERMINAL, F.col("semaine").alias("date"), F.col("ratio").alias("valeur")))

In [ ]:
@regle("16", "Trou horaire récurrent")
def regle_trou_horaire(donnees, seuils):
    """Une heure habituellement active reste vide plus d'un jour sur deux dans le mois.

    L'attendu de l'heure ce mois-là = rythme annuel de l'heure x (activité du mois / activité moyenne de l'année),
    pour ne pas signaler la basse saison d'un terminal saisonnier.
    """
    grille = donnees.grille_horaire().withColumn("mois", F.trunc("date", "month"))
    taux = grille.groupBy(CLE_TERMINAL, "heure").agg(F.avg("n").alias("taux_annuel"))
    niveau_mois = grille.groupBy(CLE_TERMINAL, "mois").agg(F.avg("n").alias("niveau_mois"))
    niveau_annee = grille.groupBy(CLE_TERMINAL).agg(F.avg("n").alias("niveau_annee"))
    mensuel = (grille.groupBy(CLE_TERMINAL, "heure", "mois")
               .agg(F.count("*").alias("jours"), F.sum((F.col("n") == 0).cast("int")).alias("jours_vides"))
               .join(taux, [CLE_TERMINAL, "heure"]).join(niveau_mois, [CLE_TERMINAL, "mois"]).join(niveau_annee, CLE_TERMINAL)
               .withColumn("attendu", F.col("taux_annuel") * F.col("niveau_mois") / F.col("niveau_annee"))
               .withColumn("part_vide", F.col("jours_vides") / F.col("jours")))
    return (mensuel.filter((F.col("jours") >= seuils["trou_jours_min"]) &
                           (F.col("part_vide") >= seuils["trou_part_jours_vides"]) &
                           (F.col("attendu") >= seuils["trou_taux_horaire_min"]))
            .groupBy(CLE_TERMINAL, "mois").agg(F.max("part_vide").alias("valeur"))
            .select(CLE_TERMINAL, F.col("mois").alias("date"), "valeur"))

In [ ]:
# ------------------------------------------------------------------ combinaisons de petits écarts
def donnees_divergences(donnees, nature):
    """Divergences horaires ou réseaux, calculées une seule fois et partagées entre les règles."""
    calculs = {"horaires": divergences_horaires, "reseaux": divergences_reseaux}
    return donnees.memoriser("divergences_" + nature, lambda: calculs[nature](donnees.retraits))

In [ ]:
def ecarts_divergence(divergences, nom_colonne, dispersion_min=0.01):
    """Écart robuste de la divergence de chaque journée à la médiane du terminal : (div - médiane) / (1,4826 MAD)."""
    mediane = divergences.groupBy(CLE_TERMINAL).agg(centile("divergence", 0.5).alias("med"))
    avec_med = divergences.join(mediane, CLE_TERMINAL).withColumn("dev", F.abs(F.col("divergence") - F.col("med")))
    mad = avec_med.groupBy(CLE_TERMINAL).agg(centile("dev", 0.5).alias("mad"))
    echelle = F.greatest(1.4826 * F.col("mad"), F.lit(dispersion_min))
    return (avec_med.join(mad, CLE_TERMINAL)
            .select(CLE_TERMINAL, "date", ((F.col("divergence") - F.col("med")) / echelle).alias(nom_colonne)))

In [ ]:
def caracteristiques_journalieres(donnees):
    """Écarts robustes de chaque journée terminal à son habitude (8 mêmes jours de semaine), en pandas.

    Colonnes z_<indicateur> pour INDICATEURS_MULTIVARIES, plus les divergences horaire et réseaux centrées.
    """
    meme_jour = fenetre_meme_jour_semaine()
    rafale = donnees.transactions.groupBy(CLE_TERMINAL, "date").agg(F.max("nb_ope_5min").alias("rafale"))
    table = (donnees.journees.join(rafale, [CLE_TERMINAL, "date"], "left").fillna(0, subset=["rafale"])
             .withColumn("montant_moyen", F.col("montant") / F.greatest(F.col("retraits"), F.lit(1)))
             .withColumn("n8", F.count("retraits").over(meme_jour)))
    colonnes_z = []
    for indicateur, plancher in INDICATEURS_MULTIVARIES.items():
        moyenne = F.avg(indicateur).over(meme_jour)
        dispersion = F.coalesce(F.stddev(indicateur).over(meme_jour), F.lit(0.0))
        minimum = (F.sqrt(F.greatest(moyenne, F.lit(1.0))) if plancher == "racine"
                   else F.greatest(plancher * moyenne, F.lit(1.0)))
        table = table.withColumn("z_" + indicateur, (F.col(indicateur) - moyenne) / F.greatest(dispersion, minimum))
        colonnes_z.append("z_" + indicateur)
    for nom, divergences in (("profil", donnees_divergences(donnees, "horaires")),
                             ("reseaux", donnees_divergences(donnees, "reseaux"))):
        table = table.join(ecarts_divergence(divergences, "z_" + nom), [CLE_TERMINAL, "date"], "left")
        colonnes_z.append("z_" + nom)
    resultat = en_pandas(table.filter(F.col("n8") >= 4).select(CLE_TERMINAL, "date", *colonnes_z), max_lignes=20000000)
    resultat[colonnes_z] = resultat[colonnes_z].fillna(0.0).clip(-10, 10)
    return resultat

In [ ]:
def scores_multivaries(donnees, seuils):
    """Score d'atypicité de chaque journée terminal (Isolation Forest) et les 3 indicateurs les plus en cause."""
    try:
        from sklearn.ensemble import IsolationForest
    except ImportError:
        print("scikit-learn indisponible dans ce kernel : règle 17 ignorée.")
        return None
    table = caracteristiques_journalieres(donnees)
    colonnes = [c for c in table.columns if c.startswith("z_")]
    echantillon = table.sample(min(len(table), seuils["multivarie_echantillon"]), random_state=0)
    try:
        modele = IsolationForest(n_estimators=200, contamination="auto", behaviour="new", random_state=0)
    except TypeError:
        modele = IsolationForest(n_estimators=200, contamination="auto", random_state=0)
    modele.fit(echantillon[colonnes].values)
    table["score"] = -modele.score_samples(table[colonnes].values)
    valeurs = np.abs(table[colonnes].values)
    ordre = np.argsort(-valeurs, axis=1)[:, :3]
    noms = np.array([c[2:] for c in colonnes])
    table["explication"] = [", ".join("{} ({:+.1f})".format(noms[j], table[colonnes[j]].iat[i]) for j in ligne)
                            for i, ligne in enumerate(ordre)]
    return table

In [ ]:
@regle("17", "Combinaison de petits écarts")
def regle_multivariee(donnees, seuils):
    """Journées dont la combinaison d'écarts est la plus rare, parmi celles qu'aucune autre règle n'a signalées."""
    scores = donnees.memoriser("scores_multivaries", lambda: scores_multivaries(donnees, seuils))
    if scores is None:
        return None
    deja_vues = [(str(g), d) in donnees.journees_signalees
                 for g, d in zip(scores[CLE_TERMINAL], pd.to_datetime(scores["date"]).dt.date)]
    candidates = scores[~pd.Series(deja_vues, index=scores.index)]
    retenues = candidates.nlargest(max(1, int(len(scores) * seuils["multivarie_part"])), "score")
    spark = SparkSession.builder.getOrCreate()
    return spark.createDataFrame(pd.DataFrame({CLE_TERMINAL: retenues[CLE_TERMINAL].astype(str).values,
                                               "date": pd.to_datetime(retenues["date"]).dt.date.values,
                                               "valeur": retenues["score"].astype(float).values}))

### Code : Règles avancées 18 à 26
Anomalies avancées (règles 18 à 26) : matériel, zone géographique, fraude coordonnée, réseaux, comportements.

18  cassette vide ou défaillante : une coupure disparaît des billets délivrés
19  incident de zone : plusieurs terminaux d'une même zone muets à la même heure (une seule cause commune)
20  attaque coordonnée : activité suspecte sur plusieurs terminaux d'une même zone dans la même heure
21  motif de capture inhabituel : pic de cartes volées, de codes faux ou d'incidents de lecture
22  inaboutis concentrés sur un réseau : un réseau échoue alors que les autres passent
23  rythme trop régulier : retraits espacés de façon quasi constante (script, test)
24  activité à une heure habituellement morte pour ce terminal
25  même carte sur plusieurs terminaux en peu de temps (nécessite un identifiant de carte anonymisé)
26  écart à l'année précédente (nécessite plus d'un an de données)

In [ ]:
def zones(donnees):
    """Zone de chaque terminal (num_automate, zone) selon l'option « zone » : departement, code_postal ou ville."""
    referentiel, choix = donnees.referentiel, donnees.conf["options"]["zone"]
    if referentiel is None:
        return None
    if choix == "departement" and "code_postal" in referentiel.columns:
        code = F.lpad(F.regexp_replace(F.col("code_postal").cast("string"), "[^0-9AB]", ""), 5, "0")
        zone = F.when(code.startswith("97") | code.startswith("98"), code.substr(1, 3)).otherwise(code.substr(1, 2))
    elif choix in referentiel.columns:
        zone = F.col(choix).cast("string")
    else:
        return None
    return referentiel.select(CLE_TERMINAL, zone.alias("zone")).filter(F.col("zone").isNotNull())

In [ ]:
# ------------------------------------------------------------------ 18 : cassettes
@regle("18", "Cassette vide ou défaillante")
def regle_cassette(donnees, seuils):
    """Une coupure habituellement délivrée disparaît (ou presque) des billets délivrés dans la journée."""
    if not donnees.coupures_presentes:
        return None
    retraits = donnees.ops.filter(F.col("est_retrait") == 1)
    colonnes = ["nb{}".format(i) for i in range(1, 5)]
    par_jour = retraits.groupBy(CLE_TERMINAL, "date").agg(
        F.count("*").alias("retraits"), *[F.sum(F.col("billets_" + c)).alias(c) for c in colonnes])
    total = sum(F.col(c) for c in colonnes)
    habitude = par_jour.groupBy(CLE_TERMINAL).agg(*[(F.sum(c) / F.sum(total)).alias("part_" + c) for c in colonnes])
    jour = par_jour.join(habitude, CLE_TERMINAL).filter(F.col("retraits") >= seuils["cassette_retraits_min"])
    alertes = None
    for c in colonnes:
        disparue = ((F.col("part_" + c) >= seuils["cassette_part_habituelle_min"]) &
                    (F.col(c) / F.greatest(total, F.lit(1))
                     <= seuils["cassette_part_restante_max"] * F.col("part_" + c)))
        alertes = disparue if alertes is None else alertes | disparue
    return jour.filter(alertes).select(CLE_TERMINAL, "date", F.col("retraits").alias("valeur"))

In [ ]:
# ------------------------------------------------------------------ 19 et 20 : zone géographique
@regle("19", "Incident de zone")
def regle_incident_zone(donnees, seuils):
    """Plusieurs terminaux d'une même zone dans un silence improbable à la même heure (cause commune).

    Un terminal compte comme muet à une heure s'il est dans un silence de probabilité < `zone_silence_proba`
    (règle 01 : rythme horaire du terminal recalé sur l'activité du jour, donc insensible à la saison).
    """
    groupes = zones(donnees)
    if groupes is None:
        return None
    spark = SparkSession.builder.getOrCreate()
    heures = spark.range(0, 24).select(F.col("id").cast("int").alias("heure"))
    muets = (donnees.silences().filter(F.col("proba_hasard") < seuils["zone_silence_proba"])
             .crossJoin(heures)
             .filter((F.col("heure") >= F.col("debut_silence")) &
                     (F.col("heure") < F.col("debut_silence") + F.col("silence_h")))
             .select(CLE_TERMINAL, "date", "heure").join(groupes, CLE_TERMINAL))
    simultanes = (muets.groupBy("zone", "date", "heure").agg(F.countDistinct(CLE_TERMINAL).alias("terminal_muets"))
                  .filter(F.col("terminal_muets") >= seuils["zone_terminal_min"]))
    return (muets.join(simultanes, ["zone", "date", "heure"])
            .groupBy(CLE_TERMINAL, "date").agg(F.max("terminal_muets").alias("valeur")))

In [ ]:
def activite_suspecte_horaire(donnees, seuils):
    """terminal x jour x heure présentant une rafale, un montant répété ou un pic de cartes étrangères rares."""
    def calcul():
        meme_montant = (Window.partitionBy(CLE_TERMINAL, "montant").orderBy("t")
                        .rangeBetween(0, seuils["meme_montant_fenetre_s"]))
        return (donnees.retraits.withColumn("meme_montant", F.count("*").over(meme_montant))
                .groupBy(CLE_TERMINAL, "date", "heure")
                .agg(F.max("nb_ope_5min").alias("rafale"), F.max("meme_montant").alias("meme_montant"),
                     F.sum("est_etranger").alias("etrangers"))
                .filter((F.col("rafale") >= seuils["coordonnee_rafale_min"]) |
                        (F.col("meme_montant") >= seuils["coordonnee_meme_montant_min"]) |
                        (F.col("etrangers") >= seuils["coordonnee_etrangers_min"])))
    return donnees.memoriser("activite_suspecte_horaire", calcul)

In [ ]:
@regle("20", "Attaque coordonnée")
def regle_attaque_coordonnee(donnees, seuils):
    """Activité suspecte (rafale, montant répété, cartes étrangères) sur plusieurs terminaux d'une zone dans l'heure."""
    groupes = zones(donnees)
    if groupes is None:
        return None
    suspects = activite_suspecte_horaire(donnees, seuils).join(groupes, CLE_TERMINAL)
    simultanes = (suspects.groupBy("zone", "date", "heure").agg(F.countDistinct(CLE_TERMINAL).alias("terminal_suspects"))
                  .filter(F.col("terminal_suspects") >= seuils["coordonnee_terminal_min"]))
    return (suspects.join(simultanes, ["zone", "date", "heure"])
            .groupBy(CLE_TERMINAL, "date").agg(F.max("terminal_suspects").alias("valeur")))

In [ ]:
# ------------------------------------------------------------------ 21 et 22 : captures et réseaux
@regle("21", "Motif de capture inhabituel")
def regle_motif_capture(donnees, seuils):
    """Captures d'un même motif dans la journée, improbables vu l'habitude du terminal pour ce motif (Poisson)."""
    captures = donnees.ops.filter((F.col("est_capture") == 1) &
                                  F.col("motif_capture").isin(seuils["motifs_surveilles"]))
    par_jour = captures.groupBy(CLE_TERMINAL, "date", "motif_capture").agg(F.count("*").alias("n"))
    nb_jours = donnees.journees.groupBy(CLE_TERMINAL).agg(F.count("*").alias("jours"))
    habitude = (par_jour.groupBy(CLE_TERMINAL, "motif_capture").agg(F.sum("n").alias("total"))
                .join(nb_jours, CLE_TERMINAL).withColumn("par_jour", F.col("total") / F.col("jours")))
    # probabilité d'observer au moins n captures de ce motif par hasard (Poisson, premier terme de la queue)
    lam = F.greatest(F.col("par_jour"), F.lit(1e-3))
    proba = F.pow(lam, F.col("n")) * F.exp(-lam) / F.factorial(F.col("n").cast("int"))
    return (par_jour.join(habitude, [CLE_TERMINAL, "motif_capture"])
            .filter((F.col("n") >= seuils["motif_min"]) & (proba < seuils["motif_proba"]))
            .groupBy(CLE_TERMINAL, "date").agg(F.max("n").alias("valeur")))

In [ ]:
@regle("22", "Inaboutis concentrés sur un réseau")
def regle_inaboutis_reseau(donnees, seuils):
    """Les retraits d'un réseau échouent massivement alors que ceux des autres réseaux passent."""
    retraits = donnees.ops.filter(F.col("est_retrait") == 1)
    par_reseau = retraits.groupBy(CLE_TERMINAL, "date", "groupe_reseau").agg(
        F.count("*").alias("n"), F.avg("inabouti").alias("taux"))
    habitude = retraits.groupBy(CLE_TERMINAL, "groupe_reseau").agg(F.avg("inabouti").alias("taux_habituel"))
    autres = (par_reseau.alias("a").join(par_reseau.alias("b"), [CLE_TERMINAL, "date"])
              .filter(F.col("a.groupe_reseau") != F.col("b.groupe_reseau"))
              .groupBy(CLE_TERMINAL, "date", F.col("a.groupe_reseau").alias("groupe_reseau"))
              .agg((F.sum(F.col("b.taux") * F.col("b.n")) / F.sum("b.n")).alias("taux_autres")))
    return (par_reseau.join(habitude, [CLE_TERMINAL, "groupe_reseau"]).join(autres, [CLE_TERMINAL, "date", "groupe_reseau"])
            .filter((F.col("n") >= seuils["reseau_retraits_min"]) &
                    (F.col("taux") >= seuils["reseau_taux_min"]) &
                    (F.col("taux") >= seuils["reseau_facteur"] * F.greatest(F.col("taux_habituel"), F.lit(0.01))) &
                    (F.col("taux_autres") <= seuils["reseau_taux_autres_max"]))
            .groupBy(CLE_TERMINAL, "date").agg(F.max("taux").alias("valeur")))

In [ ]:
# ------------------------------------------------------------------ 23 et 24 : comportements
@regle("23", "Rythme trop régulier")
def regle_rythme_regulier(donnees, seuils):
    """Le même délai entre deux retraits (à la seconde près) revient bien plus souvent que le hasard ne le permet.

    Un humain ne retire pas à intervalle constant ; un script ou un test, si. Dans une fenêtre glissante, on
    compare le nombre de délais identiques au nombre attendu si les arrivées étaient aléatoires (loi
    exponentielle des délais au rythme local), puis on calcule la probabilité de hasard (Poisson). Les terminaux
    très chargés, où les délais courts se répètent naturellement, ne sont donc pas signalés.
    """
    fenetre = seuils["regulier_fenetre_s"]
    autour = Window.partitionBy(CLE_TERMINAL).orderBy("t").rangeBetween(-fenetre, fenetre)
    meme_delai = Window.partitionBy(CLE_TERMINAL, "delai").orderBy("t").rangeBetween(-fenetre, fenetre)
    rythme = F.col("retraits_autour") / F.lit(2.0 * fenetre)
    attendu = F.greatest(F.col("retraits_autour") * rythme * F.exp(-rythme * F.col("delai")), F.lit(1e-3))
    k = F.col("repetitions")
    # factorielle plafonnée à 20! (limite Spark) : la probabilité est alors surestimée, donc prudente
    proba = F.exp(k * F.log(attendu) - attendu - F.log(F.factorial(F.least(k, F.lit(20)).cast("int"))))
    return (donnees.retraits
            .withColumn("delai", F.col("delai_precedente_s").cast("int"))
            .withColumn("retraits_autour", F.count("*").over(autour))
            .filter(F.col("delai").between(seuils["regulier_delai_min_s"], seuils["regulier_delai_max_s"]))
            .withColumn("repetitions", F.count("*").over(meme_delai))
            .filter((k >= seuils["regulier_nb_retraits"]) & (proba < seuils["regulier_proba"]))
            .groupBy(CLE_TERMINAL, "date").agg(F.max("repetitions").alias("valeur")))

In [ ]:
@regle("24", "Activité à une heure morte")
def regle_heure_morte(donnees, seuils):
    """Plusieurs retraits à une heure où le terminal n'opère presque jamais (ex. agence fermée)."""
    retraits = donnees.ops.filter(F.col("est_retrait") == 1)
    par_heure = retraits.groupBy(CLE_TERMINAL, "date", "heure").agg(F.count("*").alias("n"))
    nb_jours = donnees.journees.groupBy(CLE_TERMINAL).agg(F.count("*").alias("jours"))
    habitude = (par_heure.groupBy(CLE_TERMINAL, "heure").agg(F.count("*").alias("jours_actifs_heure"))
                .join(nb_jours, CLE_TERMINAL).withColumn("part_jours", F.col("jours_actifs_heure") / F.col("jours")))
    resultat = (par_heure.join(habitude, [CLE_TERMINAL, "heure"])
                .filter((F.col("n") >= seuils["heure_morte_retraits_min"]) &
                        (F.col("part_jours") <= seuils["heure_morte_part_max"]) & (F.col("jours") >= 60)))
    sites = donnees.conf["options"]["sites_interieurs"]
    if sites and donnees.referentiel is not None and "libelle_site" in donnees.referentiel.columns:
        resultat = resultat.join(donnees.referentiel.filter(F.col("libelle_site").isin(sites)).select(CLE_TERMINAL),
                                 CLE_TERMINAL)
    return resultat.groupBy(CLE_TERMINAL, "date").agg(F.max("n").alias("valeur"))

In [ ]:
# ------------------------------------------------------------------ 25 et 26 : données supplémentaires
@regle("25", "Même carte sur plusieurs terminaux")
def regle_meme_carte(donnees, seuils):
    """Une carte utilisée sur plusieurs terminaux en peu de temps, ou à une vitesse de déplacement impossible."""
    if not donnees.colonne_presente("id_carte") or donnees.referentiel is None or \
            "latitude" not in donnees.referentiel.columns:
        return None
    position = donnees.referentiel.select(CLE_TERMINAL, "longitude", "latitude")
    par_carte = Window.partitionBy("id_carte").orderBy("t")
    retraits = (donnees.retraits.filter(F.col("id_carte").isNotNull()).join(position, CLE_TERMINAL)
                .withColumn("terminal_suivant", F.lead(CLE_TERMINAL).over(par_carte))
                .withColumn("t_suivant", F.lead("t").over(par_carte))
                .withColumn("lat2", F.lead("latitude").over(par_carte))
                .withColumn("lon2", F.lead("longitude").over(par_carte)))
    distance_km = 111.0 * F.sqrt(F.pow(F.col("lat2") - F.col("latitude"), 2) +
                                 F.pow((F.col("lon2") - F.col("longitude")) * F.cos(F.radians("latitude")), 2))
    vitesse = distance_km / F.greatest((F.col("t_suivant") - F.col("t")) / 3600.0, F.lit(1.0 / 60))
    return (retraits.filter((F.col("terminal_suivant") != F.col(CLE_TERMINAL)) &
                            (vitesse >= seuils["carte_vitesse_max_kmh"]))
            .groupBy(CLE_TERMINAL, "date").agg(F.max(vitesse).alias("valeur")))

In [ ]:
@regle("26", "Écart à l'année précédente")
def regle_annee_precedente(donnees, seuils):
    """Retraits du jour très différents de ceux des mêmes jours de semaine, un an plus tôt (52 semaines)."""
    journees = donnees.journees
    bornes = journees.agg(F.min("date"), F.max("date")).collect()[0]
    if (bornes[1] - bornes[0]).days < 400:
        return None
    il_y_a_un_an = journees.select(CLE_TERMINAL, F.date_add("date", 364).alias("date"), F.col("retraits").alias("n1"))
    meme_jour = fenetre_meme_jour_semaine(4)
    return (journees.join(il_y_a_un_an, [CLE_TERMINAL, "date"])
            .withColumn("n1_moyen", F.avg("n1").over(meme_jour))
            .filter(F.col("n1_moyen") >= seuils["annee_habitude_min"])
            .withColumn("valeur", F.col("retraits") / F.col("n1_moyen"))
            .filter((F.col("valeur") >= seuils["annee_ratio"]) | (F.col("valeur") <= 1.0 / seuils["annee_ratio"]))
            .select(CLE_TERMINAL, "date", "valeur"))

### Code : Opérations en anomalie
Opérations en anomalie : pour chaque journée signalée, les opérations précises qui constituent l'anomalie.

Seules les anomalies portées par des opérations sont concernées (rafale, montant répété, inaboutis, captures,
cartes étrangères, nuit, carte expirée, attaque coordonnée, motif de capture, réseau en échec, rythme régulier,
heure morte). Les anomalies d'absence (silence, journée sans opération, trou horaire, incident de zone) et de
forme de journée (volume, montant moyen, forme horaire, réseaux, pairs, dérive, combinaison) n'ont pas
d'opération fautive : elles figurent dans la photo quotidienne des terminaux.

In [ ]:
CLE_OPERATION = [CLE_TERMINAL, "date", "rang_dans_journee"]

In [ ]:
def journees_du_type(evenements, code):
    """(terminal, date) signalés par l'anomalie `code`."""
    return evenements.filter(F.substring("type", 1, 2) == code).select(CLE_TERMINAL, "date").distinct()

In [ ]:
def _marquer(table, code, raison):
    """Opérations marquées : clé de l'opération, code de l'anomalie et raison au niveau de l'opération."""
    return table.select(*CLE_OPERATION, F.lit(code).alias("code"), raison.alias("raison_operation"))

In [ ]:
def _texte(*morceaux):
    return F.concat(*[m if not isinstance(m, str) else F.lit(m) for m in morceaux])

In [ ]:
def operations_par_regle(donnees, evenements, seuils):
    """Dictionnaire code -> opérations marquées (une table Spark par anomalie portée par des opérations)."""
    tr, retraits = donnees.transactions, donnees.retraits
    jours = {code: journees_du_type(evenements, code) for code in
             ("03", "04", "05", "06", "07", "08", "09", "20", "21", "22", "23", "24")}
    marques = {}

    p99 = tr.groupBy(CLE_TERMINAL).agg(centile("nb_ope_5min", 0.99).alias("p99"))
    marques["03"] = _marquer(
        tr.join(jours["03"], [CLE_TERMINAL, "date"]).join(F.broadcast(p99), CLE_TERMINAL)
        .filter((F.col("nb_ope_5min") >= seuils["rafale_min_10min"]) &
                (F.col("nb_ope_5min") >= seuils["rafale_facteur"] * F.col("p99"))),
        "03", _texte("rafale : ", F.col("nb_ope_5min").cast("string"), " opérations en 10 minutes"))

    fenetre = seuils["meme_montant_fenetre_s"]
    meme_montant = Window.partitionBy(CLE_TERMINAL, "montant").orderBy("t").rangeBetween(-fenetre, fenetre)
    marques["04"] = _marquer(
        retraits.join(jours["04"], [CLE_TERMINAL, "date"]).withColumn("n", F.count("*").over(meme_montant))
        .filter(F.col("n") >= seuils["meme_montant_min"]),
        "04", _texte("retrait de ", F.col("montant").cast("int").cast("string"), " euros répété ",
                     F.col("n").cast("string"), " fois en ", str(fenetre // 60), " minutes"))

    suivant = Window.partitionBy(CLE_TERMINAL).orderBy("t")
    marques["05"] = _marquer(
        retraits.join(jours["05"], [CLE_TERMINAL, "date"])
        .withColumn("inabouti_suivant", F.coalesce(F.lead("inabouti").over(suivant), F.lit(0)))
        .filter((F.col("inabouti") == 1) & ((F.col("inabouti_precedent") == 1) | (F.col("inabouti_suivant") == 1))),
        "05", F.lit("retrait inabouti dans une série d'inaboutis consécutifs"))

    heure_autour = Window.partitionBy(CLE_TERMINAL).orderBy("t").rangeBetween(-3600, 3600)
    marques["06"] = _marquer(
        tr.filter(F.col("est_capture_externe") == 1).join(jours["06"], [CLE_TERMINAL, "date"])
        .withColumn("n", F.count("*").over(heure_autour)).filter(F.col("n") >= seuils["captures_heure_min"]),
        "06", _texte("capture de carte hors réseau interne (", F.col("n").cast("string"), " captures en une heure)"))

    etrangers = retraits.filter(F.col("est_etranger") == 1).join(jours["07"], [CLE_TERMINAL, "date"])
    par_heure = Window.partitionBy(CLE_TERMINAL, "date", "heure")
    marques["07"] = _marquer(
        etrangers.withColumn("n", F.count("*").over(par_heure)).filter(F.col("n") >= seuils["etrangers_heure_min"]),
        "07", _texte("carte étrangère rare (", F.col("cat_reseau"), ") : ", F.col("n").cast("string"),
                     " retraits de ce type entre ", F.col("heure").cast("string"), " h et ",
                     (F.col("heure") + 1).cast("string"), " h"))

    marques["08"] = _marquer(
        retraits.join(jours["08"], [CLE_TERMINAL, "date"]).filter(F.col("heure") < 5),
        "08", _texte("retrait de nuit à ", F.col("heure").cast("string"), " h sur un terminal qui n'opère pas la nuit"))

    validite = donnees.validite_operations()
    if validite is not None:
        marques["09"] = _marquer(
            validite.join(jours["09"], [CLE_TERMINAL, "date"]).filter(F.col("mois_restants") < 0),
            "09", _texte("carte expirée depuis ", F.ceil(-F.col("mois_restants")).cast("int").cast("string"),
                         " mois"))

    heures_attaque = activite_suspecte_horaire(donnees, seuils).select(CLE_TERMINAL, "date", "heure")
    marques["20"] = _marquer(
        retraits.join(jours["20"], [CLE_TERMINAL, "date"]).join(heures_attaque, [CLE_TERMINAL, "date", "heure"]),
        "20", _texte("retrait pendant l'heure d'une attaque coordonnée (", F.col("heure").cast("string"),
                     " h, plusieurs terminaux de la zone touchés)"))

    captures_motif = (tr.filter((F.col("est_capture") == 1) & F.col("motif_capture").isin(seuils["motifs_surveilles"]))
                      .join(jours["21"], [CLE_TERMINAL, "date"])
                      .withColumn("n", F.count("*").over(Window.partitionBy(CLE_TERMINAL, "date", "motif_capture"))))
    marques["21"] = _marquer(
        captures_motif.filter(F.col("n") >= seuils["motif_min"]),
        "21", _texte("capture pour motif ", F.col("motif_capture"), " (", F.col("n").cast("string"),
                     " dans la journée)"))

    par_reseau = Window.partitionBy(CLE_TERMINAL, "date", "groupe_reseau")
    marques["22"] = _marquer(
        retraits.join(jours["22"], [CLE_TERMINAL, "date"]).withColumn("taux_reseau", F.avg("inabouti").over(par_reseau))
        .filter((F.col("inabouti") == 1) & (F.col("taux_reseau") >= seuils["reseau_taux_min"])),
        "22", _texte("retrait inabouti sur le réseau ", F.col("groupe_reseau"), " (",
                     F.round(100 * F.col("taux_reseau")).cast("int").cast("string"), " % d'échecs ce jour)"))

    marques["23"] = _marquer(
        rythme_regulier_operations(retraits.join(jours["23"], [CLE_TERMINAL, "date"]), seuils),
        "23", _texte("délai de ", F.col("delai").cast("string"), " s depuis le retrait précédent : ce délai revient ",
                     F.col("repetitions").cast("string"), " fois en 30 minutes (script ?)"))

    nb_jours = donnees.journees.groupBy(CLE_TERMINAL).agg(F.count("*").alias("jours"))
    jours_heure = (donnees.retraits.groupBy(CLE_TERMINAL, "heure").agg(F.countDistinct("date").alias("jours_heure"))
                   .join(nb_jours, CLE_TERMINAL).withColumn("part_jours", F.col("jours_heure") / F.col("jours")))
    marques["24"] = _marquer(
        retraits.join(jours["24"], [CLE_TERMINAL, "date"]).join(jours_heure, [CLE_TERMINAL, "heure"])
        .withColumn("n", F.count("*").over(par_heure))
        .filter((F.col("n") >= seuils["heure_morte_retraits_min"]) &
                (F.col("part_jours") <= seuils["heure_morte_part_max"])),
        "24", _texte("retrait à ", F.col("heure").cast("string"), " h, heure où le terminal n'opère presque jamais"))
    return marques

In [ ]:
def rythme_regulier_operations(retraits, seuils):
    """Retraits dont le délai depuis le précédent revient bien plus souvent que le hasard (règle 23)."""
    fenetre = seuils["regulier_fenetre_s"]
    autour = Window.partitionBy(CLE_TERMINAL).orderBy("t").rangeBetween(-fenetre, fenetre)
    meme_delai = Window.partitionBy(CLE_TERMINAL, "delai").orderBy("t").rangeBetween(-fenetre, fenetre)
    rythme = F.col("retraits_autour") / F.lit(2.0 * fenetre)
    attendu = F.greatest(F.col("retraits_autour") * rythme * F.exp(-rythme * F.col("delai")), F.lit(1e-3))
    k = F.col("repetitions")
    # factorielle plafonnée à 20! (limite Spark) : la probabilité est alors surestimée, donc prudente
    proba = F.exp(k * F.log(attendu) - attendu - F.log(F.factorial(F.least(k, F.lit(20)).cast("int"))))
    return (retraits
            .withColumn("delai", F.col("delai_precedente_s").cast("int"))
            .withColumn("retraits_autour", F.count("*").over(autour))
            .filter(F.col("delai").between(seuils["regulier_delai_min_s"], seuils["regulier_delai_max_s"]))
            .withColumn("repetitions", F.count("*").over(meme_delai))
            .filter((k >= seuils["regulier_nb_retraits"]) & (proba < seuils["regulier_proba"])))

In [ ]:
def operations_anomalies(donnees, evenements, seuils):
    """Une ligne par opération en anomalie : détail de l'opération, anomalies et raisons au niveau de l'opération.

    Colonnes : num_automate, date, heure, minute, horodatage, type_op, montant, réseau, type de client, motif de
    capture, inabouti, nb_anomalies, anomalies (codes), raisons.
    """
    noms = F.create_map([F.lit(x) for r in REGLES for x in (r.code, r.nom)])
    marques = list(operations_par_regle(donnees, evenements, seuils).values())
    toutes = reduce(lambda a, b: a.unionByName(b), marques)
    par_operation = (toutes.withColumn("anomalie", F.coalesce(noms[F.col("code")], F.col("code")))
                     .groupBy(*CLE_OPERATION)
                     .agg(F.countDistinct("code").alias("nb_anomalies"),
                          F.concat_ws(", ", F.sort_array(F.collect_set("anomalie"))).alias("anomalies"),
                          F.concat_ws(" ; ", F.collect_set("raison_operation")).alias("raisons")))
    colonnes = [c for c in ("heure", "minute", "ts", "type_op", "montant", "cat_reseau", "groupe_reseau",
                            "type_client", "motif_capture", "inabouti", "top_annule")
                if c in donnees.transactions.columns]
    return (donnees.transactions.select(*CLE_OPERATION, *colonnes)
            .join(par_operation, CLE_OPERATION)
            .withColumnRenamed("ts", "horodatage").withColumnRenamed("rang_dans_journee", "rang_operation_jour"))

### Code : Priorisation
Du catalogue d'anomalies au livrable métier : score de risque par terminal, niveau, raisons en clair,
regroupement des incidents de zone, jours particuliers et mesure de la qualité à partir des retours du métier.

In [ ]:
# Gravité de chaque anomalie (1 = à surveiller, 2 = à traiter, 3 = urgent) ; modifiable via `gravites=`.
GRAVITES = {"01": 3, "02": 3, "03": 3, "04": 3, "05": 2, "06": 2, "07": 2, "08": 2, "09": 1, "10": 2, "11": 2,
            "12": 1, "13": 1, "14": 2, "15": 1, "16": 2, "17": 1, "18": 3, "19": 3, "20": 3, "21": 2, "22": 2,
            "23": 2, "24": 1, "25": 3, "26": 1}
# Phrase lisible de chaque anomalie ; {v} est la valeur qui a déclenché l'alerte.
PHRASES = {
    "01": "aucun retrait pendant {v:.0f} h alors que le terminal est habituellement actif",
    "02": "aucune opération de la journée (environ {v:.0f} retraits attendus)",
    "03": "rafale de {v:.0f} opérations en 10 minutes",
    "04": "{v:.0f} retraits du même montant en 5 minutes",
    "05": "{v:.0f} retraits inaboutis consécutifs",
    "06": "{v:.0f} cartes capturées (hors réseau interne) en moins d'une heure",
    "07": "{v:.0f} retraits de cartes étrangères rares en une heure",
    "08": "{v:.0f} retraits entre 0 h et 5 h sur un terminal qui n'en fait presque jamais",
    "09": "{v:.0f} retrait(s) avec une carte expirée",
    "10": "retraits multipliés par {v:.1f} par rapport à l'habitude",
    "11": "montant moyen multiplié par {v:.1f}",
    "12": "répartition horaire de la journée inhabituelle (divergence {v:.2f})",
    "13": "mélange des réseaux de cartes inhabituel (divergence {v:.2f})",
    "14": "activité à {v:.0%} de l'habitude alors que les terminaux comparables sont normaux",
    "15": "baisse progressive : activité des 4 dernières semaines à {v:.0%} des 8 précédentes",
    "16": "une heure habituellement active reste vide {v:.0%} des jours du mois",
    "17": "combinaison inhabituelle de plusieurs petits écarts (score {v:.2f})",
    "18": "une coupure n'est plus délivrée (cassette vide ou défaillante ?) sur {v:.0f} retraits",
    "19": "incident de zone : {v:.0f} terminal de la zone muets à la même heure",
    "20": "activité suspecte simultanée sur {v:.0f} terminal de la zone (attaque coordonnée ?)",
    "21": "{v:.0f} captures d'un même motif (carte volée, code faux, incident de lecture...), improbable",
    "22": "{v:.0%} d'inaboutis sur un seul réseau de cartes",
    "23": "le même délai entre deux retraits revient {v:.0f} fois en 30 min, improbable (script ?)",
    "24": "{v:.0f} retraits à une heure où le terminal n'opère presque jamais",
    "25": "même carte sur plusieurs terminaux à une vitesse impossible ({v:.0f} km/h)",
    "26": "retraits multipliés par {v:.1f} par rapport à l'année précédente",
}
SYMPTOMES_DE_ZONE = ("01", "02", "10", "12", "14")   # absorbés par un incident de zone (19) le même jour

In [ ]:
def jours_particuliers(dates, autres_jours=()):
    """Jours fériés français (si la librairie holidays est disponible) et jours indiqués dans `autres_jours`."""
    particuliers = set(pd.to_datetime(list(autres_jours)).date)
    try:
        import holidays
        annees = sorted({d.year for d in dates})
        particuliers |= set(holidays.France(years=annees).keys())
    except ImportError:
        pass
    return particuliers

In [ ]:
def phrase(type_anomalie, valeur):
    """Raison lisible d'une alerte."""
    modele = PHRASES.get(str(type_anomalie)[:2])
    try:
        return modele.format(v=valeur) if modele else str(type_anomalie)
    except (ValueError, TypeError):
        return str(type_anomalie)

In [ ]:
def score_de_risque(evenements, date_analyse=None, fenetre_jours=28, gravites=None, autres_jours_particuliers=(),
                    seuils_niveaux=(3, 6)):
    """Une ligne par terminal alerté à `date_analyse` (par défaut le dernier jour) : score, niveau et raisons.

    score = somme des gravités des anomalies du jour x (1 + 0,1 x jours alertés sur `fenetre_jours`)
            x 0,5 si jour férié ou particulier.
    Les symptômes (silence, volume, forme horaire, écart aux pairs) d'un terminal pris dans un incident de zone
    sont regroupés dans l'alerte de zone. Niveaux : Modéré, Élevé (>= seuils_niveaux[0]), Critique (>= [1]).
    """
    gravites = dict(GRAVITES, **(gravites or {}))
    table = evenements.copy()
    table["date"] = pd.to_datetime(table["date"])
    table["code"] = table["type"].str[:2]
    date_analyse = pd.Timestamp(date_analyse) if date_analyse is not None else table["date"].max()
    du_jour = table[table["date"] == date_analyse].copy()
    if du_jour.empty:
        return pd.DataFrame()
    en_zone = set(du_jour.loc[du_jour["code"] == "19", CLE_TERMINAL])
    absorbe = du_jour[CLE_TERMINAL].isin(en_zone) & du_jour["code"].isin(SYMPTOMES_DE_ZONE)
    du_jour = du_jour[~absorbe]
    du_jour["gravite"] = du_jour["code"].map(gravites).fillna(1)
    du_jour["raison"] = [phrase(t, v) for t, v in zip(du_jour["type"], du_jour["valeur"])]
    debut = date_analyse - pd.Timedelta(days=fenetre_jours)
    recents = table[(table["date"] > debut) & (table["date"] <= date_analyse)]
    colonne_recurrence = "jours_alertes_{}j".format(fenetre_jours)
    recurrence = recents.groupby(CLE_TERMINAL)["date"].nunique().rename(colonne_recurrence)
    particulier = date_analyse.date() in jours_particuliers([date_analyse], autres_jours_particuliers)
    alertes = du_jour.groupby(CLE_TERMINAL).agg(
        gravite_totale=("gravite", "sum"), gravite_max=("gravite", "max"),
        nb_anomalies=("type", "nunique"),
        anomalies=("type", lambda types: ", ".join(sorted(set(types)))),
        raisons=("raison", lambda raisons: " ; ".join(raisons))).join(recurrence)
    alertes["score"] = (alertes["gravite_totale"] * (1 + 0.1 * alertes[colonne_recurrence])
                        * (0.5 if particulier else 1.0)).round(2)
    bas, haut = seuils_niveaux
    alertes["niveau"] = np.select([alertes["score"] >= haut, alertes["score"] >= bas], ["Critique", "Élevé"],
                                  "Modéré")
    alertes["date"] = date_analyse.date()
    alertes["jour_particulier"] = particulier
    alertes["avis_metier"] = ""   # à remplir par le métier : « vraie anomalie » ou « fausse alerte »
    return alertes.sort_values("score", ascending=False).reset_index()

In [ ]:
def tableau_metier(alertes, par_terminal, colonnes_infos=("ville", "code_postal", "libelle_site")):
    """Tableau final pour l'Excel J-3 et Qlik : informations du terminal + niveau, score, raisons, avis_metier."""
    infos = [c for c in colonnes_infos if c in par_terminal.columns]
    colonnes = [CLE_TERMINAL, "date", "niveau", "score", "nb_anomalies", "raisons", "anomalies",
                alertes.columns[alertes.columns.str.startswith("jours_alertes")][0], "jour_particulier", "avis_metier"]
    return alertes[colonnes].merge(par_terminal[[CLE_TERMINAL] + infos], on=CLE_TERMINAL, how="left")[
        colonnes[:2] + infos + colonnes[2:]]

In [ ]:
def precision_par_regle(evenements, retours):
    """Précision de chaque règle d'après les retours du métier.

    `retours` : DataFrame (num_automate, date, avis_metier) avec avis_metier = « vraie anomalie » ou
    « fausse alerte ». Renvoie, par type d'anomalie, le nombre d'alertes jugées et la part de vraies anomalies.
    """
    juges = retours.dropna(subset=["avis_metier"]).copy()
    juges = juges[juges["avis_metier"].astype(str).str.strip() != ""]
    juges["date"] = pd.to_datetime(juges["date"])
    juges["vraie"] = juges["avis_metier"].astype(str).str.lower().str.startswith("vrai")
    table = evenements.assign(date=pd.to_datetime(evenements["date"])).merge(
        juges[[CLE_TERMINAL, "date", "vraie"]], on=[CLE_TERMINAL, "date"])
    return (table.groupby("type")["vraie"].agg(alertes_jugees="size", precision="mean")
            .sort_values("precision"))

In [ ]:
def selection_quotidienne(alertes, budget=30, niveaux=("Critique", "Élevé")):
    """Alertes réellement transmises au métier : niveaux retenus, puis les `budget` meilleurs scores.

    Le catalogue peut repérer beaucoup de journées atypiques ; le métier ne peut en traiter qu'une petite
    partie chaque jour. On ne transmet donc que les plus graves, dans la limite de sa capacité de traitement.
    """
    retenues = alertes[alertes["niveau"].isin(niveaux)] if niveaux else alertes
    return retenues.sort_values("score", ascending=False).head(budget)

In [ ]:
def alertes_toute_periode(evenements, budget=30, niveaux=("Critique", "Élevé"), **options_score):
    """Score de risque de chaque terminal alerté, pour chaque jour de la période, et alertes transmises (`transmis`).

    Chaque jour est traité comme le serait le jour J-3 en production (récurrence sur les 28 jours précédents,
    budget quotidien d'alertes transmises).
    """
    jours = []
    for jour in sorted(pd.to_datetime(evenements["date"]).dt.normalize().unique()):
        alertes = score_de_risque(evenements, jour, **options_score)
        if alertes.empty:
            continue
        transmises = set(selection_quotidienne(alertes, budget, niveaux)[CLE_TERMINAL])
        jours.append(alertes.assign(transmis=alertes[CLE_TERMINAL].isin(transmises).astype(int)))
    if not jours:
        return pd.DataFrame()
    table = pd.concat(jours, ignore_index=True)
    table = table.rename(columns={c: "jours_alertes_28j" for c in table.columns if c.startswith("jours_alertes")})
    return table[[CLE_TERMINAL, "date", "niveau", "score", "nb_anomalies", "anomalies", "raisons", "jours_alertes_28j",
                  "jour_particulier", "transmis"]]

In [ ]:
def photo_quotidienne(journees, alertes, infos_terminal, colonnes_infos=("ville", "code_postal", "libelle_site")):
    """Photo de chaque terminal pour chaque jour de sa période d'activité (Spark) : atypique ou non, et pourquoi.

    Une ligne par terminal et par jour, y compris les jours sans opération (retraits = 0). Colonnes : activité du jour,
    atypique (0/1), niveau (Normal, Modéré, Élevé, Critique), score, anomalies, raisons en clair, transmis (0/1 :
    dans le budget quotidien), avis_metier (vide, à remplir par le métier).
    """
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F

    spark = SparkSession.builder.getOrCreate()
    bornes = journees.groupBy(CLE_TERMINAL).agg(F.min("date").alias("premier_jour"), F.max("date").alias("dernier_jour"))
    debut, fin = journees.agg(F.min("date"), F.max("date")).collect()[0]
    calendrier = (spark.range((fin - debut).days + 1)   # date_add en SQL : compatible Spark 2.3
                  .select(F.expr("date_add(to_date('{}'), cast(id as int))".format(debut)).alias("date")))
    grille = (bornes.crossJoin(calendrier)
              .filter(F.col("date").between(F.col("premier_jour"), F.col("dernier_jour")))
              .select(CLE_TERMINAL, "date"))
    activite = journees.select(CLE_TERMINAL, "date", "retraits", "inaboutis", "captures", "etrangers", "montant")
    photo = (grille.join(activite, [CLE_TERMINAL, "date"], "left")
             .fillna(0, subset=["retraits", "inaboutis", "captures", "etrangers", "montant"]))
    if alertes is not None and len(alertes):
        a = alertes.copy()
        a["date"] = pd.to_datetime(a["date"]).dt.strftime("%Y-%m-%d")
        a["jour_particulier"] = a["jour_particulier"].astype(int)
        a = spark.createDataFrame(a.astype({"score": float, "nb_anomalies": int, "jours_alertes_28j": int,
                                            "transmis": int})).withColumn("date", F.to_date("date"))
        photo = photo.join(a, [CLE_TERMINAL, "date"], "left")
    else:
        for colonne, valeur in [("niveau", None), ("score", 0.0), ("nb_anomalies", 0), ("anomalies", ""),
                                ("raisons", ""), ("jours_alertes_28j", 0), ("jour_particulier", 0), ("transmis", 0)]:
            photo = photo.withColumn(colonne, F.lit(valeur))
    photo = (photo.withColumn("atypique", F.col("niveau").isNotNull().cast("int"))
             .fillna({"niveau": "Normal", "score": 0.0, "nb_anomalies": 0, "anomalies": "", "raisons": "",
                      "jours_alertes_28j": 0, "jour_particulier": 0, "transmis": 0})
             .withColumn("avis_metier", F.lit("")))
    infos = [c for c in colonnes_infos if c in infos_terminal.columns]
    if infos:
        photo = photo.join(spark.createDataFrame(infos_terminal[[CLE_TERMINAL] + infos].astype(str)), CLE_TERMINAL, "left")
    return photo.select(CLE_TERMINAL, "date", *infos, "retraits", "inaboutis", "captures", "etrangers",
                        F.round("montant", 2).alias("montant"), "atypique", "niveau", "score", "nb_anomalies",
                        "anomalies", "raisons", "jours_alertes_28j", "jour_particulier", "transmis", "avis_metier")

In [ ]:
def synthese_annuelle(alertes, nb_terminal, par_terminal=None, colonnes_infos=("ville", "libelle_site"), nb_top=20):
    """Bilan de la période à partir de `alertes_toute_periode` : charge quotidienne, comportements, terminal.

    Renvoie un dictionnaire de tables pandas :
    - `par_jour` : terminal atypiques par jour et par niveau (jours sans alerte compris : 0) ;
    - `chiffres_cles` : moyennes et maximums par jour, en nombre et en % du parc ;
    - `comportements` : chaque anomalie, en journées terminal, terminal touchés et moyenne par jour ;
    - `terminal` : les `nb_top` terminal les plus atypiques (jours atypiques, jours graves, score cumulé, anomalie dominante).
    """
    table = alertes.copy()
    table["date"] = pd.to_datetime(table["date"])
    jours = pd.date_range(table["date"].min(), table["date"].max())
    par_jour = (table.pivot_table(index="date", columns="niveau", values=CLE_TERMINAL, aggfunc="count")
                .reindex(index=jours, columns=["Modéré", "Élevé", "Critique"]).fillna(0).astype(int))
    par_jour["atypiques"] = par_jour.sum(axis=1)
    par_jour["graves"] = par_jour["Élevé"] + par_jour["Critique"]
    par_jour["transmis"] = table.groupby("date")["transmis"].sum().reindex(jours).fillna(0).astype(int)
    chiffres = pd.DataFrame({
        "moyenne_par_jour": par_jour[["atypiques", "graves", "transmis"]].mean().round(1),
        "mediane_par_jour": par_jour[["atypiques", "graves", "transmis"]].median(),
        "maximum_par_jour": par_jour[["atypiques", "graves", "transmis"]].max(),
    })
    chiffres["pct_parc_moyen"] = (100 * chiffres["moyenne_par_jour"] / max(nb_terminal, 1)).round(2)
    chiffres.index = ["terminal atypiques (tous niveaux)", "terminal Élevé ou Critique", "alertes transmises"]
    detail = table.assign(anomalie=table["anomalies"].str.split(", ")).explode("anomalie")
    comportements = detail.groupby("anomalie").agg(
        journees_terminal=("date", "size"), terminal_touches=(CLE_TERMINAL, "nunique"), score_moyen=("score", "mean"))
    comportements = comportements.sort_values("journees_terminal", ascending=False)
    comportements["moyenne_par_jour"] = (comportements["journees_terminal"] / len(jours)).round(2)
    total = comportements["journees_terminal"].sum()
    comportements["pct_des_alertes"] = (100 * comportements["journees_terminal"] / total).round(1)
    comportements["score_moyen"] = comportements["score_moyen"].round(2)
    dominante = detail.groupby([CLE_TERMINAL, "anomalie"]).size().reset_index(name="n").sort_values("n")
    dominante = dominante.drop_duplicates(CLE_TERMINAL, keep="last").set_index(CLE_TERMINAL)
    terminal = table.groupby(CLE_TERMINAL).agg(
        jours_atypiques=("date", "nunique"),
        jours_graves=("niveau", lambda n: int(n.isin(["Élevé", "Critique"]).sum())),
        jours_transmis=("transmis", "sum"), score_cumule=("score", "sum"), score_max=("score", "max"),
        derniere_alerte=("date", "max"))
    terminal["anomalie_dominante"] = dominante["anomalie"]
    terminal["fois_anomalie_dominante"] = dominante["n"]
    terminal["pct_jours_atypiques"] = (100 * terminal["jours_atypiques"] / len(jours)).round(1)
    terminal["score_cumule"] = terminal["score_cumule"].round(1)
    terminal = terminal.sort_values(["score_cumule", "jours_graves"], ascending=False).head(nb_top).reset_index()
    if par_terminal is not None:
        infos = [c for c in colonnes_infos if c in par_terminal.columns]
        terminal = terminal.merge(par_terminal[[CLE_TERMINAL] + infos], on=CLE_TERMINAL, how="left")[
            [CLE_TERMINAL] + infos + [c for c in terminal.columns if c != CLE_TERMINAL]]
    return {"par_jour": par_jour, "chiffres_cles": chiffres, "comportements": comportements.reset_index(),
            "terminal": terminal}

### Code : Analyses métier
Ce que l'analyse apprend au métier, au-delà des alertes : saturation, besoin en billets, créneau de
maintenance, fiabilité et tendance, temps entre pannes, usage par rapport aux terminaux comparables.

In [ ]:
CODES_PANNE = ("01", "02", "05", "18", "19")       # silence, journée muette, inaboutis en série, cassette, zone
CODES_FIABILITE = ("01", "02", "05", "06", "18", "22")

In [ ]:
def saturation_horaire(retraits, capacite_horaire=30):
    """Par terminal : retraits de l'heure la plus chargée (99e centile) et nombre d'heures à capacité.

    `capacite_horaire` : nombre de retraits qu'un terminal peut servir en une heure (environ 2 minutes par client).
    """
    par_heure = retraits.groupBy(CLE_TERMINAL, "date", "heure").agg(F.count("*").alias("n"))
    return en_pandas(par_heure.groupBy(CLE_TERMINAL).agg(
        centile("n", 0.99).alias("heure_chargee_p99"), F.max("n").alias("heure_max"),
        F.sum((F.col("n") >= capacite_horaire).cast("int")).alias("heures_a_capacite"),
        F.countDistinct("date").alias("jours")), max_lignes=200000).sort_values("heures_a_capacite", ascending=False)

In [ ]:
def besoin_billets(ops):
    """Billets délivrés en moyenne par coupure (nb1-nb4), terminal et jour de semaine, et montant moyen délivré."""
    retraits = ops.filter(F.col("est_retrait") == 1)
    par_jour = retraits.groupBy(CLE_TERMINAL, "date", "jour_semaine").agg(
        F.sum("montant_billets").alias("montant_delivre"),
        *[F.sum("billets_nb{}".format(i)).alias("billets_nb{}".format(i)) for i in range(1, 5)])
    return en_pandas(par_jour.groupBy(CLE_TERMINAL, "jour_semaine").agg(
        F.avg("montant_delivre").alias("montant_moyen_delivre"),
        *[F.avg("billets_nb{}".format(i)).alias("billets_nb{}_par_jour".format(i)) for i in range(1, 5)]),
        max_lignes=2000000)

In [ ]:
def creneau_maintenance(retraits, heure_debut=6, heure_fin=22):
    """Par terminal : l'heure de la plage [heure_debut, heure_fin[ la moins chargée en moyenne."""
    par_heure = en_pandas(retraits.filter(F.col("heure").between(heure_debut, heure_fin - 1))
                          .groupBy(CLE_TERMINAL, "heure").agg(F.count("*").alias("n")), max_lignes=2000000)
    jours = en_pandas(retraits.groupBy(CLE_TERMINAL).agg(F.countDistinct("date").alias("jours")), max_lignes=200000)
    table = (par_heure.pivot(index=CLE_TERMINAL, columns="heure", values="n")
             .reindex(columns=range(heure_debut, heure_fin)).fillna(0))
    table = table.div(jours.set_index(CLE_TERMINAL)["jours"].reindex(table.index), axis=0)
    return pd.DataFrame({"heure_la_plus_creuse": table.idxmin(axis=1),
                         "retraits_moyens_a_cette_heure": table.min(axis=1).round(2),
                         "heure_la_plus_chargee": table.idxmax(axis=1)})

In [ ]:
def fiabilite(evenements, par_terminal, semaines_recentes=8):
    """Incidents (pannes, inaboutis, captures, cassettes) pour 1 000 retraits, et tendance récente.

    tendance = incidents par semaine sur les `semaines_recentes` dernières semaines / avant ; > 1 : se dégrade.
    """
    incidents = evenements[evenements["type"].str[:2].isin(CODES_FIABILITE)].copy()
    incidents["date"] = pd.to_datetime(incidents["date"])
    limite = incidents["date"].max() - pd.Timedelta(weeks=semaines_recentes)
    duree_avant = max((limite - incidents["date"].min()).days / 7, 1)
    table = incidents.groupby(CLE_TERMINAL).agg(incidents=("type", "size"),
                                           recents=("date", lambda d: (d > limite).sum()))
    table["tendance"] = ((table["recents"] / semaines_recentes) /
                         ((table["incidents"] - table["recents"]).clip(lower=1) / duree_avant)).round(2)
    table = table.join(par_terminal.set_index(CLE_TERMINAL)["retraits"])
    table["incidents_pour_1000_retraits"] = (1000 * table["incidents"] / table["retraits"].clip(lower=1)).round(2)
    return table.sort_values("incidents_pour_1000_retraits", ascending=False)

In [ ]:
def temps_entre_pannes(evenements):
    """Par terminal touché : nombre de pannes (jours) et nombre moyen de jours entre deux pannes."""
    pannes = evenements[evenements["type"].str[:2].isin(CODES_PANNE)][[CLE_TERMINAL, "date"]].drop_duplicates()
    pannes = pannes.assign(date=pd.to_datetime(pannes["date"])).sort_values([CLE_TERMINAL, "date"])
    pannes["ecart"] = pannes.groupby(CLE_TERMINAL)["date"].diff().dt.days
    return (pannes.groupby(CLE_TERMINAL).agg(jours_de_panne=("date", "size"), jours_entre_pannes=("ecart", "mean"))
            .round(1).sort_values("jours_entre_pannes"))

In [ ]:
def usage_par_rapport_aux_pairs(par_terminal, groupe="libelle_site", sous_utilise=0.3, sature=3.0):
    """Retraits par jour du terminal / médiane de son groupe : sous-utilisé (< 0,3) ou très sollicité (> 3)."""
    if groupe not in par_terminal.columns:
        return pd.DataFrame()
    table = par_terminal[[CLE_TERMINAL, groupe, "retraits_par_jour_actif"]].copy()
    table["mediane_groupe"] = table.groupby(groupe)["retraits_par_jour_actif"].transform("median")
    table["rapport_aux_pairs"] = (table["retraits_par_jour_actif"] / table["mediane_groupe"]).round(2)
    table["lecture"] = np.select([table["rapport_aux_pairs"] < sous_utilise, table["rapport_aux_pairs"] > sature],
                                 ["sous-utilisé : à déplacer ou à retirer ?", "très sollicité : à doubler ?"], "")
    return table[table["lecture"] != ""].sort_values("rapport_aux_pairs")

### Code : Graphiques
Graphiques : chaque fonction reçoit des tableaux déjà calculés et ne fait que dessiner.

In [ ]:
COULEURS_ANOMALIES = {"01": "black", "02": "dimgray", "03": "tab:red", "04": "tab:pink", "05": "tab:orange",
                      "06": "tab:purple", "07": "tab:green", "08": "navy", "09": "tab:brown", "10": "tab:blue",
                      "11": "tab:olive", "12": "tab:cyan", "13": "gold", "14": "crimson", "15": "teal",
                      "16": "sienna", "17": "magenta", "18": "darkgoldenrod", "19": "darkred", "20": "indigo",
                      "21": "darkviolet", "22": "chocolate", "23": "deepskyblue", "24": "darkslategray",
                      "25": "olivedrab", "26": "slateblue"}
ASPECT_FRANCE = 1 / np.cos(np.radians(46.5))

In [ ]:
# ------------------------------------------------------------------ briques
def couleur_anomalie(nom):
    """Couleur fixe d'une anomalie à partir de son numéro (« 03 Rafale » -> rouge)."""
    return COULEURS_ANOMALIES.get(str(nom)[:2], "grey")

In [ ]:
def afficher_matrice(ax, matrice, titre, cmap="viridis", annoter=False, etiquettes_x=None, **options):
    """Carte de chaleur d'un DataFrame pandas, avec ses étiquettes (et ses valeurs si `annoter`)."""
    image = ax.imshow(np.ma.masked_invalid(matrice.values.astype(float)), aspect="auto", cmap=cmap,
                      interpolation="nearest", **options)
    ax.set_yticks(range(matrice.shape[0]))
    ax.set_yticklabels([str(i)[:32] for i in matrice.index])
    ax.set_xticks(range(matrice.shape[1]))
    ax.set_xticklabels(etiquettes_x if etiquettes_x is not None else matrice.columns)
    if annoter:
        for i in range(matrice.shape[0]):
            for j in range(matrice.shape[1]):
                ax.text(j, i, "{:.0f}".format(matrice.values[i, j]), ha="center", va="center", fontsize=8)
    ax.set_title(titre)
    plt.colorbar(image, ax=ax)
    return image

In [ ]:
def tracer_matrice(matrice, titre, cmap="viridis", taille=(14, 4), xlabel=None, **options):
    """Figure contenant une seule carte de chaleur."""
    fig, ax = plt.subplots(figsize=taille)
    afficher_matrice(ax, matrice, titre, cmap=cmap, **options)
    if xlabel:
        ax.set_xlabel(xlabel)
    plt.show()

In [ ]:
def carte_france(ax, points, valeurs=None, titre="", fond=None, cmap="viridis", taille=6):
    """Nuage de terminaux sur leurs coordonnées (sans fond de carte), avec un fond gris optionnel."""
    if fond is not None:
        ax.scatter(fond["longitude"], fond["latitude"], s=1, color="#d5d9e0")
    if valeurs is None:
        ax.scatter(points["longitude"], points["latitude"], s=taille)
    else:
        ordre = valeurs.sort_values().index
        nuage = ax.scatter(points.loc[ordre, "longitude"], points.loc[ordre, "latitude"], c=valeurs[ordre],
                           s=taille, cmap=cmap, alpha=0.85)
        plt.colorbar(nuage, ax=ax, shrink=0.7)
    ax.set_aspect(ASPECT_FRANCE)
    ax.set_title(titre, fontsize=10)

In [ ]:
def nuage_par_categorie(ax, table, x, y, categorie, titre, echelle_x_log=True, nb_categories=8):
    """Nuage de points coloré par catégorie (les plus fréquentes ; les autres en gris), avec légende."""
    categories = table[categorie].fillna("inconnu")
    palette = plt.get_cmap("tab10")
    legende = {c: palette(i) for i, c in enumerate(categories.value_counts().head(nb_categories).index)}
    ax.scatter(table[x], table[y], s=10, alpha=0.6, c=[legende.get(c, (0.7, 0.7, 0.7, 1.0)) for c in categories])
    for nom, couleur in legende.items():
        ax.scatter([], [], color=couleur, label=str(nom)[:20])
    if echelle_x_log:
        ax.set_xscale("log")
    ax.set_xlabel(x)
    ax.set_title(titre)
    ax.legend(fontsize=7, loc="upper right")

In [ ]:
def grille_axes(nb, colonnes=4, hauteur=3.4, largeur=18):
    """Grille de sous-graphiques pour `nb` panneaux ; les panneaux en trop sont masqués."""
    lignes = int(np.ceil(nb / colonnes))
    fig, axes = plt.subplots(lignes, colonnes, figsize=(largeur, hauteur * lignes), squeeze=False)
    for ax in axes.ravel()[nb:]:
        ax.axis("off")
    return fig, axes.ravel()[:nb]

In [ ]:
# ------------------------------------------------------------------ volumes et rythmes
def tracer_volumes(jour):
    """Retraits par jour (avec moyennes glissantes) et nombre de terminaux actifs."""
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(jour.index, jour["retraits"], lw=0.6, alpha=0.5, label="par jour")
    for fenetre in (7, 28):
        axes[0].plot(jour.index, jour["retraits"].rolling(fenetre, center=True).mean(), lw=1.6,
                     label="moyenne {} jours".format(fenetre))
    axes[0].set_title("Retraits par jour, parc complet")
    axes[0].legend()
    axes[1].plot(jour.index, jour["terminal_actifs"], lw=0.8, color="tab:green")
    axes[1].set_title("terminal ayant au moins une opération dans la journée")
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_calendrier(serie, titre, cmap="YlGnBu"):
    """Calendrier (jours de semaine x semaines) d'une série journalière indexée par date."""
    semaine = ((serie.index - serie.index[0]).days + serie.index[0].dayofweek) // 7
    calendrier = np.full((7, semaine.max() + 1), np.nan)
    calendrier[serie.index.dayofweek, semaine] = serie.values
    debuts = [i for i, d in enumerate(serie.index) if d.day == 1]
    fig, ax = plt.subplots(figsize=(16, 3.5))
    image = ax.imshow(calendrier, aspect="auto", cmap=cmap, interpolation="nearest")
    ax.set_yticks(range(7))
    ax.set_yticklabels(ORDRE_JOURS)
    ax.set_xticks([semaine[i] for i in debuts])
    ax.set_xticklabels([serie.index[i].strftime("%b") for i in debuts])
    ax.set_title(titre)
    plt.colorbar(image, ax=ax)
    plt.show()

In [ ]:
def tracer_profils(profil_horaire, profil_hebdo):
    """Profils normalisés de tous les terminaux (une ligne par terminal) et leurs distributions."""
    fig, axes = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={"width_ratios": [3, 1.3]})
    afficher_matrice(axes[0], profil_horaire, "Profil horaire de chaque terminal (part de ses retraits par heure)",
                     cmap="magma", vmax=np.percentile(profil_horaire.values, 99))
    axes[0].set_yticks([])
    axes[0].set_ylabel("{} terminal, triés par heure de pointe".format(len(profil_horaire)))
    afficher_matrice(axes[1], profil_hebdo, "Profil hebdomadaire (trié par part du week-end)")
    axes[1].set_yticks([])
    plt.tight_layout()
    plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    axes[0].bar(range(24), np.bincount(profil_horaire.values.argmax(axis=1), minlength=24))
    axes[0].set_title("Heure de pointe des terminaux")
    axes[1].hist((profil_horaire[[0, 1, 2, 3, 4, 5, 22, 23]].sum(axis=1) * 100).clip(upper=40), bins=60)
    axes[1].set_yscale("log")
    axes[1].set_title("% des retraits entre 22 h et 6 h, par terminal")
    axes[2].hist((profil_hebdo["sam"] + profil_hebdo["dim"]) * 100, bins=60)
    axes[2].axvline(200 / 7, color="grey", lw=1)
    axes[2].set_title("% des retraits le week-end, par terminal (trait : répartition uniforme)")
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_carte_parc(ecarts):
    """Carte terminal x jour des écarts à l'habitude (log2) : bleu = en dessous, rouge = au-dessus."""
    fig, ax = plt.subplots(figsize=(16, 10))
    image = ax.imshow(np.ma.masked_invalid(ecarts.values), aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3,
                      interpolation="nearest")
    debuts = [i for i, d in enumerate(ecarts.columns) if d.day == 1]
    ax.set_xticks(debuts)
    ax.set_xticklabels([ecarts.columns[i].strftime("%b") for i in debuts])
    ax.set_ylabel("{} terminal, du plus actif (haut) au moins actif (bas)".format(len(ecarts)))
    ax.set_title("Retraits du jour / habitude du terminal pour ce jour de semaine (log2)")
    plt.colorbar(image, ax=ax, label="log2(retraits / habitude)")
    plt.show()

In [ ]:
# ------------------------------------------------------------------ parc, géographie, relations
def tracer_parc(par_terminal):
    """Volume par terminal, concentration de l'activité, taux de présence."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    axes[0].hist(np.log10(par_terminal["retraits_par_jour_actif"].clip(lower=0.1)), bins=60)
    axes[0].set_title("Retraits par jour actif (log10)")
    tries = np.sort(par_terminal["retraits"].values)[::-1]
    axes[1].plot(np.arange(1, len(tries) + 1) / len(tries) * 100, np.cumsum(tries) / tries.sum() * 100)
    axes[1].set_title("% des retraits réalisés par les x % de terminaux les plus actifs")
    axes[1].set_xlabel("% des terminaux")
    axes[2].hist(par_terminal["taux_presence"], bins=50)
    axes[2].set_title("Taux de présence (jours actifs / durée de vie)")
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_cartes_geographiques(en_metropole):
    """Quatre cartes de France : volume, cartes étrangères rares, inaboutis, captures."""
    vues = [("retraits_par_jour_actif", "Retraits par jour actif (log10)", "viridis", True),
            ("part_etrangers_pct", "% de cartes étrangères rares", "magma_r", False),
            ("taux_inaboutis_pct", "% de retraits inaboutis", "Reds", False),
            ("captures_pour_1000", "Captures pour 1 000 retraits", "Purples", False)]
    fig, axes = plt.subplots(2, 2, figsize=(16, 15))
    for ax, (colonne, titre, cmap, echelle_log) in zip(axes.ravel(), vues):
        valeurs = en_metropole[colonne].clip(upper=en_metropole[colonne].quantile(0.99))
        valeurs = np.log10(valeurs.clip(lower=0.1)) if echelle_log else valeurs
        carte_france(ax, en_metropole, valeurs, titre, cmap=cmap)
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_volume_par_site(par_terminal, nb_sites=12):
    """Retraits par jour actif selon le type de site d'implantation (boîtes à moustaches)."""
    sites = par_terminal["libelle_site"].value_counts().head(nb_sites).index
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.boxplot([par_terminal.loc[par_terminal["libelle_site"] == s, "retraits_par_jour_actif"].dropna() for s in sites],
               showfliers=False)
    ax.set_xticklabels([str(s)[:22] for s in sites], rotation=30, ha="right")
    ax.set_yscale("log")
    ax.set_title("Retraits par jour actif selon le type de site d'implantation")
    plt.show()

In [ ]:
def tracer_relations(terminal_actifs, categorie):
    """Nuages : volume du terminal face à ses taux (inaboutis, captures, étrangers, montant, nuit)."""
    vues = [("taux_inaboutis_pct", "Taux d'inaboutis (%) selon le volume"),
            ("captures_pour_1000", "Captures pour 1 000 retraits selon le volume"),
            ("part_etrangers_pct", "% de cartes étrangères rares selon le volume"),
            ("montant_moyen", "Montant moyen selon le volume"),
            ("part_nuit_pct", "% de retraits de nuit selon le volume")]
    fig, axes = grille_axes(len(vues), colonnes=3, hauteur=5.2, largeur=19)
    for ax, (colonne, titre) in zip(axes, vues):
        nuage_par_categorie(ax, terminal_actifs, "retraits_par_jour_actif", colonne, categorie, titre)
    plt.tight_layout()
    plt.show()

In [ ]:
# ------------------------------------------------------------------ montants, inaboutis, captures
def tracer_montants(tranches, quantiles_heure, montant_jour_heure):
    """Distribution des montants, montant selon l'heure (médiane et dispersion), montant jour x heure."""
    fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
    axes[0].bar(tranches["tranche"], tranches["count"], width=9)
    axes[0].set_yscale("log")
    axes[0].set_title("Montants retirés (tranches de 10 €, log)")
    axes[1].fill_between(quantiles_heure.index, quantiles_heure["p10"], quantiles_heure["p90"], alpha=0.15,
                         label="10 % - 90 %")
    axes[1].fill_between(quantiles_heure.index, quantiles_heure["p25"], quantiles_heure["p75"], alpha=0.3,
                         label="25 % - 75 %")
    axes[1].plot(quantiles_heure.index, quantiles_heure["p50"], marker="o", label="médiane")
    axes[1].set_title("Montant retiré selon l'heure")
    axes[1].legend()
    afficher_matrice(axes[2], montant_jour_heure, "Montant moyen (€) : jour x heure", cmap="cividis")
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_taux(taux_heure, taux_reseau, titre):
    """Taux (en %) selon l'heure et selon le réseau."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 4))
    axes[0].plot(taux_heure.iloc[:, 0], taux_heure["taux"], marker="o")
    axes[0].set_title("{} selon l'heure (%)".format(titre))
    axes[1].barh(taux_reseau.iloc[:, 0].astype(str), taux_reseau["taux"])
    axes[1].set_title("{} selon le réseau (%)".format(titre))
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_histogramme(valeurs, titre, bins=60, log=True):
    """Histogramme simple."""
    plt.hist(valeurs, bins=bins)
    if log:
        plt.yscale("log")
    plt.title(titre)
    plt.show()

In [ ]:
def tracer_reseaux(parts_mois, profils_heure):
    """Part mensuelle des réseaux (hors réseau principal) et profil horaire de chaque réseau."""
    fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
    parts_mois.drop(columns=[parts_mois.sum().idxmax()]).plot(
        ax=axes[0], title="Part mensuelle des réseaux (hors principal), %")
    profils_heure.plot(ax=axes[1], title="Profil horaire de chaque réseau (part de ses retraits)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ------------------------------------------------------------------ dynamique dans la journée
def tracer_silences_et_delais(debut_duree, delais_log):
    """Silences improbables (durée x heure de début) et délais entre retraits (échelle log)."""
    fig, axes = plt.subplots(1, 2, figsize=(17, 5))
    if len(debut_duree):
        afficher_matrice(axes[0], debut_duree, "Silences improbables : durée (h) x heure de début", cmap="Greys")
        axes[0].set_xlabel("heure de début")
    axes[1].bar(delais_log["log10_delai"], delais_log["count"], width=0.09)
    axes[1].set_xticks([0, 1, np.log10(60), np.log10(600), np.log10(3600), np.log10(86400)])
    axes[1].set_xticklabels(["1 s", "10 s", "1 min", "10 min", "1 h", "1 j"])
    axes[1].set_yscale("log")
    axes[1].set_title("Délai entre deux retraits successifs (échelle log)")
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_ecart_parc_heure(ecarts):
    """Parc jour x heure : log2(retraits / habitude) ; une case bleue isolée = incident commun."""
    fig, ax = plt.subplots(figsize=(10, 14))
    image = ax.imshow(ecarts.values, aspect="auto", cmap="RdBu_r", vmin=-2, vmax=2, interpolation="nearest")
    reperes = [i for i, d in enumerate(ecarts.index) if d.day == 1]
    ax.set_yticks(reperes)
    ax.set_yticklabels([ecarts.index[i].strftime("%m/%y") for i in reperes], fontsize=7)
    ax.set_xticks(range(24))
    ax.set_xlabel("heure")
    ax.set_title("Parc complet : retraits par heure / habitude (log2, bleu = déficit)")
    plt.colorbar(image, label="log2(retraits / habitude)")
    plt.show()

In [ ]:
def regarder_journee(ops, num_automate, date_txt):
    """Retraits d'un terminal par quart d'heure ce jour-là, comparés à la médiane des 8 mêmes jours précédents."""
    jour_cible = pd.Timestamp(date_txt)
    dates = [(jour_cible - pd.Timedelta(weeks=k)).strftime("%Y-%m-%d") for k in range(0, 9)]
    detail = en_pandas(
        ops.filter((F.col(CLE_TERMINAL) == num_automate) & F.col("date").cast("string").isin(dates))
        .groupBy("date", "type_op", ((F.col("heure") * 60 + F.col("minute")) / 15).cast("int").alias("quart"))
        .agg(F.count("*").alias("n")))
    if detail.empty:
        print("Aucune opération trouvée.")
        return
    detail["date"] = pd.to_datetime(detail["date"])
    retraits = (detail[detail["type_op"] != "CAPTURE"].groupby(["date", "quart"])["n"].sum()
                .unstack(fill_value=0).reindex(columns=range(96), fill_value=0))
    du_jour = retraits.loc[jour_cible] if jour_cible in retraits.index else pd.Series(0, index=range(96))
    reference = retraits.drop(index=jour_cible, errors="ignore").median()
    heures = np.arange(96) / 4
    plt.figure(figsize=(14, 4))
    plt.step(heures, reference.values, where="post", color="grey", label="médiane des 8 mêmes jours précédents")
    plt.step(heures, du_jour.values, where="post", color="tab:red", label=str(date_txt))
    captures = detail[(detail["type_op"] == "CAPTURE") & (detail["date"] == jour_cible)]
    if len(captures):
        plt.scatter(captures["quart"] / 4, [0] * len(captures), marker="x", s=80, color="black", label="captures")
    plt.title("terminal {} : retraits par quart d'heure".format(num_automate))
    plt.xlabel("heure")
    plt.legend()
    plt.show()
    print("Ce jour : {} retraits ; référence : {:.0f}".format(int(du_jour.sum()), reference.sum()))

In [ ]:
def carte_heures_terminal(ops, num_automate, debut=None, fin=None):
    """Retraits d'un terminal par heure, un jour par ligne ; croix = inaboutis, ronds = captures."""
    selection = ops.filter(F.col(CLE_TERMINAL) == num_automate)
    if debut:
        selection = selection.filter(F.col("date") >= F.lit(debut).cast("date"))
    if fin:
        selection = selection.filter(F.col("date") <= F.lit(fin).cast("date"))
    detail = en_pandas(selection.groupBy("date", "heure").agg(F.sum("est_retrait").alias("retraits"),
                                                              F.sum("inabouti").alias("inaboutis"),
                                                              F.sum("est_capture").alias("captures")))
    if detail.empty:
        print("Aucune opération.")
        return
    detail["date"] = pd.to_datetime(detail["date"])
    jours = pd.date_range(detail["date"].min(), detail["date"].max())
    grille = (detail.pivot(index="date", columns="heure", values="retraits")
              .reindex(index=jours, columns=range(24)).fillna(0))
    fig, ax = plt.subplots(figsize=(11, max(4, len(jours) * 0.04)))
    image = ax.imshow(np.log1p(grille.values), aspect="auto", cmap="magma", interpolation="nearest")
    for colonne, marque, couleur in [("inaboutis", "x", "cyan"), ("captures", "o", "lime")]:
        lignes = detail[detail[colonne] > 0]
        ax.scatter(lignes["heure"], [jours.get_loc(d) for d in lignes["date"]], marker=marque, s=12,
                   color=couleur, label=colonne)
    reperes = [i for i, d in enumerate(jours) if d.day == 1 or i == 0]
    ax.set_yticks(reperes)
    ax.set_yticklabels([jours[i].strftime("%d/%m/%y") for i in reperes], fontsize=7)
    ax.set_xticks(range(24))
    ax.set_xlabel("heure")
    ax.set_title("terminal {} : retraits par heure (log), un jour par ligne".format(num_automate))
    ax.legend(loc="upper right", fontsize=8)
    plt.colorbar(image, label="log(1 + retraits)")
    plt.show()

In [ ]:
# ------------------------------------------------------------------ catalogue des anomalies
def tracer_frequences(recap):
    """Journées terminal concernées, % de terminaux touchés, % de terminaux qui récidivent, par anomalie."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 0.45 * len(recap) + 2), sharey=True)
    for ax, (colonne, titre, couleur) in zip(axes, [
            ("journees_terminal", "Journées terminal concernées", "tab:blue"),
            ("pct_terminal_touches", "% des terminaux touchés au moins une fois", "tab:orange"),
            ("pct_terminal_recurrents", "% des terminaux touchés qui récidivent (>= 4 semaines)", "tab:red")]):
        ax.barh(np.arange(len(recap)), recap[colonne], color=[couleur_anomalie(t) for t in recap.index])
        ax.set_title(titre)
    axes[0].set_yticks(np.arange(len(recap)))
    axes[0].set_yticklabels(recap.index)
    axes[0].invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_recurrence(par_type_terminal):
    """Répartition des terminaux touchés selon le nombre de semaines où l'anomalie revient."""
    classes = pd.cut(par_type_terminal["semaines"], [0, 1, 3, 9, 1000], labels=["1 semaine", "2-3", "4-9", "10 et plus"])
    (pd.crosstab(par_type_terminal["type"], classes, normalize="index") * 100).plot(
        kind="barh", stacked=True, figsize=(14, 0.4 * par_type_terminal["type"].nunique() + 2), colormap="RdYlGn_r",
        title="Récurrence : terminal touchés selon le nombre de semaines avec l'anomalie (%)")
    plt.gca().invert_yaxis()
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
def tracer_intensites(valeurs_types):
    """Distribution de la valeur qui a déclenché chaque anomalie."""
    types = sorted(valeurs_types["type"].unique())
    fig, axes = grille_axes(len(types))
    for ax, nom in zip(axes, types):
        valeurs = valeurs_types.loc[valeurs_types["type"] == nom, "valeur"].abs()
        ax.hist(valeurs.clip(upper=valeurs.quantile(0.99)), bins=30, color=couleur_anomalie(nom))
        ax.set_title("{} (n = {})".format(nom, len(valeurs)), fontsize=9)
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_evolution(par_jour_type):
    """Nombre d'événements sur 7 jours glissants, par anomalie."""
    fig, ax = plt.subplots(figsize=(16, 5))
    for nom, groupe in par_jour_type.groupby("type"):
        serie = groupe.set_index("date")["count"].sort_index()
        ax.plot(serie.index, serie.rolling(7, min_periods=1).sum(), label=nom, color=couleur_anomalie(nom))
    ax.set_title("Événements sur 7 jours glissants, par anomalie")
    ax.legend(fontsize=8, ncol=3)
    plt.show()

In [ ]:
def tracer_cartes_anomalies(occurrences, positions):
    """Une petite carte de France par anomalie : terminal touchés, couleur selon le nombre d'occurrences."""
    fig, axes = grille_axes(occurrences.shape[1], hauteur=4.6)
    for ax, nom in zip(axes, occurrences.columns):
        compte = occurrences[nom].reindex(positions.index).fillna(0)
        touches = compte[compte > 0]
        carte_france(ax, positions.loc[touches.index], np.log1p(touches), "{} ({} terminal)".format(nom, len(touches)),
                     fond=positions, cmap="autumn_r", taille=10)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_galerie(journees, evenements, terminal_choisis, synthese):
    """Retraits par jour des terminaux choisis ; chaque anomalie est marquée à sa date avec sa couleur."""
    fig, axes = grille_axes(len(terminal_choisis), colonnes=3, hauteur=3.5, largeur=19)
    for ax, terminal in zip(axes, terminal_choisis):
        serie = journees[journees[CLE_TERMINAL] == terminal].set_index("date")["retraits"].sort_index()
        ax.plot(serie.index, serie.values, lw=0.7, color="grey")
        for nom, groupe in evenements[evenements[CLE_TERMINAL] == terminal].groupby("type"):
            ax.scatter(groupe["date"], serie.reindex(groupe["date"]).fillna(0), s=30, label=nom[:2],
                       color=couleur_anomalie(nom), zorder=3)
        ax.set_title("terminal {} ({:.0f} anomalies)".format(terminal, synthese.loc[terminal, "evenements"]), fontsize=9)
        ax.legend(fontsize=7, ncol=5)
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_chronologie(evenements, terminal_choisis):
    """Un point par anomalie : terminal (lignes) x date ; un alignement vertical = épisode commun."""
    position = {terminal: i for i, terminal in enumerate(terminal_choisis)}
    fig, ax = plt.subplots(figsize=(16, 0.27 * len(terminal_choisis) + 2))
    for nom, groupe in evenements[evenements[CLE_TERMINAL].isin(terminal_choisis)].groupby("type"):
        ax.scatter(groupe["date"], groupe[CLE_TERMINAL].map(position), s=22, label=nom, color=couleur_anomalie(nom))
    ax.set_yticks(range(len(terminal_choisis)))
    ax.set_yticklabels(terminal_choisis, fontsize=7)
    ax.invert_yaxis()
    ax.set_title("Chronologie des anomalies des terminaux les plus touchés")
    ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.0, 0.5))
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_explications(scores, part=0.001):
    """Indicateurs le plus souvent en cause dans les journées les plus atypiques (règle 17)."""
    plus_atypiques = scores.nlargest(max(1, int(len(scores) * part)), "score")
    colonnes = [c for c in scores.columns if c.startswith("z_")]
    poids = plus_atypiques[colonnes].abs().mean().sort_values()
    fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
    axes[0].hist(scores["score"], bins=80)
    axes[0].axvline(plus_atypiques["score"].min(), color="red", lw=1)
    axes[0].set_yscale("log")
    axes[0].set_title("Score d'atypicité des journées terminal (trait : seuil retenu)")
    axes[1].barh([c[2:] for c in poids.index], poids.values, color="tab:purple")
    axes[1].set_title("Écart moyen (en écarts habituels) dans les journées retenues")
    plt.tight_layout()
    plt.show()

In [ ]:
def tracer_charge_quotidienne(charge, budget):
    """Nombre de terminaux alertés chaque jour, par niveau, avec la capacité de traitement du métier."""
    table = charge.groupby(["jour", "niveau"]).size().unstack(fill_value=0)
    table = table.reindex(columns=[n for n in ("Critique", "Élevé", "Modéré") if n in table.columns])
    couleurs = {"Critique": "tab:red", "Élevé": "tab:orange", "Modéré": "gold"}
    ax = table.plot(kind="bar", stacked=True, figsize=(16, 4.5), color=[couleurs[c] for c in table.columns],
                    width=0.85)
    ax.axhline(budget, color="black", lw=1.2, ls="--", label="capacité quotidienne du métier")
    ax.set_xticks(range(0, len(table), 7))
    ax.set_xticklabels([pd.Timestamp(j).strftime("%d/%m") for j in table.index[::7]], rotation=0)
    ax.set_title("terminal alertés par jour et par niveau (60 derniers jours)")
    ax.legend()
    plt.show()

In [ ]:
def tracer_synthese_annuelle(synthese):
    """Bilan de la période : terminal atypiques par jour (moyenne glissante), comportements et terminal les plus atypiques."""
    par_jour, comportements, terminal = synthese["par_jour"], synthese["comportements"], synthese["terminal"]
    fig = plt.figure(figsize=(16, 13))
    ax = fig.add_subplot(3, 1, 1)
    couleurs = {"Modéré": "#f2c94c", "Élevé": "#f2994a", "Critique": "#d64545"}
    ax.stackplot(par_jour.index, *[par_jour[n].rolling(7, min_periods=1).mean() for n in couleurs],
                 labels=list(couleurs), colors=list(couleurs.values()), alpha=0.85)
    ax.plot(par_jour.index, par_jour["transmis"].rolling(7, min_periods=1).mean(), color="black", lw=1.2,
            label="transmis au métier")
    ax.set_title("terminal atypiques par jour, par niveau (moyenne sur 7 jours)")
    ax.legend(loc="upper left", ncol=4)
    ax = fig.add_subplot(3, 2, 3)
    haut = comportements.head(15).iloc[::-1]
    ax.barh(haut["anomalie"], haut["journees_terminal"], color=[couleur_anomalie(a) for a in haut["anomalie"]])
    ax.set_title("Comportements atypiques les plus fréquents (journées terminal)")
    ax = fig.add_subplot(3, 2, 4)
    ax.barh(haut["anomalie"], haut["terminal_touches"], color=[couleur_anomalie(a) for a in haut["anomalie"]])
    ax.set_yticklabels([])
    ax.set_title("Nombre de terminaux touchés au moins une fois")
    ax = fig.add_subplot(3, 1, 3)
    top = terminal.head(20).iloc[::-1]
    ax.barh(top[CLE_TERMINAL], top["jours_graves"], color="#d64545", label="jours Élevé ou Critique")
    ax.barh(top[CLE_TERMINAL], top["jours_atypiques"] - top["jours_graves"], left=top["jours_graves"],
            color="#f2c94c", label="jours Modéré")
    for y, (jours, dominante) in enumerate(zip(top["jours_atypiques"], top["anomalie_dominante"])):
        ax.text(jours + 0.3, y, str(dominante), va="center", fontsize=8)
    ax.set_xlim(0, 1.45 * max(top["jours_atypiques"].max(), 1))
    ax.set_title("terminal les plus atypiques de la période (classés par score cumulé) et leur anomalie dominante")
    ax.legend(loc="lower right")
    fig.tight_layout()
    plt.show()

### Configuration complète
Valeurs par défaut, remplacées par celles de la section 1.

In [ ]:
conf = configuration(
    colonnes=COLONNES,
    metier={"codes_reseau": CODES_RESEAU, "codes_privative": CODES_CARTE_PRIVATIVE,
            "groupes_reseau": GROUPES_RESEAU, "reseau_interne": RESEAU_INTERNE, "valeur_porteur": VALEUR_PORTEUR,
            "reseaux_etrangers": RESEAUX_ETRANGERS_RARES, "types_operation": TYPES_OPERATION,
            "motifs_capture": MOTIFS_CAPTURE},
    options={"format_date": FORMAT_DATE, "diviseur_montant": DIVISEUR_MONTANT,
             "date_debut": DATE_DEBUT, "date_fin": DATE_FIN},
    seuils={},               # ex. {"rafale_min_10min": 15}
)
SEUILS = conf["seuils"]

## 2. Lecture et correspondance des colonnes
Le tableau indique, pour chaque colonne attendue, la colonne retenue dans la table. Une colonne mal trouvée
se corrige dans la configuration (`colonnes=`).

In [ ]:
source = lire_table(DATASET_OPERATIONS, TABLE_HIVE_OPERATIONS)
brut, correspondance = standardiser(source, conf)
display(pd.DataFrame([{"colonne attendue": cle, "colonne retenue": nom or "ABSENTE"}
                      for cle, nom in correspondance.items()]))
retenues = set(n for n in correspondance.values() if n)
print("Colonnes de la table non utilisées :", [c for c in source.columns if c not in retenues] or "aucune")
brut.show(5, truncate=False)
nb_lignes_brut = brut.count()
print("Lignes :", fmt_nombre(nb_lignes_brut))

## 3. Qualité de la donnée brute
Valeurs manquantes, valeurs prises par les colonnes catégorielles, formats, doublons exacts.

In [ ]:
display(qualite_colonnes(brut, nb_lignes_brut))
for colonne in ["lib_ope", "type_reseau", "code_carte_privative", "top_annule", "lib_motif_capture_carte",
                "coupure1", "coupure2", "coupure3", "coupure4", "libelle_site"]:
    if correspondance.get(colonne):
        print("\n=== {} ===".format(colonne))
        display(valeurs_prises(brut, colonne, nb_lignes_brut))

In [ ]:
date_op = expression_date(dict(brut.dtypes)["date_operation"], conf["options"]["format_date"])
formats = controle_formats(brut, date_op, expression_heure())
for cle, valeur in formats.items():
    print("{:<28}: {}".format(cle, valeur))
print("Montant médian : {:.2f} €".format((formats["montant_median_brut"] or 0) / conf["options"]["diviseur_montant"]))
nb_doublons = nb_lignes_brut - brut.distinct().count()
print("Lignes en double exact : {} ({:.3f} %)".format(fmt_nombre(nb_doublons), 100 * nb_doublons / nb_lignes_brut))

## 4. Préparation : une ligne par opération, enrichie
Chaque opération reçoit ses propres indicateurs (type, réseau, inabouti, capture interne / hors réseau interne) puis des
indicateurs calculés à partir de ses voisines sur le même terminal : délai depuis l'opération précédente,
opérations à +/- 5 minutes, même montant que la précédente, captures dans l'heure, rang dans la journée.

In [ ]:
ops = construire_operations(brut, correspondance, conf)
resume = resumer(ops)
display(en_pandas(brut.groupBy("lib_ope", expression_type_operation(conf["metier"], correspondance)
                               .alias("type_op"))
                  .count().orderBy(F.desc("count"))))
transactions = enrichir_transactions(ops, conf["options"]["persister"])
journees = vue_terminal_jour(ops, conf["options"]["persister"])
referentiel = referentiel_terminal(brut, correspondance)
retraits = ops.filter(F.col("est_retrait") == 1)
validite = lire_validite(retraits)
donnees = DonneesAnalyse(ops, transactions, journees, conf, validite, referentiel, correspondance)
exemple = transactions.select(CLE_TERMINAL).limit(1).collect()[0][0]
display(en_pandas(transactions.filter(F.col(CLE_TERMINAL) == exemple).orderBy("t").limit(20)
                  .select(CLE_TERMINAL, "ts", "type_op", "cat_reseau", "montant", "montant_billets", "inabouti",
                          "delai_precedente_s", "nb_ope_5min", "capture_dans_l_heure")))
if DATASET_SORTIE_TRANSACTIONS:
    import dataiku
    import dataiku.spark as dkuspark
    dkuspark.write_with_schema(dataiku.Dataset(DATASET_SORTIE_TRANSACTIONS), transactions.drop("t"))

### Indicateurs par réseau et par type de client (Porteur / Non_porteur), comptés transaction par transaction
`type_client` n'est pas renseigné pour le réseau interne : toute opération interne est comptée comme « Porteur ».

In [ ]:
display(indicateurs_par_reseau(ops))
if correspondance["type_client"]:    # Porteur / Non_porteur (les opérations interne sont forcées à Porteur)
    display(indicateurs_par(ops, ["type_client"]))

## 5. Volumes dans le temps
**Comment lire :** une chute isolée du parc entier signale un événement commun (incident réseau, jour
férié) ; elle ne doit pas être imputée aux terminaux individuellement.

In [ ]:
jour = volumes_par_jour(ops)
tracer_volumes(jour)
print("Journées du parc les plus éloignées de la normale du même jour de semaine :")
display(jour.reindex(jour["ratio_vs_meme_jour"].sub(1).abs().sort_values(ascending=False).index).head(15)
        [["jour_semaine", "ferie", "retraits", "ratio_vs_meme_jour", "terminal_actifs", "captures", "inaboutis"]])
tracer_calendrier(jour["retraits"], "Calendrier du parc : retraits par jour")
tracer_matrice(croisement_jour_heure(retraits, jour.groupby("jour_semaine").size()),
                  "Retraits moyens par jour : jour de semaine x heure", xlabel="heure")

## 6. Comportements types des terminaux
Profils normalisés (part des retraits par heure et par jour de semaine), une ligne par terminal : les terminaux qui se
ressemblent forment des bandes ; un terminal qui ne ressemble à aucun autre ressort.

**Carte du parc** : une ligne par terminal, une colonne par jour, couleur = retraits / habitude du terminal pour ce jour
de semaine (log2). Trait vertical = événement commun ; trait horizontal = terminal durablement anormal ; point
isolé = anomalie ponctuelle.

In [ ]:
profil_horaire = profils_normalises(retraits, "heure", range(24))
profil_horaire = profil_horaire.iloc[pd.Series(profil_horaire.values.argmax(axis=1)).argsort().values]
profil_hebdo = profils_normalises(retraits, "jour_semaine", ORDRE_JOURS)
profil_hebdo = profil_hebdo.iloc[(profil_hebdo["sam"] + profil_hebdo["dim"]).argsort().values]
tracer_profils(profil_horaire, profil_hebdo)
journees_pd = en_pandas(journees, max_lignes=20000000)
journees_pd["date"] = pd.to_datetime(journees_pd["date"])
tracer_carte_parc(carte_parc(journees_pd))

## 7. Le parc, sa géographie et les relations entre indicateurs

In [ ]:
par_terminal = statistiques_par_terminal(ops).merge(en_pandas(referentiel), on=CLE_TERMINAL, how="left")
display(par_terminal.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T.round(2))
tracer_parc(par_terminal)
geo = geographie_presente(correspondance)
if geo:
    display(controles_geographiques(par_terminal, conf["options"]["metropole"]))
    tracer_cartes_geographiques(filtrer_metropole(par_terminal, conf["options"]["metropole"]))
if "libelle_site" in par_terminal.columns:
    tracer_volume_par_site(par_terminal)
    tracer_relations(par_terminal[par_terminal["retraits"] >= 500], "libelle_site")

## 8. Montants, réseaux, inaboutis, captures, validité des cartes

In [ ]:
tracer_montants(tranches_montants(retraits), quantiles_montant(retraits, "heure"),
                   croisement_jour_heure(retraits, None, valeur="montant"))
display(montants_frequents(retraits, resume["retraits"]))
print(controles_montants(retraits))
display(ecarts_billets(retraits))

In [ ]:
tracer_reseaux(repartition_reseaux(retraits, "mois"), profils_reseaux(retraits, "heure"))
tracer_histogramme(par_terminal.loc[par_terminal["retraits"] >= 1000, "part_etrangers_pct"],
                      "Part de cartes étrangères rares par terminal (%)")

In [ ]:
tracer_taux(taux_par(retraits, "heure", "inabouti"), taux_par(retraits, "groupe_reseau", "inabouti"),
               "Taux d'inaboutis")
display(taux_captures(ops))
tracer_histogramme(par_terminal.loc[par_terminal["retraits"] >= 500, "taux_inaboutis_pct"], "Taux d'inaboutis par terminal (%)")
if validite is not None:
    expirees = validite.agg(F.avg((F.col("mois_restants") < 0).cast("int"))).collect()[0][0]
    print("Retraits avec une carte expirée : {:.3f} %".format(100 * (expirees or 0)))

## 9. Dynamique à l'intérieur de la journée
**Silences** : heures consécutives sans retrait, comparées au rythme horaire habituel du terminal recalé sur
l'activité du jour ; probabilité qu'un tel silence arrive par hasard = exp(-attendu) (loi de Poisson).

In [ ]:
retraits_tx = donnees.retraits
display(tranches_delais(retraits_tx))
silences = donnees.silences()
improbables = silences.filter(F.col("proba_hasard") < SEUILS["silence_proba"])
print("Journées terminal avec un silence improbable :", fmt_nombre(improbables.count()))
tracer_silences_et_delais(silences_debut_duree(improbables), delais_echelle_log(retraits_tx))
tracer_ecart_parc_heure(ecart_parc_jour_heure(retraits))

## 10. Catalogue des anomalies

| # | Anomalie | Ce qui la déclenche |
|---|---|---|
| 01 | Silence anormal | heures sans retrait, probabilité < 1/100 000 vu le rythme du terminal |
| 02 | Journée sans opération | aucun retrait, seulement pour un terminal très fréquenté (top 10 %) |
| 03 | Rafale | >= 10 opérations en 10 min et x2 le niveau élevé habituel du terminal |
| 04 | Même montant répété | >= 6 retraits du même montant en 5 minutes |
| 05 | Inaboutis en série | >= 3 retraits inaboutis consécutifs |
| 06 | Captures hors réseau interne en série | >= 3 captures hors réseau interne en une heure |
| 07 | Afflux de cartes étrangères | réseaux étrangers rares : >= 20 en 1 h et x5 l'habitude |
| 08 | Nuit inhabituelle | >= 3 retraits entre 0 h et 5 h sur un terminal qui n'en fait presque jamais |
| 09 | Carte expirée | date de validité dépassée |
| 10 | Volume journalier anormal | retraits x3 vs 8 mêmes jours de semaine et 7 derniers jours |
| 11 | Montant moyen anormal | montant moyen x3 vs 8 mêmes jours de semaine |
| 12 | Forme horaire inhabituelle | répartition des retraits dans la journée inhabituelle (Jensen-Shannon) |
| 13 | Mélange de réseaux inhabituel | part des réseaux de cartes inhabituelle (Jensen-Shannon) |
| 14 | Écart aux terminaux pairs | le terminal décroche (< 40 %) quand ses pairs restent normaux, ou l'inverse |
| 15 | Dérive lente | 4 semaines < 60 % des 8 précédentes, sans arrêt net, pairs stables |
| 16 | Trou horaire récurrent | une heure active vide plus d'un jour sur deux dans le mois |
| 17 | Combinaison de petits écarts | Isolation Forest sur 9 écarts robustes, journées non vues ailleurs |
| 18 | Cassette vide ou défaillante | une coupure habituelle (>= 10 % des billets) disparaît des billets délivrés |
| 19 | Incident de zone | >= 5 terminaux d'un même département en silence improbable à la même heure |
| 20 | Attaque coordonnée | rafale, montant répété ou cartes étrangères sur >= 3 terminaux d'une zone dans l'heure |
| 21 | Motif de capture inhabituel | >= 3 captures d'un motif, probabilité de hasard < 1/10 000 (Poisson) |
| 22 | Inaboutis concentrés sur un réseau | >= 30 % d'échecs sur un réseau, les autres réseaux < 10 % |
| 23 | Rythme trop régulier | même délai (à la seconde) entre deux retraits >= 8 fois en 30 min, hasard < 1e-6 |
| 24 | Activité à une heure morte | >= 3 retraits à une heure active moins de 1 % des jours |
| 25 | Même carte sur plusieurs terminaux | vitesse de déplacement > 300 km/h (identifiant de carte nécessaire) |
| 26 | Écart à l'année précédente | retraits x3 ou /3 vs l'année précédente (plus d'un an de données) |

Seuils : `SEUILS_DEFAUT` (surcharges dans la section 1). Pour ajouter une anomalie : une fonction
décorée par `@regle("NN", "Nom")` dans la section « Code de l'analyse ».

In [ ]:
evenements = detecter_anomalies(donnees, SEUILS)
print("Événements repérés :", fmt_nombre(evenements.count()))
recap, par_type_terminal = synthese_par_type(evenements, len(par_terminal), resume["jours"])
display(recap.round(2))
tracer_frequences(recap)
tracer_recurrence(par_type_terminal)

### 10.1 Intensité, évolution et cumul des anomalies
**Cumul** : part des journées de l'anomalie (ligne) qui présentent aussi l'anomalie (colonne). Une forte
valeur indique deux symptômes d'une même cause.

In [ ]:
evenements_pd = en_pandas(evenements.select(CLE_TERMINAL, "date", "type", "valeur"), max_lignes=20000000)
evenements_pd["date"] = pd.to_datetime(evenements_pd["date"])
tracer_intensites(evenements_pd)
tracer_evolution(evenements_pd.groupby(["date", "type"]).size().rename("count").reset_index())
cumul = cooccurrences(evenements)
tracer_matrice(cumul, "% des journées de l'anomalie (ligne) avec aussi l'anomalie (colonne)", cmap="Purples",
                  taille=(12, 9), annoter=True, etiquettes_x=[t[:2] for t in cumul.columns], vmax=60)

### 10.2 Anomalies cachées : ce que les règles directes ne voient pas
Journées signalées **uniquement** par les règles 12 à 17. Pour la règle 17, les indicateurs en cause sont
donnés en « écarts habituels » (+ = au-dessus de l'habitude du terminal).

In [ ]:
codes_directs = ["{:02d}".format(i) for i in range(1, 12)]
journees_directes = set(map(tuple, evenements_pd[evenements_pd["type"].str[:2].isin(codes_directs)]
                            [[CLE_TERMINAL, "date"]].values))
cachees = evenements_pd[~evenements_pd["type"].str[:2].isin(codes_directs)].copy()
cachees["vue_aussi_par_une_regle_directe"] = [(g, d) in journees_directes
                                              for g, d in zip(cachees[CLE_TERMINAL], cachees["date"])]
display(cachees.groupby("type")["vue_aussi_par_une_regle_directe"].agg(["size", "mean"])
        .rename(columns={"size": "journees", "mean": "part_vue_par_regle_directe"}).round(2))
scores = donnees.resultat("scores_multivaries")
if scores is not None:
    tracer_explications(scores, SEUILS["multivarie_part"])
    display(scores.nlargest(15, "score")[[CLE_TERMINAL, "date", "score", "explication"]])

### 10.3 Où sont les anomalies ?

In [ ]:
synthese_terminal, occurrences = synthese_par_terminal(par_type_terminal, par_terminal)
if geo:
    positions = filtrer_metropole(par_terminal.set_index(CLE_TERMINAL)[["longitude", "latitude"]],
                                      conf["options"]["metropole"])
    tracer_cartes_anomalies(occurrences, positions)
if "libelle_site" in par_terminal.columns:
    site = par_terminal.set_index(CLE_TERMINAL)["libelle_site"]
    part_site = (occurrences > 0).join(site).groupby("libelle_site").mean().reindex(
        site.value_counts().head(12).index) * 100
    tracer_matrice(part_site, "% des terminaux de chaque type de site touchés par chaque anomalie", cmap="Oranges",
                      taille=(16, 6), annoter=True, etiquettes_x=[t[:2] for t in part_site.columns])

### 10.4 Les terminaux les plus atypiques : galerie, chronologie, détail

In [ ]:
display(synthese_terminal.head(30)[[c for c in ("ville", "libelle_site", "retraits_par_jour_actif")
                               if c in synthese_terminal.columns] + ["nb_anomalies_differentes", "evenements", "detail"]])
tracer_galerie(journees_pd, evenements_pd, list(synthese_terminal.index[:12]), synthese_terminal)
tracer_chronologie(evenements_pd, list(synthese_terminal.sort_values("evenements", ascending=False).index[:40]))

### 10.5 Un exemple réel par anomalie
Le cas le plus marqué de chaque anomalie : la journée du terminal, quart d'heure par quart d'heure, comparée aux
8 mêmes jours de semaine précédents. Pour approfondir un terminal : `carte_heures_terminal(ops, "terminal...")`.

In [ ]:
exemples = exemples_par_type(evenements)
display(exemples)
for _, cas in exemples.iterrows():
    print("\n=== {} - terminal {} le {} (valeur {:.2f}) ===".format(cas["type"], cas[CLE_TERMINAL], cas["date"], cas["valeur"]))
    regarder_journee(ops, cas[CLE_TERMINAL], str(cas["date"]))

### 10.6 Choisir les anomalies à retenir
Une anomalie utile est assez fréquente, touche plusieurs terminaux et revient, sans être si fréquente qu'elle
décrive en fait un comportement normal.

In [ ]:
display(aide_au_choix(recap).round(2))

## 11. Priorisation : ce qui est réellement transmis au métier
Le catalogue repère toutes les journées atypiques ; les transmettre toutes noierait le métier. Pour le jour
analysé (J-3), chaque terminal alerté reçoit un **score de risque** :

score = somme des gravités des anomalies du jour (1 à surveiller, 2 à traiter, 3 urgent)
        x (1 + 0,1 x nombre de jours alertés sur les 28 derniers jours) x 0,5 si jour férié ou particulier.

Les symptômes d'un terminal pris dans un incident de zone sont regroupés dans une seule alerte de zone. Niveaux :
Modéré (< 3), Élevé (3 à 6), Critique (>= 6). Seuls les niveaux Élevé et Critique sont transmis, dans la
limite de `BUDGET_ALERTES_JOUR` (capacité de traitement du métier).

In [ ]:
BUDGET_ALERTES_JOUR = 30
DATE_ANALYSE = None          # None = dernier jour de la table ; en production : J-3
alertes = score_de_risque(evenements_pd, DATE_ANALYSE,
                                       autres_jours_particuliers=conf["options"]["jours_particuliers"])
transmises = selection_quotidienne(alertes, BUDGET_ALERTES_JOUR)
print("terminal analysés : {} | terminal avec au moins une anomalie ce jour : {} | alertes transmises : {}".format(
    len(par_terminal), len(alertes), len(transmises)))
print(alertes["niveau"].value_counts().to_string())
tableau = tableau_metier(transmises, par_terminal)
display(tableau)

### Combien d'alertes chaque jour ?
Nombre de terminaux alertés par jour et par niveau sur toute la période : c'est la charge que représenterait
chaque niveau pour le métier. Le budget quotidien plafonne ce qui est transmis.

In [ ]:
charge = pd.concat([score_de_risque(evenements_pd, jour_analyse)
                    .assign(jour=jour_analyse)[["jour", "niveau"]]
                    for jour_analyse in sorted(evenements_pd["date"].unique())[-60:]])
tracer_charge_quotidienne(charge, BUDGET_ALERTES_JOUR)

### Mesurer la qualité avec le métier
Le tableau transmis contient une colonne `avis_metier` (« vraie anomalie » / « fausse alerte »). Une fois
remplie et rechargée, `precision_par_regle(evenements_pd, retours)` donne la précision de chaque
règle, pour ajuster ses seuils ou sa gravité.

## 12. Ce que l'analyse apprend au métier
Au-delà des alertes, les mêmes transactions répondent à des questions d'exploitation.

### 12.1 Saturation : quels terminal atteignent leur capacité ?
Capacité retenue : 30 retraits par heure (environ 2 minutes par client). Un terminal souvent à capacité fait
attendre ses clients : candidat à un second terminal.

In [ ]:
saturation = saturation_horaire(retraits)
display(saturation.head(20))
tracer_histogramme(saturation["heure_chargee_p99"], "Retraits de l'heure la plus chargée (99e centile), par terminal")

### 12.2 Besoin en billets par coupure et par jour de semaine
Billets délivrés en moyenne par cassette : base pour planifier les chargements (jour et coupures à prévoir).

In [ ]:
billets = besoin_billets(ops)
billets["jour"] = billets["jour_semaine"].map(JOURS_FR)
parc_billets = billets.groupby("jour")[[c for c in billets.columns if c.startswith("billets_nb")]].sum()
tracer_matrice(parc_billets.reindex(ORDRE_JOURS).T, "Billets délivrés par jour, parc complet : cassette x jour",
                  cmap="Greens", taille=(12, 3.5), annoter=True)
display(billets.sort_values("montant_moyen_delivre", ascending=False).head(15))

### 12.3 Créneau de maintenance : l'heure la plus creuse de chaque terminal (6 h - 22 h)

In [ ]:
creneaux = creneau_maintenance(retraits)
display(creneaux["heure_la_plus_creuse"].value_counts().sort_index().rename("terminal").to_frame().T)
display(creneaux.head(15))

### 12.4 Fiabilité, tendance et temps entre pannes
Incidents (pannes, inaboutis, captures, cassettes) pour 1 000 retraits ; tendance > 1 : le terminal se dégrade
depuis 8 semaines (maintenance préventive à prévoir).

In [ ]:
fiabilite_terminaux = fiabilite(evenements_pd, par_terminal)
print("terminaux qui se dégradent (tendance >= 2) :", int((fiabilite_terminaux["tendance"] >= 2).sum()))
display(fiabilite_terminaux.head(20))
display(temps_entre_pannes(evenements_pd).head(20))

### 12.5 Usage par rapport aux terminaux comparables
Retraits par jour du terminal rapportés à la médiane de son type de site : sous-utilisé (< 0,3) ou très
sollicité (> 3).

In [ ]:
display(usage_par_rapport_aux_pairs(par_terminal))

## 13. Photo quotidienne des terminaux : dataset de sortie
**Une ligne par terminal et par jour** sur toute la période (y compris les jours sans opération), atypique ou non :
activité du jour, `atypique` (0/1), `niveau` (Normal, Modéré, Élevé, Critique), `score`, `anomalies`,
`raisons` en clair, `transmis` (1 = dans le budget quotidien d'alertes) et `avis_metier` à remplir par le
métier. Chaque jour est scoré comme le jour J-3 en production. Écrit dans `DATASET_SORTIE_PHOTO` (section 1).

In [ ]:
alertes_annee = alertes_toute_periode(
    evenements_pd, BUDGET_ALERTES_JOUR, autres_jours_particuliers=conf["options"]["jours_particuliers"])
photo = photo_quotidienne(journees, alertes_annee, par_terminal)
resume_photo = en_pandas(photo.groupBy("niveau").agg(F.count("*").alias("lignes_terminal_jour"),
                                                     F.sum("transmis").alias("transmises")).orderBy("niveau"))
print("Lignes terminal x jour :", fmt_nombre(int(resume_photo["lignes_terminal_jour"].sum())))
display(resume_photo)
jour_exemple = alertes_annee.sort_values("score", ascending=False)["date"].iloc[0] if len(alertes_annee) else None
if jour_exemple is not None:
    print("Exemple : photo du", jour_exemple, "(terminal atypiques d'abord)")
    display(en_pandas(photo.filter(F.col("date") == F.lit(str(jour_exemple)).cast("date"))
                      .orderBy(F.desc("score"), CLE_TERMINAL).limit(25)))
if DATASET_SORTIE_PHOTO:
    import dataiku
    import dataiku.spark as dkuspark
    dkuspark.write_with_schema(dataiku.Dataset(DATASET_SORTIE_PHOTO), photo)
    print("Dataset écrit :", DATASET_SORTIE_PHOTO)

## 14. Synthèse de l'année
**Combien de terminaux atypiques en moyenne chaque jour, quels comportements ressortent le plus, quels terminal sont les
plus atypiques sur l'année.** Calculé à partir de la photo quotidienne (section 13).

**Comment lire :** en haut, la charge quotidienne par niveau (un pic = un événement de la période : incident de
zone, attaque...) ; au milieu, les comportements les plus fréquents et le nombre de terminaux qu'ils touchent (un
comportement fréquent mais concentré sur peu de terminaux désigne des terminaux à problème récurrent) ; en bas, les terminaux à
traiter en priorité sur l'année, avec leur anomalie dominante.

In [ ]:
synthese_annee = synthese_annuelle(alertes_annee, len(par_terminal), par_terminal)
print("Parc : {} terminal | {} jours".format(len(par_terminal), len(synthese_annee["par_jour"])))
display(synthese_annee["chiffres_cles"])
tracer_synthese_annuelle(synthese_annee)
print("Comportements atypiques les plus fréquents sur l'année :")
display(synthese_annee["comportements"])
print("terminal les plus atypiques sur l'année :")
display(synthese_annee["terminal"])

## 15. Opérations en anomalie : dataset de sortie
**Une ligne par opération qui constitue une anomalie** : par exemple chacun des 25 retraits d'une rafale, chacun
des 7 retraits de 300 euros répétés, chaque capture d'une série, chaque retrait d'une carte expirée. Pour chaque
opération : son détail (heure, montant, réseau, type de client...), ses anomalies et la raison au niveau de
l'opération. Les anomalies d'absence (silence, journée sans opération, incident de zone) et de forme de journée
(volume, forme horaire, réseaux, pairs, dérive) n'ont pas d'opération fautive : elles sont dans la photo
quotidienne (section 13). Écrit dans `DATASET_SORTIE_OPERATIONS` (section 1).

In [ ]:
operations_en_anomalie = persister(operations_anomalies(donnees, evenements, SEUILS))
print("Opérations en anomalie :", fmt_nombre(operations_en_anomalie.count()))
display(en_pandas(operations_en_anomalie.groupBy("anomalies").agg(F.count("*").alias("operations"),
                                                                  F.countDistinct(CLE_TERMINAL).alias("terminal"))
                  .orderBy(F.desc("operations"))))
print("Exemple : les opérations en anomalie du terminal le plus touché")
terminal_exemple = (operations_en_anomalie.groupBy(CLE_TERMINAL, "date").count().orderBy(F.desc("count")).first())
if terminal_exemple:
    display(en_pandas(operations_en_anomalie.filter((F.col(CLE_TERMINAL) == terminal_exemple[CLE_TERMINAL]) &
                                                    (F.col("date") == terminal_exemple["date"]))
                      .orderBy("horodatage").limit(40)))
if DATASET_SORTIE_OPERATIONS:
    import dataiku
    import dataiku.spark as dkuspark
    dkuspark.write_with_schema(dataiku.Dataset(DATASET_SORTIE_OPERATIONS), operations_en_anomalie)
    print("Dataset écrit :", DATASET_SORTIE_OPERATIONS)

In [ ]:
for table in (evenements, journees, transactions, ops):
    table.unpersist()